In [ ]:
#####-------------------------------- NOTE MAIN IWSLT17 EN-ZH NOTE --------------------------------------------------#####
##########################################################################################################################
######################|--------------------------------------------------------------|####################################
###################################🔗 MAIN | TRAIN | TEST LOOP 🔗########################################################
######################|--------------------------------------------------------------|####################################
##########################################################################################################################
#####-------------------------------- NOTE MAIN IWSLT17 EN-ZH NOTE --------------------------------------------------#####



# 📄  main_IWSLT17En_Zh.py
########################################################################################################################
####-------| NOTE 1.1 IMPORTS LIBRARIES  | XXX -------------------------------------------------------##################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 📜 === Enable flexible CUDA memory allocation to reduce fragmentation ===
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ======================================================================================================
# 📜 === Core Libraries ===
# ======================================================================================================
import sys
import numpy as np
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import time
import random
import math
import re
import glob
from copy import deepcopy
import sacrebleu  # ✅ for BLEU evaluation
from timm.loss import LabelSmoothingCrossEntropy
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ======================================================================================================
# 📜 === Fairseq & Supporting Imports ===
# ======================================================================================================
import importlib.metadata as importlib_metadata
import fairseq  # type: ignore
from fairseq.data import Dictionary, data_utils, iterators          # type: ignore
from fairseq.tasks.translation import TranslationTask               # type: ignore
from fairseq import options                                         # type: ignore
from fairseq.dataclass.utils import convert_namespace_to_omegaconf  # type: ignore                                             
from fairseq import utils                                           # type: ignore 
import hydra                                                        # type: ignore  
# ────────────────────────────────────────────────────────────────────────────────────────────────
from ptflops import get_model_complexity_info
from calflops import calculate_flops
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ======================================================================================================
# ♻️ === Print environment summary for sanity check ===
# ======================================================================================================
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Fairseq:", fairseq.__version__)
print("OmegaConf:", importlib_metadata.version("omegaconf"))
print("Hydra-Core:", importlib_metadata.version("hydra-core"))
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 1.2. DEFINE PROJECT PATH | XXX ---------------------------------------------------####################
########################################################################################################################

# ✅ Define working directory
Project_PATH = r"C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH"
if os.getcwd() != Project_PATH:
    os.chdir(Project_PATH)
print(f"✅ Current working directory: {os.getcwd()}")


# ✅ Define absolute paths
PROJECT_PATH = Project_PATH
MODELS_PATH = os.path.join(Project_PATH, "models")
ACTIVATION_PATH = os.path.join(Project_PATH, "activation")


# ✅ Ensure necessary paths are in sys.path
for path in [PROJECT_PATH, MODELS_PATH, ACTIVATION_PATH]:
    if path not in sys.path:
        sys.path.append(path)

# ✅ Print updated sys.path for debugging
print("✅ sys.path updated:")
for path in sys.path:
    print("   📂", path)
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 1.3. OTHER IMPORTS | XXX ---------------------------------------------------------####################
########################################################################################################################

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 📜 ============  Imput parser safe for afno because of --fno-bias, etc  =======================
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 1️⃣✅ Import parser from parser_IWSLT17En_Zh.py
from parser_IWSLT17En_Zh import get_parser

# ✅ Create parser and parse arguments
parser = get_parser()
# args, unknown = parser.parse_known_args()

# ✅ IMPORTANT: Do NOT read Jupyter / VSCode kernel arguments
# This prevents the "--f" ambiguity issue
exp_args = parser.parse_args(args=[])

num_aug_splits = exp_args.aug_splits

print(f"✅ Parser imported successfully | num_aug_splits = {num_aug_splits}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣✅ Import Fairseq parser/config
from fairseq_config_IWSLT17En_Zh import get_fairseq_parser
# ✅ Build Fairseq configuration
fairseq_args, cfg = get_fairseq_parser(exp_args)


num_aug_splits = exp_args.aug_splits

print(f"✅ Parser imported successfully | num_aug_splits = {num_aug_splits}")

print(f"✅ Encoder Embed Dim: {fairseq_args.encoder_embed_dim}")
print(f"✅ Decoder Layers: {fairseq_args.decoder_layers}")
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 3️⃣✅ Import model summary utility
from models.model_summary.model_summary import save_model_summary
from models.model_summary.model_summary_full import save_model_summary_full

print("✅ model_summary.py loaded successfully")
print("✅ model_summary_full.py loaded successfully")
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 2. SEEDING FOR REPRODUCIBILITY | XXX ---------------------------------------------####################
########################################################################################################################

# ────────────────────────────────────────────────────────────────────────────────────────────────
def set_seed_torch(seed):
    torch.manual_seed(seed)
# ────────────────────────────────────────────────────────────────────────────────────────────────
def set_seed_main(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True      # ✅ Default: "False" for Faster, non-deterministic kernels | "True" to Ensure deterministic behavior for CuDNN (Slower)
    torch.backends.cudnn.benchmark = False         # ✅ Default: "True" for Autotune kernels for performance   | "False" Disable CuDNN's autotuning for reproducibility (Slower)

    torch.backends.cuda.matmul.allow_tf32 = False  # ✅ Disable TF32 (strict reproducibility)
    torch.backends.cudnn.allow_tf32 = False        # ✅ Disable TF32 (strict reproducibility)

    # torch.use_deterministic_algorithms(True, warn_only=True)

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ ============= Define Seed =============
seed1, seed2 = exp_args.seed1, exp_args.seed2
set_seed_torch(seed1)  
set_seed_main(seed2)  
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 3.1. PROCESSING DATASET 1 | XXX ---- TOKENIZATION & DATASET PREPARATION |---------####################
########################################################################################################################
########################################################################################################################
####-------| NOTE 3.1. PROCESSING DATASET 1 | XXX ------------| ENGLISH→CHINESE |-------------------####################
########################################################################################################################

"""
Stage 1:
Train SentencePiece tokenizer, encode the raw English→Chinese corpus,
and create aligned train/test text splits for neural machine translation.
"""

# ─────────────────────────────────────────────────────────────────────────────────────────────────
print(
    f"🔎🔎 run_dataset_tokenization: {exp_args.run_dataset_tokenization} "
    f"{'(RUNNING)✔️✔️' if exp_args.run_dataset_tokenization else '(SKIPPED)❌❌'}"
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if exp_args.run_dataset_tokenization:
    from datasets import load_dataset    
    import sentencepiece as spm

    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Path for EN→ZH dataset
    DATA_PATH = r"C:\Users\emeka\Research\ModelCUDA\Big_Data_Journal\Comparison\Code\Paper\github3\IWSLT\Transformer_Baselines_EN_ZH\Dataset"
    os.makedirs(DATA_PATH, exist_ok=True)
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  1️⃣ ======== Download and export official IWSLT17 dataset =======================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Download to your folder (as the Hugging Face cache)
    en_zh = load_dataset(
        "IWSLT/iwslt2017", "iwslt2017-en-zh",
        cache_dir=DATA_PATH,
        trust_remote_code=True
    )
    zh_en = load_dataset(
        "IWSLT/iwslt2017", "iwslt2017-zh-en",
        cache_dir=DATA_PATH,
        trust_remote_code=True
    )  # optional
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  2️⃣ ======== Dataset verification ===============================================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ sanity check (should show ~231k train)
    print(f"EN→ZH dataset: {en_zh}")
    print(f"ZH→EN dataset: {zh_en}")

    print("EN→ZH splits:", list(en_zh.keys()), {k: len(en_zh[k]) for k in en_zh.keys()})
    print("ZH→EN splits:", list(zh_en.keys()), {k: len(zh_en[k]) for k in zh_en.keys()})

    # optional assertions
    assert set(en_zh.keys()) == {"train", "validation", "test"}
    assert set(zh_en.keys()) == {"train", "validation", "test"}
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  3️⃣ ======== Export official train/validation/test splits ======================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Export official IWSLT17 splits to plain text (EN/ZH) under raw_zh_en_dataset_path
    def dump_split(ds_dict, split, src, tgt, out_dir):
        src_fp = os.path.join(out_dir, f"{split}.{src}")
        tgt_fp = os.path.join(out_dir, f"{split}.{tgt}")
        with open(src_fp, "w", encoding="utf-8") as fs, open(tgt_fp, "w", encoding="utf-8") as ft:
            for row in ds_dict[split]:
                tr = row["translation"]
                fs.write((tr[src] or "").strip() + "\n")
                ft.write((tr[tgt] or "").strip() + "\n")
        print(f"Wrote {split}: {src_fp} | {tgt_fp}")

    for split in ["train", "validation", "test"]:
        dump_split(en_zh, split, "en", "zh", DATA_PATH)
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  4️⃣ ========  Train SentencePiece model =========================================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────     

    print("🧠 Training SentencePiece model for EN→ZH...")

    # Use TRAIN split for SPM training
    SRC_EN = os.path.join(DATA_PATH, "train.en")
    TGT_ZH = os.path.join(DATA_PATH, "train.zh")

    spm_model_prefix = os.path.join(DATA_PATH, "spm_enzh")

    spm.SentencePieceTrainer.Train(
        input=f"{SRC_EN},{TGT_ZH}",
        model_prefix=spm_model_prefix,
        vocab_size=10000,
        character_coverage=0.9995,   # ✅ --character_coverage for rich charcater langauage: Japanese or Chinese = 0.9995 | other languages with small character set: 1.0
        model_type="bpe"
    )
    print(f"✅ SentencePiece model saved to: {spm_model_prefix}.model / .vocab")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  5️⃣ ========  Encode files ======================================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    print("⚙️ Encoding dataset...")

    # ✅ Load the trained SentencePiece model
    sp = spm.SentencePieceProcessor(model_file=f"{spm_model_prefix}.model")

    def encode_file(in_file, out_file):
        with open(in_file, encoding="utf-8") as fin, open(out_file, "w", encoding="utf-8") as fout:
            for line in fin:
                fout.write(" ".join(sp.encode(line.strip(), out_type=str)) + "\n")

    # ✅ Encode all official splits (no custom re-split)
    for split in ["train", "validation", "test"]:
        encode_file(os.path.join(DATA_PATH, f"{split}.en"), os.path.join(DATA_PATH, f"{split}.spm.en"))
        encode_file(os.path.join(DATA_PATH, f"{split}.zh"), os.path.join(DATA_PATH, f"{split}.spm.zh"))
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  6️⃣ ======== Use official train/validation/test splits =========================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print(f"✅ Using official IWSLT17 train/validation/test splits.")

    print(f"✅ Encoded EN→ZH data saved under: {DATA_PATH}")
    print(f"✅ Next: run fairseq-preprocess to binarize (on *.spm.en/*.spm.zh).")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 3.2. PROCESSING DATASET 2 | XXX ----- FAIRSEQ PREPROCESSING & BINARIZATION |------####################
########################################################################################################################
########################################################################################################################
####-------| NOTE 3.2. PROCESSING DATASET 2 | XXX ------------| ENGLISH→CHINESE |--------------------####################
########################################################################################################################

"""
Stage 2:
Preprocess the tokenized English→Chinese dataset using fairseq-preprocess
to build vocabularies and generate Fairseq binary training files.
"""

# ─────────────────────────────────────────────────────────────────────────────────────────────────
print(
    f"🔎🔎 run_fairseq_preprocessing: {exp_args.run_fairseq_preprocessing} "
    f"{'(RUNNING)✔️✔️' if exp_args.run_fairseq_preprocessing else '(SKIPPED)❌❌'}"
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if exp_args.run_fairseq_preprocessing:
        
    import subprocess


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  1️⃣ ======== Check dataset files before preprocessing ==========================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    # ✅ Same folder you used above
    DATA_DIR = r"C:\Users\emeka\Research\ModelCUDA\Big_Data_Journal\Comparison\Code\Paper\github3\IWSLT\Transformer_Baselines_EN_ZH\Dataset"
    DESTDIR  = r"data-bin\iwslt17.en-zh"  # name it how you like
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    required_files = [
        os.path.join(DATA_DIR, "train.spm.en"),
        os.path.join(DATA_DIR, "train.spm.zh"),
        os.path.join(DATA_DIR, "validation.spm.en"),
        os.path.join(DATA_DIR, "validation.spm.zh"),
        os.path.join(DATA_DIR, "test.spm.en"),
        os.path.join(DATA_DIR, "test.spm.zh"),
    ]

    for f in required_files:
        assert os.path.exists(f), f"❌ Missing file: {f}"

    print("✅ All SentencePiece files found.")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    
    
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  2️⃣ ======== Configure preprocessing ============================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print("🚀 Running fairseq-preprocess for EN→ZH...")

    cmd = [
        "fairseq-preprocess",
        "--source-lang", "en",             # ✅Source = English
        "--target-lang", "zh",             # ✅Target = Chinese
        # use the SPM-encoded prefixes you just created
        "--trainpref", os.path.join(DATA_DIR, "train.spm"),
        "--validpref", os.path.join(DATA_DIR, "validation.spm"),
        "--testpref",  os.path.join(DATA_DIR, "test.spm"),
        "--destdir", DESTDIR,
        "--workers", "8"
    ]
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  3️⃣ ======== Run preprocessing ==================================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    subprocess.run(cmd, check=True)
    print(f"✅ Preprocessing complete! Binarized EN→ZH data saved in {DESTDIR}")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 4. FAIRSEQ SETUP, ACTIVATION PATCHING | XXX --------------------------------------####################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ========  4.1 Device Configuration =========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  4.2 Custom Activation Registration & Patching ====================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# Ensure activation folder is in sys.path
if ACTIVATION_PATH not in sys.path:
    sys.path.append(ACTIVATION_PATH)

# Backup original get_activation_fn
orig_get_activation_fn = utils.get_activation_fn
# ────────────────────────────────────────────────────────────────────────────────────────────────
def custom_get_activation_fn(activation: str):
    activation = activation.lower()
    if activation == "tanhexp":
        from activation.TanhExp import TanhExp
        print("✅ Registered custom activation: TanhExp")
        return TanhExp()
    
    elif activation == "fftgate":
        from activation.FFTGate import FFTGate
        print("✅ Registered custom activation: FFTGate")
        return FFTGate()
    
    elif activation == "geglu":        
        from activation.GEGLU import GEGLU
        print("✅ Registered custom activation: GEGLU")
        return GEGLU()

    else:
        # ✅ fallback to original fairseq activations
        return orig_get_activation_fn(activation)
# ────────────────────────────────────────────────────────────────────────────────────────────────
# Patch fairseq utils
utils.get_activation_fn = custom_get_activation_fn

# ────────────────────────────────────────────────────────────────────────────────────────────────
print(f"⚡ Using activation function: {exp_args.act_name.lower()}")

# Test-call the patched activation to confirm it works
try:
    act_fn = utils.get_activation_fn(exp_args.act_name.lower())
    print(f"🔍 Activation function resolved: {act_fn}")
except Exception as e:
    print(f"❌ Failed to resolve activation '{exp_args.act_name.lower()}': {e}")
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Manually inject custom activation into config
cfg.model.activation_fn = exp_args.act_name.lower()
print(f"⚡ Overwrote cfg.model.activation_fn = {cfg.model.activation_fn}")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ========  4.3 Fairseq Task & Dataset Initialization ========================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Load dictionaries
src_dict = Dictionary.load(os.path.join(fairseq_args.data, f"dict.{fairseq_args.source_lang}.txt"))
tgt_dict = Dictionary.load(os.path.join(fairseq_args.data, f"dict.{fairseq_args.target_lang}.txt"))

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Setup task
task = TranslationTask.setup_task(cfg.task)
task.load_dataset("train")
task.load_dataset("test")
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ✅✅ Print translation direction (EN→ZH confirmation)
print(f"🌍 Translation direction: {fairseq_args.source_lang} → {fairseq_args.target_lang}")
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ✅✅ Inspect dataset lengths after loading
train_dataset = task.dataset("train")
test_dataset  = task.dataset("test")
# ────────────────────────────────────────────────────────────────────────────────────────────────

print(f"📏 Train set - Max source length: {train_dataset.src_sizes.max()}, Max target length: {train_dataset.tgt_sizes.max()}")
print(f"📏 Test set  - Max source length: {test_dataset.src_sizes.max()}, Max target length: {test_dataset.tgt_sizes.max()}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Inspect data you’d lose if you lower max_positions
def check_percentage(dataset, N):
    src_long = (dataset.src_sizes > N).sum()
    tgt_long = (dataset.tgt_sizes > N).sum()
    total = len(dataset)

    print(f"Dataset size: {total}")
    print(f"  > {N} tokens (source): {src_long} ({100*src_long/total:.2f}%)")
    print(f"  > {N} tokens (target): {tgt_long} ({100*tgt_long/total:.2f}%)")
    print()

# ✅✅ Check on train + test
check_percentage(train_dataset, exp_args.max_source_positions)
check_percentage(test_dataset, exp_args.max_source_positions)
# ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 5. MODEL INITIALIZATION | XXX ----------------------------------------------------####################
########################################################################################################################

# ================================================================================================
# 🏷️========================= MODEL INITIALIZATION===============================================
# ================================================================================================
# ================================================================================================
####------------------ 0️⃣ 1️⃣ 2️⃣ 3️⃣ 4️⃣ 5️⃣ 6️⃣ 7️⃣ 8️⃣ 9️⃣ ------------------------------------####



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  ======== 🔖🔖  5.1 Model Construction 🔖🔖====================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 


# ================================================================================================
# 1️⃣ 🧠 Build Fairseq Baseline Transformer
# ================================================================================================
# ================================================================================================
# 1️⃣ 🧠 Build Fairseq Baseline Transformer
# ================================================================================================

if exp_args.net == "Transformer":

    model = task.build_model(cfg.model)

    print(f"✅ Loaded Model: {model.__class__.__name__}")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ================================================================================================
# 2️⃣ 🧠 Build RTM_Former 📐🔑🏷️
# ================================================================================================
# ================================================================================================
# 2️⃣ 🧠 Build RTM_Former 📐🔑🏷️
# ================================================================================================
elif exp_args.net == "RTM_Former":


    rtm_layers = []

    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 1️⃣ 🧩🧠 === FFN Insertion === ♻️♻️ ========================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 1️⃣ 🧩🧠 === FFN Insertion === ♻️♻️ ========================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    if exp_args.rtm_insertion_type == "ffn":

        # ================================================================================================
        # 1️⃣.0️⃣ 🧠 Build RTM_Former  
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:ffn🧬🧬")
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:ffn🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print("\nMODEL INIT CHECK")
        print(model.decoder.layers[0].fc1.weight[0, :10])
        print()
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────────────────────────────────       
        # =============================================================================
        # 1️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder FFNs 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i >= len(model.decoder.layers) - exp_args.rtm_num_layers:

                original_activation_fn = layer.activation_fn

                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,
                    rtm_rotation=exp_args.rtm_rotation                 
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:

                    # 🔒 Configure RTM coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Activation Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ───────────────────────────────────────────────────────────────────────────────────────────────── 
                # ─────────────────────────────────────────────────────────────────────────────────────────────────           


                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                def wrapped_activation(
                    x,
                    act_fn=original_activation_fn,
                    rtm_module=rtm
                ):

                    # ==========================================================
                    # 1️⃣.2️⃣.1️⃣ Standard FFN activation
                    # ==========================================================
                    x = act_fn(x)

                    # ==========================================================
                    # 1️⃣.2️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 1️⃣.2️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm_module.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0 + exp_args.rtm_transition_amplification * transition_scale
                    # )

                    # x = x * (
                    #     1.0 + rtm_module.amplification * transition_scale
                    # )

                    global RTM_SCALE

                    x_base = x

                    x_mod = x * (
                        1.0 + rtm_module.amplification * transition_scale
                    )

                    x_mod = rtm_module(x_mod)
                    # ==========================================================
                    # 1️⃣.2️⃣.6️⃣ RTM refinement
                    # ==========================================================
                    # x = rtm_module(x)
                    x = x_base + RTM_SCALE * (x_mod - x_base)

                    return x
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               

                # =============================================================================
                # 📌📌 Replace decoder activation function
                # =============================================================================                 
                layer.activation_fn = wrapped_activation
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  


                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 FFN Insertion | RTM inserted into decoder layer {i}")                
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────




    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣ 🧩🧠 === Cross-Attention Insertion ♻️♻️ ================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣ 🧩🧠 === Cross-Attention Insertion ♻️♻️ ================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────

    elif exp_args.rtm_insertion_type == "cross_attn":
    
        # ================================================================================================
        # 2️⃣.0️⃣ 🧠 Build RTM_Former 🅰️🔼 
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:cross_attn🧬🧬")
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:cross_attn🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder Cross-Attention 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i >= len(model.decoder.layers) - exp_args.rtm_num_layers:

                # =============================================================================
                # 2️⃣.2️⃣.1️⃣📌📌 RTM module
                # =============================================================================
                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,
                    rtm_rotation=exp_args.rtm_rotation                    
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:

                    # 🔒 Configure RTM coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Attention Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────


                # =============================================================================
                # 2️⃣.2️⃣.2️⃣📌📌 Store original encoder-attention forward
                # =============================================================================
                original_encoder_attn_forward = layer.encoder_attn.forward

                # =============================================================================
                # 2️⃣.2️⃣.3️⃣📌📌 Wrapped Cross-Attention forward
                # =============================================================================
                def wrapped_encoder_attn_forward(
                    query,
                    key,
                    value,
                    **kwargs
                ):

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.1️⃣ Original Cross-Attention
                    # ==========================================================
                    x, attn = original_encoder_attn_forward(
                        query=query,
                        key=key,
                        value=value,
                        **kwargs
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0
                    #     +
                    #     exp_args.rtm_transition_amplification
                    #     *
                    #     transition_scale
                    # )

                    x = x * (
                        1.0
                        +
                        rtm.amplification
                        *
                        transition_scale
                    )

                    return x, attn
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────


                # =============================================================================
                # 📌📌 Replace encoder-attention forward
                # =============================================================================
                layer.encoder_attn.forward = wrapped_encoder_attn_forward
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────

                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 Cross-Attention Insertion | RTM inserted into decoder layer {i}") 

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────



    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 3️⃣ 🧩🧠 === FFN Depth-Weighted Insertion === ♻️♻️ =========================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 3️⃣ 🧩🧠 === FFN Depth-Weighted Insertion === ♻️♻️ =========================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    elif exp_args.rtm_insertion_type == "ffn_depth_weighted":

        # ================================================================================================
        # 1️⃣.0️⃣ 🧠 Build RTM_Former  
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: Depth-Weighted RTM_Former for insertion_typ:ffn_depth_weighted🧬🧬")
        print(f"♻️♻️ Loaded Model: Depth-Weighted RTM_Former for insertion_typ:ffn_depth_weighted🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # 🔒 IMPORTANT: reset RNG after importing RTM_Former, before the real training model is built (For Repeatable Initialization)
        set_seed_torch(seed1)
        set_seed_main(seed2)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print("\nMODEL INIT CHECK")
        print(model.decoder.layers[0].fc1.weight[0, :10])
        print()
        # ==========================================================
        # 🚦🔍 Select layers
        # ========================================================== 
        selected_layers = list(
            range(
                len(model.decoder.layers) - exp_args.rtm_num_layers,
                len(model.decoder.layers)
            )
        )
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────────────────────────────────       
        # =============================================================================
        # 1️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder FFNs 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i in selected_layers:

                original_activation_fn = layer.activation_fn

                layer_rank = selected_layers.index(i)


                # =====================================================
                # ⏭️⏭️ Forward direction Layer scale ⏭️⏭️
                # =====================================================               
                if exp_args.rtm_layer_scale_direction == "forward":
                    layer_scale = exp_args.rtm_layer_scale_min + (
                        1.0 - exp_args.rtm_layer_scale_min
                    ) * (
                        layer_rank / max(1, exp_args.rtm_num_layers - 1)
                    )
                # =====================================================
                # ⏪⏪ Reverse direction Layer scale ⏪⏪
                # =====================================================  
                elif exp_args.rtm_layer_scale_direction == "reverse":
                    layer_scale = 1.0 - (
                        1.0 - exp_args.rtm_layer_scale_min
                    ) * (
                        layer_rank / max(1, exp_args.rtm_num_layers - 1)
                    )
                # =====================================================
                # ⏪🔒 Reverse direction + Fixed Final Layer Scale ⏪🔒
                # =====================================================
                elif exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                    if layer_rank == exp_args.rtm_num_layers - 1:
                        layer_scale = exp_args.rtm_final_layer_scale
                    else:
                        layer_scale = 1.0 - (
                            1.0 - exp_args.rtm_layer_scale_min
                        ) * (
                            layer_rank / max(1, exp_args.rtm_num_layers - 2)
                        ) 
                # ────────────────────────────────────────────────────────────────────────────────────────
                else:
                    raise ValueError(f"Unknown RTM layer scale direction: {exp_args.rtm_layer_scale_direction}")
                # ────────────────────────────────────────────────────────────────────────────────────────

                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,  
                    gate_type=exp_args.rtm_gate_type,
                    rtm_rotation=exp_args.rtm_rotation                                     
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility + Parameter Tracking)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:
                    
                    # 🔒 Configure RTM learnable coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Activation Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ─────────────────────────────────────────────────────────────────────────────────────────────────                
                # ─────────────────────────────────────────────────────────────────────────────────────────────────

                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                def wrapped_activation(
                    x,
                    act_fn=original_activation_fn,
                    rtm_module=rtm,
                    layer_scale=layer_scale
                ):

                    # ==========================================================
                    # 1️⃣.2️⃣.1️⃣ Standard FFN activation
                    # ==========================================================
                    x = act_fn(x)

                    # ==========================================================
                    # 1️⃣.2️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 1️⃣.2️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm_module.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0 + exp_args.rtm_transition_amplification * transition_scale
                    # )

                    # x = x * (
                    #     1.0 + rtm_module.amplification * transition_scale
                    # )

                    global RTM_SCALE

                    x_base = x

                    x_mod = x * (
                        1.0 + rtm_module.amplification * transition_scale
                    )

                    x_mod = rtm_module(x_mod)
                    # ==========================================================
                    # 1️⃣.2️⃣.6️⃣ RTM refinement
                    # ==========================================================

                    # ----------------------------------------------------------
                    # 🏔️ Depth-Aware RTM Scaling (Layer-Adaptive Modulation)
                    # ---------------------------------------------------------- 
                    if exp_args.rtm_depth_weight:

                        x = x_base + RTM_SCALE * layer_scale * (x_mod - x_base)

                    # ----------------------------------------------------------
                    # ⚖️ Uniform RTM Scaling (Same Modulation Across Layers)
                    # ----------------------------------------------------------
                    else:

                        x = x_base + RTM_SCALE * (x_mod - x_base)
                    # ─────────────────────────────────────────────────────────────────────────────────────────────────

                    return x
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               

                # =============================================================================
                # 📌📌 Replace decoder activation function
                # =============================================================================                 
                layer.activation_fn = wrapped_activation
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  


                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 FFN Insertion | RTM inserted into decoder layer {i} | layer_scale={layer_scale:.3f}")              
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: Depth-Weighted RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    else:
        raise ValueError(f"Unknown RTM insertion type: {exp_args.rtm_insertion_type}")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print(f"📣📣 RTM_Former for insertion_typ={exp_args.rtm_insertion_type} | Loaded Model={exp_args.net }♻️♻️")
    print(f"📣📣 RTM_Former for insertion_typ={exp_args.rtm_insertion_type} | Loaded Model={exp_args.net }♻️♻️")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────



    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  ======== 🔖  5.1.1 Model Configuration Hyperparameter Summary ==================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 
    if exp_args.net == "RTM_Former":

        rtm_log = (
            f"📣 RTM Configuration             : "
            f"alpha={exp_args.rtm_alpha} | "
            f"amp={exp_args.rtm_transition_amplification} | "
            f"layers={exp_args.rtm_num_layers} | "
            f"insertion={exp_args.rtm_insertion_type} | "
            f"scale_init={exp_args.rtm_scale} | "
            f"start_epoch={exp_args.rtm_start_epoch} | "
            f"full_epoch={exp_args.rtm_full_epoch}"
        )

        # Depth-weighted RTM only
        if exp_args.rtm_insertion_type == "ffn_depth_weighted":

            rtm_log += (
                f" | layer_scale_min={exp_args.rtm_layer_scale_min}"
                f" | layer_direction={exp_args.rtm_layer_scale_direction}"
                f" | gate={exp_args.rtm_gate_type}"
            )

            # Only exists for reverse_final_fixed
            if exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                rtm_log += (
                    f" | final_layer_scale={exp_args.rtm_final_layer_scale}"
                )

        print(rtm_log)


    else:
        print(
            f"📣 Baseline Configuration        : "
            f"model={exp_args.net}"
        )
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ======== 🔖🔖  5.2  Insert model to device 🔖🔖================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

model = model.to(DEVICE)
 # ─────────────────────────────────────────────────────────────────────────────────────────────────
print(f"🚀🚀 Model moved to device: {DEVICE} ♻️♻️")
print(f"🚀🚀 Model moved to device: {DEVICE} ♻️♻️")
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ======== 🔖🔖 5.3  Model Validation & Sanity Checks 🔖🔖=======================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 🔍 Print custom activations to confirm they are used
for name, module in model.named_modules():
    if isinstance(module, nn.Module) and module.__class__.__name__ in ["TanhExp", "FFTGate"]:
        print(f"🔎 Found custom activation in {name}: {module}")


# ✅ Sanity check Device
if DEVICE.type == "cuda":
    print(f"🚀 Model moved to CUDA: {torch.cuda.get_device_name(DEVICE)}")
else:
    print("⚠️ Using CPU (CUDA not available)")
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 6. PATH DEFINATION | XXX --------------------------------------------------------#####################
########################################################################################################################


# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 1️⃣ 📌📌 ======== ENSURE DIRECTORY EXIST ========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# # ─────────────────────────────────────────────────────────────────────────────────────────────────
# # 🟡 === Checkpoint directories ===
# if not os.path.exists('checkpoint'):
#     os.makedirs('checkpoint')
# # ─────────────────────────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Results directories ===
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}')

if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === LR Log directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Training Log directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Model Summary directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')
# ────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣ 📌📌 ======== DEFINE DIRECTORY ==============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.1️⃣🔖 ==== Checkpoint Directory Selection | Normal Run vs Crash Run | save every epoch ===== 
# ────────────────────────────────────────────────────────────────────────────────────────────────

# 🅰️💾 Normal run: save checkpoints in standard folder
if not exp_args.crash_run:
    checkpoint_dir = f'./checkpoints/{exp_args.dataset_name}/{exp_args.net}/'

    # 📁 Create checkpoint folder if it does not exist
    os.makedirs(checkpoint_dir, exist_ok=True)

    print(f"🅰️💾 Normal Run Enabled | Saving checkpoints to: {checkpoint_dir}")
#---------------------------------------------------------------------------------------------------
# 🅱️♻️ Crash run: save checkpoints in recovery folder
else:
    checkpoint_dir = f'./checkpoints_crash_run/{exp_args.dataset_name}/{exp_args.net}/'

    # 📁 Create checkpoint folder if it does not exist    
    os.makedirs(checkpoint_dir, exist_ok=True)

    print(f"🅱️♻️ Crash Run Enabled | Saving recovery checkpoints to: {checkpoint_dir}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.2️⃣🔖 === Main Test & Train Results === 
train_results_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/Train_{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
test_results_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/Test_{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.3️⃣🔖  === LR, Training & Summary logs  === 
LR_save_paths = {
    "LR_history": f"./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_LR_log.txt"
}
save_paths = {
    "log_history": f"./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_training_logs.txt"
}
# ────────────────────────────────────────────────────────────────────────────────────────────────

# 2️⃣.4️⃣🔖 ===  Model summary path === 
model_summary_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_model_summary.txt'
model_summary_full_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_model_summary_full.txt'
# ────────────────────────────────────────────────────────────────────────────────────────────────

# # ✅🔖 === Path template for saving checkpoints at every epoch | Part2 (if scratch occured) ===
# checkpoint_dir = f'./checkpoints/{exp_args.dataset_name}/tanhexp_AfterCrash_41_49epochs/'
# os.makedirs(checkpoint_dir, exist_ok=True)
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 7. TRAINING INITIALIZATION & OPTIMIZATION SETUP | XXX ---------------------------#####################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ========  7.1 AMP GradScaler ===============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Initialize AMP GradScaler once globall
scaler = torch.cuda.amp.GradScaler()
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  Training Criterion ===============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

criterion = LabelSmoothingCrossEntropy(smoothing=exp_args.smoothing)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ========   Optimizer Configuration =========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ♻️ Adam optimizer (β1=0.9, β2=0.98, eps=1e-9) ♻️
# ─────────────────────────────────────────────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=exp_args.lr,
    betas=(0.9, 0.98),
    eps=exp_args.eps,
    weight_decay=exp_args.weight_decay,
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 4️⃣ ========  Learning Rate Scheduler ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ✅  Transformer LR schedule (warmup + inverse sqrt decay)
# --------------------------

warmup_updates = exp_args.warmup_updates   # configurable warmup steps
peak_lr = exp_args.lr

# ────────────────────────────────────────────────────────────────────────────────────────────────
def lr_lambda(step):
    step = max(1, step)
    if step == 1:
        print(f"🚀 Warmup started at step {step} (target peak LR = {peak_lr}).")
    if step == warmup_updates:
        actual_lr = optimizer.param_groups[0]["lr"]
        print(f"✅ Warmup finished at step {step}. Peak LR reached: {actual_lr:.8f}")
    if step == warmup_updates + 1:
        print(f"📉 Inverse square root decay started at step {step}.")
    
    if step <= warmup_updates:
        # Linear warmup
        scale = step / warmup_updates
    else:
        # Inverse square root decay
        scale = (warmup_updates ** 0.5) / (step ** 0.5)
    return scale

# ────────────────────────────────────────────────────────────────────────────────────────────────

lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 5️⃣ ========  Global Tracking Variables ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ global initialization
best_bleu = 0.0
total_test_duration = 0.0
RTM_SCALE_LOG = ""
# ─────────────────────────────────────────────────────────────────────────────────────────────────

Python: 3.10.16 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:19:12) [MSC v.1929 64 bit (AMD64)]
Torch: 2.2.0+cu121 | CUDA available: True
Fairseq: 0.12.2
OmegaConf: 2.0.6
Hydra-Core: 1.0.7
✅ Current working directory: C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH
✅ sys.path updated:
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\python310.zip
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\DLLs
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env
   📂 
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\win32
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\win32\lib
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\Pythonwin
   📂 C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH
   📂 C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH\models
   📂 C:\

c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\modules\module.py:1150: UserWarning: expandable_segments not supported on this platform (Triggered internally at ..\c10/cuda/CUDAAllocatorConfig.h:30.)
  return t.to(device, dtype if t.is_floating_point() or t.is_complex() else None, non_blocking)


In [ ]:
########################################################################################################################
####-------| NOTE 11. TEST ALL CHECKPOINTS (BLEU per epoch)  | XXX -----------------------------########################
########################################################################################################################
########################################################################################################################
####-------| NOTE 11. TEST ALL CHECKPOINTS (BLEU per epoch)  | XXX -----------------------------########################
########################################################################################################################


# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ======== Paths for checkpoint BLEU sweeps ===================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Paths for checkpoint BLEU sweeps
test_checkpoint_results_path = (
    f'./Results/{exp_args.dataset_name}/{exp_args.net}/'
    f'TestCheckpoints_{exp_args.net}_'
    f'{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
)
# ────────────────────────────────────────────────────────────────────────────────────────────────


# # ✅ Paths for checkpoint BLEU sweeps | Part2 (if Cratch occured)
# test_checkpoint_results_path = (
#     f'./Results/{exp_args.dataset_name}/{exp_args.act_name}/'
#     f'TestCheckpoints_{exp_args.act_name}_{exp_args.net}_'
#     f'{exp_args.dataset_name}_{exp_args.optimizer1}_tanhexp_AfterCrash_41_49epochs.txt'
# )
# ────────────────────────────────────────────────────────────────────────────────────────────────


# # ─────────────────────────────────────────────────────────────────────────────────────────────────
# #  2️⃣ ========  BLEU Detokenization (🔖 EN/FR/DE) ================================================
# # ───────────────────────────────────────────────────────────────────────────────────────────────── 
# # ────────────────────────────────────────────────────────────────────────────────────────────────
# def detokenize_for_bleu(s: str) -> str:
#     s = s.replace("@@ ", "").replace("@@", "")  # BPE
#     s = s.replace("▁", " ")                     # SentencePiece
#     s = " ".join(s.split())                     # normalize spaces
#     s = re.sub(r"\s+([?.!,])", r"\1", s)        # punctuation
#     return s.strip()
# # ────────────────────────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  BLEU Detokenization (🔖 Chinese) =================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
def detokenize_for_bleu(s: str) -> str:
    # For Chinese, don't remove SentencePiece boundaries or spaces | Only remove BPE continuation markers
    # sacreBLEU's "zh" tokenizer expects raw characters.
    return s.replace("@@ ", "").replace("@@", "").strip()
# ────────────────────────────────────────────────────────────────────────────────────────────────




# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ======== Test_Checkpoint Function ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ test_checkpoint function
def test_checkpoint_epochs(task, model, criterion, checkpoint_dir, test_checkpoint_results_path):
    """
    Evaluate BLEU (beam=5) for all checkpoints in checkpoint_dir.
    Save results to test_checkpoint_results_path in the same format as normal test logs.
    """
# ────────────────────────────────────────────────────────────────────────────────────────────────
    global RTM_SCALE, RTM_SCALE_LOG
    # ────────────────────────────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Collect checkpoints sorted by epoch number in filename
    def extract_epoch_num(filename):
        match = re.search(r"epoch_(\d+)", filename)
        return int(match.group(1)) if match else float("inf")  # inf if no epoch number

    ckpts = sorted(
        glob.glob(os.path.join(checkpoint_dir, "*.pt")),
        key=lambda x: extract_epoch_num(os.path.basename(x))
    )

    if not ckpts:
        print(f"⚠️ No checkpoints found in {checkpoint_dir}")
        return

    # # ────────────────────────────────────────────────────────────────────────────────────────────────
    # # ✅ Reset output file
    # os.makedirs(os.path.dirname(test_checkpoint_results_path), exist_ok=True)
    # with open(test_checkpoint_results_path, "w", encoding="utf-8") as f:
    #     f.write("")
    # # ────────────────────────────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 📦 Reset output file + write model status
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    #1️⃣ Create output folder if it does not exist
    os.makedirs(os.path.dirname(test_checkpoint_results_path), exist_ok=True)

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Normal run: reset file
    # ✅ Crash run : append to existing file
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣♻️ Normal run: reset file
    if not exp_args.crash_run:

        with open(test_checkpoint_results_path, "w", encoding="utf-8") as f:
            f.write("")

    # 3️⃣⏭️ Crash run : append to existing file
    else:

        with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
            f.write("\n♻️ Resuming checkpoint testing after crash\n\n")
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 4️⃣✍️  Write test result header (model information)
    with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
        f.write("")

        # ---------------------------------------------------------------------------
        # 4️⃣.1️⃣🏷️ Testing Model Status
        # ---------------------------------------------------------------------------
        if exp_args.net == "RTM_Former":

            f.write(
                f"Model: RTM_Former\n\n"
            )

        else:

            f.write(
                f"Model: Baseline Fairseq Transformer\n\n"
            )        
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ────────────────────────────────────────────────────────────────────────────────────────────────



    # ────────────────────────────────────────────────────────────────────────────────────────────────
    best_bleu = 0.0
    total_test_duration = 0.0
    # ────────────────────────────────────────────────────────────────────────────────────────────────


    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    for ckpt in ckpts:

        # # ────────────────────────────────────────────────────────────────────────────────
        # # ✅ 📌📌 Load checkpoint (OLD) 🅰️🔼 
        # # ────────────────────────────────────────────────────────────────────────────────        
        # print(f"🔄 Loading checkpoint: {ckpt}")
        # state = torch.load(ckpt, map_location=DEVICE)
        # model.load_state_dict(state["model_state"], strict=False)
        # model.to(DEVICE)
        # model.eval()
        # --------------------------------------------------------------------------------------------------
        # ────────────────────────────────────────────────────────────────────────────────
        # ✅📌📌 Load checkpoint (NEW) 🅱️🔼 Restore RTM schedule for checkpoint testing using same strategy as main loop
        # ────────────────────────────────────────────────────────────────────────────────
        if exp_args.net == "Transformer":
            print(f"🔄 Loading checkpoint Baseline Transformer : {ckpt}")
            state = torch.load(ckpt, map_location=DEVICE)
            model.load_state_dict(state["model_state"], strict=False)
            model.to(DEVICE)
            model.eval()
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        elif exp_args.net == "RTM_Former":
            print(f"🔄 Loading checkpoint RTM_Former : {ckpt}")
            state = torch.load(ckpt, map_location=DEVICE)
            model.load_state_dict(state["model_state"], strict=False)            
            

            RTM_START_EPOCH = exp_args.rtm_start_epoch
            RTM_FULL_EPOCH = exp_args.rtm_full_epoch
            ckpt_epoch = state["epoch"]

            if ckpt_epoch < RTM_START_EPOCH:
                RTM_SCALE = exp_args.rtm_scale
            else:
                progress = min(
                    1.0,
                    (ckpt_epoch - RTM_START_EPOCH) / (RTM_FULL_EPOCH - RTM_START_EPOCH)
                )

                RTM_SCALE = 0.5 * (1.0 - math.cos(math.pi * progress))

            print(f"📐 Checkpoint RTM_SCALE for epoch {ckpt_epoch}: {RTM_SCALE:.3f}")

            if exp_args.rtm_insertion_type == "ffn_depth_weighted":

                if exp_args.rtm_layer_scale_direction == "forward":
                    depth_scales = [
                        exp_args.rtm_layer_scale_min
                        + (1.0 - exp_args.rtm_layer_scale_min)
                        * (j / max(1, exp_args.rtm_num_layers - 1))
                        for j in range(exp_args.rtm_num_layers)
                    ]

                elif exp_args.rtm_layer_scale_direction == "reverse":
                    depth_scales = [
                        1.0
                        - (1.0 - exp_args.rtm_layer_scale_min)
                        * (j / max(1, exp_args.rtm_num_layers - 1))
                        for j in range(exp_args.rtm_num_layers)
                    ]

                elif exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                    depth_scales = []

                    for j in range(exp_args.rtm_num_layers):
                        if j == exp_args.rtm_num_layers - 1:
                            scale_j = exp_args.rtm_final_layer_scale
                        else:
                            scale_j = (
                                1.0
                                - (1.0 - exp_args.rtm_layer_scale_min)
                                * (j / max(1, exp_args.rtm_num_layers - 2))
                            )

                        depth_scales.append(scale_j)

                else:
                    raise ValueError(
                        f"Unknown RTM layer scale direction: {exp_args.rtm_layer_scale_direction}"
                    )

                final_scales = [RTM_SCALE * s for s in depth_scales]

                RTM_SCALE_LOG = f" | RTM_SCALE={RTM_SCALE:.3f}"
                RTM_SCALE_LOG += (
                    f" | RTM_layer_scales({exp_args.rtm_layer_scale_direction})="
                    f"{[round(s, 3) for s in final_scales]}"
                )

                print(
                    f"📐 Checkpoint final RTM layer scales "
                    f"({exp_args.rtm_layer_scale_direction}): "
                    f"{[round(s, 3) for s in final_scales]}"
                )
            model.to(DEVICE)
            model.eval()               
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        else:
            raise ValueError(f"Unknown model type: {exp_args.net}")
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ────────────────────────────────────────────────────────────────────────────────────────────────


        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Build test iterator
        test_iter = task.get_batch_iterator(
            dataset=task.dataset("test"),
            max_tokens=fairseq_args.max_tokens,   # cut in half option: fairseq_args.max_tokens // 2
            max_positions=(exp_args.max_source_positions, exp_args.max_target_positions),
            seed=seed1,
            num_shards=exp_args.num_shards,
            shard_id=exp_args.shard_id,
            num_workers=exp_args.num_workers,
            ignore_invalid_inputs=exp_args.ignore_invalid_inputs,
        ).next_epoch_itr(shuffle=False)
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ References
        true_references = []
        for sample in test_iter:
            targets = sample["target"]
            for t in targets:
                t_no_pad = t[t != tgt_dict.pad()]
                ref_str = tgt_dict.string(t_no_pad, bpe_symbol="@@")
                true_references.append(detokenize_for_bleu(ref_str))
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Rebuild iterator for decoding
        test_iter = task.get_batch_iterator(
            dataset=task.dataset("test"),
            max_tokens=fairseq_args.max_tokens,    # cut in half option: fairseq_args.max_tokens // 2
            max_positions=(exp_args.max_source_positions, exp_args.max_target_positions),
            seed=seed1,
            num_shards=exp_args.num_shards,
            shard_id=exp_args.shard_id,
            num_workers=exp_args.num_workers,
            ignore_invalid_inputs=exp_args.ignore_invalid_inputs,
        ).next_epoch_itr(shuffle=False)
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Build generator with beam=5
        gen_cfg = deepcopy(cfg.generation)
        gen_cfg.beam = exp_args.beam         # 🎀🔖 5
        gen_cfg.lenpen = exp_args.lenpen     # 🎀🔖 1.0 
        generator = task.build_generator([model], gen_cfg)
        # ────────────────────────────────────────────────────────────────────────────────────────────────


 

        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Run evaluation
        sys_outputs, test_loss = [], 0.0
        start_time = time.time()

        with torch.no_grad():
            with tqdm(enumerate(test_iter), total=len(test_iter), desc=f"Testing {os.path.basename(ckpt)}") as progress:
                for i, sample in progress:   # 👈 keep i!
                    sample = utils.move_to_cuda(sample) if torch.cuda.is_available() else sample
                    src_tokens = sample["net_input"]["src_tokens"]
                    prev_output_tokens = sample["net_input"]["prev_output_tokens"]
                    target = sample["target"]
                    src_lengths = (src_tokens != src_dict.pad()).sum(dim=1)

                    with torch.cuda.amp.autocast():
                        logits, _ = model(src_tokens, src_lengths, prev_output_tokens)
                        pad = tgt_dict.pad()
                        logits2d = logits.view(-1, logits.size(-1))
                        target1d = target.view(-1)
                        mask = target1d != pad
                        loss = criterion(logits2d[mask], target1d[mask]) if mask.any() else logits2d.sum() * 0.0
                    test_loss += loss.item()

                    hypos = task.inference_step(generator, [model], sample)
                    for j, hypos_j in enumerate(hypos):
                        hypo_tokens = [tok for tok in hypos_j[0]["tokens"].tolist() if tok != tgt_dict.pad()]
                        sys_str = tgt_dict.string(hypo_tokens, bpe_symbol="@@")
                        sys_outputs.append(detokenize_for_bleu(sys_str))

                    # ✅ Update progress bar postfix with running average
                    progress.set_postfix(loss=round(test_loss / (i + 1), 3))
       # ────────────────────────────────────────────────────────────────────────────────────────────────




       # ────────────────────────────────────────────────────────────────────────────────────────────────
        avg_loss = test_loss / max(1, len(test_iter))
        # bleu = sacrebleu.corpus_bleu(sys_outputs, [true_references])                 #🔖 EN/FR/DE
        bleu = sacrebleu.corpus_bleu(sys_outputs, [true_references], tokenize="zh")    #🔖 Chinese

        duration = time.time() - start_time
        total_test_duration += duration
       # ────────────────────────────────────────────────────────────────────────────────────────────────


       # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Write results
        with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
            f.write(f"Checkpoint {os.path.basename(ckpt)} | Test Loss: {avg_loss:.3f} | sacreBLEU: {bleu.score:.2f} | Beam=5\n")

        print(f"✅ {os.path.basename(ckpt)} | Loss {avg_loss:.3f} | sacreBLEU {bleu.score:.2f}")


        # 🧹 add visual spacing between runs
        tqdm.write("")  
       # ────────────────────────────────────────────────────────────────────────────────────────────────

       # ────────────────────────────────────────────────────────────────────────────────────────────────
        if bleu.score > best_bleu:
            best_bleu = bleu.score

    with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
        f.write(f"\n🏆 Best BLEU Score: {best_bleu:.2f}\n")
        total_mins, total_secs = divmod(total_test_duration, 60)
        f.write(f"\n🕒 Total Test Time over all checkpoints: {int(total_mins)} min {total_secs:.2f} sec\n")

    print(f"🏆 Best BLEU over checkpoints: {best_bleu:.2f}")
    # ────────────────────────────────────────────────────────────────────────────────────────────────



# ────────────────────────────────────────────────────────────────────────────────────────────────
# # ✅ Free unused GPU memory BEFORE training starts
# torch.cuda.empty_cache()
# torch.cuda.reset_peak_memory_stats()

# 4️⃣ Call the function
test_checkpoint_epochs(task, model, criterion, checkpoint_dir, test_checkpoint_results_path)
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────────────────────────

🔄 Loading checkpoint RTM_Former : ./checkpoints/IWSLT17_En_Zh/RTM_Former\epoch_0_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 0: 0.000


Testing epoch_0_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt:   0%|          | 0/82 [00:00<?, ?it/s]c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)
c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\functional.py:5109: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
Testing epoch_0_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:54<00:00,  1.51it/s, loss=7.25]


✅ epoch_0_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 7.255 | sacreBLEU 1.40

🔄 Loading checkpoint RTM_Former : ./checkpoints/IWSLT17_En_Zh/RTM_Former\epoch_1_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 1: 0.000


Testing epoch_1_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [11:43<00:00,  8.58s/it, loss=6.59]


✅ epoch_1_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 6.587 | sacreBLEU 1.97

🔄 Loading checkpoint RTM_Former : ./checkpoints/IWSLT17_En_Zh/RTM_Former\epoch_2_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 2: 0.000


...

Testing epoch_7_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [01:32<00:00,  1.12s/it, loss=4.92]


✅ epoch_7_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.920 | sacreBLEU 14.25

🔄 Loading checkpoint RTM_Former : ./checkpoints/IWSLT17_En_Zh/RTM_Former\epoch_8_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 8: 0.206


Testing epoch_8_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [01:14<00:00,  1.10it/s, loss=4.86]


✅ epoch_8_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.858 | sacreBLEU 15.07

🔄 Loading checkpoint RTM_Former : ./checkpoints/IWSLT17_En_Zh/RTM_Former\epoch_9_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 9: 0.345


: 

Restarted pytorch_env (Python 3.10.16)

In [ ]:
#####-------------------------------- NOTE MAIN IWSLT17 EN-ZH NOTE --------------------------------------------------#####
##########################################################################################################################
######################|--------------------------------------------------------------|####################################
###################################🔗 MAIN | TRAIN | TEST LOOP 🔗########################################################
######################|--------------------------------------------------------------|####################################
##########################################################################################################################
#####-------------------------------- NOTE MAIN IWSLT17 EN-ZH NOTE --------------------------------------------------#####



# 📄  main_IWSLT17En_Zh.py
########################################################################################################################
####-------| NOTE 1.1 IMPORTS LIBRARIES  | XXX -------------------------------------------------------##################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 📜 === Enable flexible CUDA memory allocation to reduce fragmentation ===
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ======================================================================================================
# 📜 === Core Libraries ===
# ======================================================================================================
import sys
import numpy as np
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import time
import random
import math
import re
import glob
from copy import deepcopy
import sacrebleu  # ✅ for BLEU evaluation
from timm.loss import LabelSmoothingCrossEntropy
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ======================================================================================================
# 📜 === Fairseq & Supporting Imports ===
# ======================================================================================================
import importlib.metadata as importlib_metadata
import fairseq  # type: ignore
from fairseq.data import Dictionary, data_utils, iterators          # type: ignore
from fairseq.tasks.translation import TranslationTask               # type: ignore
from fairseq import options                                         # type: ignore
from fairseq.dataclass.utils import convert_namespace_to_omegaconf  # type: ignore                                             
from fairseq import utils                                           # type: ignore 
import hydra                                                        # type: ignore  
# ────────────────────────────────────────────────────────────────────────────────────────────────
from ptflops import get_model_complexity_info
from calflops import calculate_flops
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ======================================================================================================
# ♻️ === Print environment summary for sanity check ===
# ======================================================================================================
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Fairseq:", fairseq.__version__)
print("OmegaConf:", importlib_metadata.version("omegaconf"))
print("Hydra-Core:", importlib_metadata.version("hydra-core"))
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 1.2. DEFINE PROJECT PATH | XXX ---------------------------------------------------####################
########################################################################################################################

# ✅ Define working directory
Project_PATH = r"C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH"
if os.getcwd() != Project_PATH:
    os.chdir(Project_PATH)
print(f"✅ Current working directory: {os.getcwd()}")


# ✅ Define absolute paths
PROJECT_PATH = Project_PATH
MODELS_PATH = os.path.join(Project_PATH, "models")
ACTIVATION_PATH = os.path.join(Project_PATH, "activation")


# ✅ Ensure necessary paths are in sys.path
for path in [PROJECT_PATH, MODELS_PATH, ACTIVATION_PATH]:
    if path not in sys.path:
        sys.path.append(path)

# ✅ Print updated sys.path for debugging
print("✅ sys.path updated:")
for path in sys.path:
    print("   📂", path)
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 1.3. OTHER IMPORTS | XXX ---------------------------------------------------------####################
########################################################################################################################

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 📜 ============  Imput parser safe for afno because of --fno-bias, etc  =======================
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 1️⃣✅ Import parser from parser_IWSLT17En_Zh.py
from parser_IWSLT17En_Zh import get_parser

# ✅ Create parser and parse arguments
parser = get_parser()
# args, unknown = parser.parse_known_args()

# ✅ IMPORTANT: Do NOT read Jupyter / VSCode kernel arguments
# This prevents the "--f" ambiguity issue
exp_args = parser.parse_args(args=[])

num_aug_splits = exp_args.aug_splits

print(f"✅ Parser imported successfully | num_aug_splits = {num_aug_splits}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣✅ Import Fairseq parser/config
from fairseq_config_IWSLT17En_Zh import get_fairseq_parser
# ✅ Build Fairseq configuration
fairseq_args, cfg = get_fairseq_parser(exp_args)


num_aug_splits = exp_args.aug_splits

print(f"✅ Parser imported successfully | num_aug_splits = {num_aug_splits}")

print(f"✅ Encoder Embed Dim: {fairseq_args.encoder_embed_dim}")
print(f"✅ Decoder Layers: {fairseq_args.decoder_layers}")
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 3️⃣✅ Import model summary utility
from models.model_summary.model_summary import save_model_summary
from models.model_summary.model_summary_full import save_model_summary_full

print("✅ model_summary.py loaded successfully")
print("✅ model_summary_full.py loaded successfully")
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 2. SEEDING FOR REPRODUCIBILITY | XXX ---------------------------------------------####################
########################################################################################################################

# ────────────────────────────────────────────────────────────────────────────────────────────────
def set_seed_torch(seed):
    torch.manual_seed(seed)
# ────────────────────────────────────────────────────────────────────────────────────────────────
def set_seed_main(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True      # ✅ Default: "False" for Faster, non-deterministic kernels | "True" to Ensure deterministic behavior for CuDNN (Slower)
    torch.backends.cudnn.benchmark = False         # ✅ Default: "True" for Autotune kernels for performance   | "False" Disable CuDNN's autotuning for reproducibility (Slower)

    torch.backends.cuda.matmul.allow_tf32 = False  # ✅ Disable TF32 (strict reproducibility)
    torch.backends.cudnn.allow_tf32 = False        # ✅ Disable TF32 (strict reproducibility)

    # torch.use_deterministic_algorithms(True, warn_only=True)

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ ============= Define Seed =============
seed1, seed2 = exp_args.seed1, exp_args.seed2
set_seed_torch(seed1)  
set_seed_main(seed2)  
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 3.1. PROCESSING DATASET 1 | XXX ---- TOKENIZATION & DATASET PREPARATION |---------####################
########################################################################################################################
########################################################################################################################
####-------| NOTE 3.1. PROCESSING DATASET 1 | XXX ------------| ENGLISH→CHINESE |-------------------####################
########################################################################################################################

"""
Stage 1:
Train SentencePiece tokenizer, encode the raw English→Chinese corpus,
and create aligned train/test text splits for neural machine translation.
"""

# ─────────────────────────────────────────────────────────────────────────────────────────────────
print(
    f"🔎🔎 run_dataset_tokenization: {exp_args.run_dataset_tokenization} "
    f"{'(RUNNING)✔️✔️' if exp_args.run_dataset_tokenization else '(SKIPPED)❌❌'}"
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if exp_args.run_dataset_tokenization:
    from datasets import load_dataset    
    import sentencepiece as spm

    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Path for EN→ZH dataset
    DATA_PATH = r"C:\Users\emeka\Research\ModelCUDA\Big_Data_Journal\Comparison\Code\Paper\github3\IWSLT\Transformer_Baselines_EN_ZH\Dataset"
    os.makedirs(DATA_PATH, exist_ok=True)
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  1️⃣ ======== Download and export official IWSLT17 dataset =======================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Download to your folder (as the Hugging Face cache)
    en_zh = load_dataset(
        "IWSLT/iwslt2017", "iwslt2017-en-zh",
        cache_dir=DATA_PATH,
        trust_remote_code=True
    )
    zh_en = load_dataset(
        "IWSLT/iwslt2017", "iwslt2017-zh-en",
        cache_dir=DATA_PATH,
        trust_remote_code=True
    )  # optional
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  2️⃣ ======== Dataset verification ===============================================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ sanity check (should show ~231k train)
    print(f"EN→ZH dataset: {en_zh}")
    print(f"ZH→EN dataset: {zh_en}")

    print("EN→ZH splits:", list(en_zh.keys()), {k: len(en_zh[k]) for k in en_zh.keys()})
    print("ZH→EN splits:", list(zh_en.keys()), {k: len(zh_en[k]) for k in zh_en.keys()})

    # optional assertions
    assert set(en_zh.keys()) == {"train", "validation", "test"}
    assert set(zh_en.keys()) == {"train", "validation", "test"}
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  3️⃣ ======== Export official train/validation/test splits ======================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Export official IWSLT17 splits to plain text (EN/ZH) under raw_zh_en_dataset_path
    def dump_split(ds_dict, split, src, tgt, out_dir):
        src_fp = os.path.join(out_dir, f"{split}.{src}")
        tgt_fp = os.path.join(out_dir, f"{split}.{tgt}")
        with open(src_fp, "w", encoding="utf-8") as fs, open(tgt_fp, "w", encoding="utf-8") as ft:
            for row in ds_dict[split]:
                tr = row["translation"]
                fs.write((tr[src] or "").strip() + "\n")
                ft.write((tr[tgt] or "").strip() + "\n")
        print(f"Wrote {split}: {src_fp} | {tgt_fp}")

    for split in ["train", "validation", "test"]:
        dump_split(en_zh, split, "en", "zh", DATA_PATH)
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  4️⃣ ========  Train SentencePiece model =========================================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────     

    print("🧠 Training SentencePiece model for EN→ZH...")

    # Use TRAIN split for SPM training
    SRC_EN = os.path.join(DATA_PATH, "train.en")
    TGT_ZH = os.path.join(DATA_PATH, "train.zh")

    spm_model_prefix = os.path.join(DATA_PATH, "spm_enzh")

    spm.SentencePieceTrainer.Train(
        input=f"{SRC_EN},{TGT_ZH}",
        model_prefix=spm_model_prefix,
        vocab_size=10000,
        character_coverage=0.9995,   # ✅ --character_coverage for rich charcater langauage: Japanese or Chinese = 0.9995 | other languages with small character set: 1.0
        model_type="bpe"
    )
    print(f"✅ SentencePiece model saved to: {spm_model_prefix}.model / .vocab")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  5️⃣ ========  Encode files ======================================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    print("⚙️ Encoding dataset...")

    # ✅ Load the trained SentencePiece model
    sp = spm.SentencePieceProcessor(model_file=f"{spm_model_prefix}.model")

    def encode_file(in_file, out_file):
        with open(in_file, encoding="utf-8") as fin, open(out_file, "w", encoding="utf-8") as fout:
            for line in fin:
                fout.write(" ".join(sp.encode(line.strip(), out_type=str)) + "\n")

    # ✅ Encode all official splits (no custom re-split)
    for split in ["train", "validation", "test"]:
        encode_file(os.path.join(DATA_PATH, f"{split}.en"), os.path.join(DATA_PATH, f"{split}.spm.en"))
        encode_file(os.path.join(DATA_PATH, f"{split}.zh"), os.path.join(DATA_PATH, f"{split}.spm.zh"))
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  6️⃣ ======== Use official train/validation/test splits =========================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print(f"✅ Using official IWSLT17 train/validation/test splits.")

    print(f"✅ Encoded EN→ZH data saved under: {DATA_PATH}")
    print(f"✅ Next: run fairseq-preprocess to binarize (on *.spm.en/*.spm.zh).")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 3.2. PROCESSING DATASET 2 | XXX ----- FAIRSEQ PREPROCESSING & BINARIZATION |------####################
########################################################################################################################
########################################################################################################################
####-------| NOTE 3.2. PROCESSING DATASET 2 | XXX ------------| ENGLISH→CHINESE |--------------------####################
########################################################################################################################

"""
Stage 2:
Preprocess the tokenized English→Chinese dataset using fairseq-preprocess
to build vocabularies and generate Fairseq binary training files.
"""

# ─────────────────────────────────────────────────────────────────────────────────────────────────
print(
    f"🔎🔎 run_fairseq_preprocessing: {exp_args.run_fairseq_preprocessing} "
    f"{'(RUNNING)✔️✔️' if exp_args.run_fairseq_preprocessing else '(SKIPPED)❌❌'}"
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if exp_args.run_fairseq_preprocessing:
        
    import subprocess


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  1️⃣ ======== Check dataset files before preprocessing ==========================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    # ✅ Same folder you used above
    DATA_DIR = r"C:\Users\emeka\Research\ModelCUDA\Big_Data_Journal\Comparison\Code\Paper\github3\IWSLT\Transformer_Baselines_EN_ZH\Dataset"
    DESTDIR  = r"data-bin\iwslt17.en-zh"  # name it how you like
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    required_files = [
        os.path.join(DATA_DIR, "train.spm.en"),
        os.path.join(DATA_DIR, "train.spm.zh"),
        os.path.join(DATA_DIR, "validation.spm.en"),
        os.path.join(DATA_DIR, "validation.spm.zh"),
        os.path.join(DATA_DIR, "test.spm.en"),
        os.path.join(DATA_DIR, "test.spm.zh"),
    ]

    for f in required_files:
        assert os.path.exists(f), f"❌ Missing file: {f}"

    print("✅ All SentencePiece files found.")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    
    
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  2️⃣ ======== Configure preprocessing ============================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print("🚀 Running fairseq-preprocess for EN→ZH...")

    cmd = [
        "fairseq-preprocess",
        "--source-lang", "en",             # ✅Source = English
        "--target-lang", "zh",             # ✅Target = Chinese
        # use the SPM-encoded prefixes you just created
        "--trainpref", os.path.join(DATA_DIR, "train.spm"),
        "--validpref", os.path.join(DATA_DIR, "validation.spm"),
        "--testpref",  os.path.join(DATA_DIR, "test.spm"),
        "--destdir", DESTDIR,
        "--workers", "8"
    ]
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  3️⃣ ======== Run preprocessing ==================================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    subprocess.run(cmd, check=True)
    print(f"✅ Preprocessing complete! Binarized EN→ZH data saved in {DESTDIR}")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 4. FAIRSEQ SETUP, ACTIVATION PATCHING | XXX --------------------------------------####################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ========  4.1 Device Configuration =========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  4.2 Custom Activation Registration & Patching ====================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# Ensure activation folder is in sys.path
if ACTIVATION_PATH not in sys.path:
    sys.path.append(ACTIVATION_PATH)

# Backup original get_activation_fn
orig_get_activation_fn = utils.get_activation_fn
# ────────────────────────────────────────────────────────────────────────────────────────────────
def custom_get_activation_fn(activation: str):
    activation = activation.lower()
    if activation == "tanhexp":
        from activation.TanhExp import TanhExp
        print("✅ Registered custom activation: TanhExp")
        return TanhExp()
    
    elif activation == "fftgate":
        from activation.FFTGate import FFTGate
        print("✅ Registered custom activation: FFTGate")
        return FFTGate()
    
    elif activation == "geglu":        
        from activation.GEGLU import GEGLU
        print("✅ Registered custom activation: GEGLU")
        return GEGLU()

    else:
        # ✅ fallback to original fairseq activations
        return orig_get_activation_fn(activation)
# ────────────────────────────────────────────────────────────────────────────────────────────────
# Patch fairseq utils
utils.get_activation_fn = custom_get_activation_fn

# ────────────────────────────────────────────────────────────────────────────────────────────────
print(f"⚡ Using activation function: {exp_args.act_name.lower()}")

# Test-call the patched activation to confirm it works
try:
    act_fn = utils.get_activation_fn(exp_args.act_name.lower())
    print(f"🔍 Activation function resolved: {act_fn}")
except Exception as e:
    print(f"❌ Failed to resolve activation '{exp_args.act_name.lower()}': {e}")
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Manually inject custom activation into config
cfg.model.activation_fn = exp_args.act_name.lower()
print(f"⚡ Overwrote cfg.model.activation_fn = {cfg.model.activation_fn}")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ========  4.3 Fairseq Task & Dataset Initialization ========================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Load dictionaries
src_dict = Dictionary.load(os.path.join(fairseq_args.data, f"dict.{fairseq_args.source_lang}.txt"))
tgt_dict = Dictionary.load(os.path.join(fairseq_args.data, f"dict.{fairseq_args.target_lang}.txt"))

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Setup task
task = TranslationTask.setup_task(cfg.task)
task.load_dataset("train")
task.load_dataset("test")
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ✅✅ Print translation direction (EN→ZH confirmation)
print(f"🌍 Translation direction: {fairseq_args.source_lang} → {fairseq_args.target_lang}")
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ✅✅ Inspect dataset lengths after loading
train_dataset = task.dataset("train")
test_dataset  = task.dataset("test")
# ────────────────────────────────────────────────────────────────────────────────────────────────

print(f"📏 Train set - Max source length: {train_dataset.src_sizes.max()}, Max target length: {train_dataset.tgt_sizes.max()}")
print(f"📏 Test set  - Max source length: {test_dataset.src_sizes.max()}, Max target length: {test_dataset.tgt_sizes.max()}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Inspect data you’d lose if you lower max_positions
def check_percentage(dataset, N):
    src_long = (dataset.src_sizes > N).sum()
    tgt_long = (dataset.tgt_sizes > N).sum()
    total = len(dataset)

    print(f"Dataset size: {total}")
    print(f"  > {N} tokens (source): {src_long} ({100*src_long/total:.2f}%)")
    print(f"  > {N} tokens (target): {tgt_long} ({100*tgt_long/total:.2f}%)")
    print()

# ✅✅ Check on train + test
check_percentage(train_dataset, exp_args.max_source_positions)
check_percentage(test_dataset, exp_args.max_source_positions)
# ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 5. MODEL INITIALIZATION | XXX ----------------------------------------------------####################
########################################################################################################################

# ================================================================================================
# 🏷️========================= MODEL INITIALIZATION===============================================
# ================================================================================================
# ================================================================================================
####------------------ 0️⃣ 1️⃣ 2️⃣ 3️⃣ 4️⃣ 5️⃣ 6️⃣ 7️⃣ 8️⃣ 9️⃣ ------------------------------------####



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  ======== 🔖🔖  5.1 Model Construction 🔖🔖====================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 


# ================================================================================================
# 1️⃣ 🧠 Build Fairseq Baseline Transformer
# ================================================================================================
# ================================================================================================
# 1️⃣ 🧠 Build Fairseq Baseline Transformer
# ================================================================================================

if exp_args.net == "Transformer":

    model = task.build_model(cfg.model)

    print(f"✅ Loaded Model: {model.__class__.__name__}")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ================================================================================================
# 2️⃣ 🧠 Build RTM_Former 📐🔑🏷️
# ================================================================================================
# ================================================================================================
# 2️⃣ 🧠 Build RTM_Former 📐🔑🏷️
# ================================================================================================
elif exp_args.net == "RTM_Former":


    rtm_layers = []

    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 1️⃣ 🧩🧠 === FFN Insertion === ♻️♻️ ========================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 1️⃣ 🧩🧠 === FFN Insertion === ♻️♻️ ========================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    if exp_args.rtm_insertion_type == "ffn":

        # ================================================================================================
        # 1️⃣.0️⃣ 🧠 Build RTM_Former  
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:ffn🧬🧬")
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:ffn🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print("\nMODEL INIT CHECK")
        print(model.decoder.layers[0].fc1.weight[0, :10])
        print()
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────────────────────────────────       
        # =============================================================================
        # 1️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder FFNs 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i >= len(model.decoder.layers) - exp_args.rtm_num_layers:

                original_activation_fn = layer.activation_fn

                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,
                    rtm_rotation=exp_args.rtm_rotation                 
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:

                    # 🔒 Configure RTM coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Activation Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ───────────────────────────────────────────────────────────────────────────────────────────────── 
                # ─────────────────────────────────────────────────────────────────────────────────────────────────           


                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                def wrapped_activation(
                    x,
                    act_fn=original_activation_fn,
                    rtm_module=rtm
                ):

                    # ==========================================================
                    # 1️⃣.2️⃣.1️⃣ Standard FFN activation
                    # ==========================================================
                    x = act_fn(x)

                    # ==========================================================
                    # 1️⃣.2️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 1️⃣.2️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm_module.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0 + exp_args.rtm_transition_amplification * transition_scale
                    # )

                    # x = x * (
                    #     1.0 + rtm_module.amplification * transition_scale
                    # )

                    global RTM_SCALE

                    x_base = x

                    x_mod = x * (
                        1.0 + rtm_module.amplification * transition_scale
                    )

                    x_mod = rtm_module(x_mod)
                    # ==========================================================
                    # 1️⃣.2️⃣.6️⃣ RTM refinement
                    # ==========================================================
                    # x = rtm_module(x)
                    x = x_base + RTM_SCALE * (x_mod - x_base)

                    return x
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               

                # =============================================================================
                # 📌📌 Replace decoder activation function
                # =============================================================================                 
                layer.activation_fn = wrapped_activation
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  


                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 FFN Insertion | RTM inserted into decoder layer {i}")                
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────




    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣ 🧩🧠 === Cross-Attention Insertion ♻️♻️ ================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣ 🧩🧠 === Cross-Attention Insertion ♻️♻️ ================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────

    elif exp_args.rtm_insertion_type == "cross_attn":
    
        # ================================================================================================
        # 2️⃣.0️⃣ 🧠 Build RTM_Former 🅰️🔼 
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:cross_attn🧬🧬")
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:cross_attn🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder Cross-Attention 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i >= len(model.decoder.layers) - exp_args.rtm_num_layers:

                # =============================================================================
                # 2️⃣.2️⃣.1️⃣📌📌 RTM module
                # =============================================================================
                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,
                    rtm_rotation=exp_args.rtm_rotation                    
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:

                    # 🔒 Configure RTM coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Attention Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────


                # =============================================================================
                # 2️⃣.2️⃣.2️⃣📌📌 Store original encoder-attention forward
                # =============================================================================
                original_encoder_attn_forward = layer.encoder_attn.forward

                # =============================================================================
                # 2️⃣.2️⃣.3️⃣📌📌 Wrapped Cross-Attention forward
                # =============================================================================
                def wrapped_encoder_attn_forward(
                    query,
                    key,
                    value,
                    **kwargs
                ):

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.1️⃣ Original Cross-Attention
                    # ==========================================================
                    x, attn = original_encoder_attn_forward(
                        query=query,
                        key=key,
                        value=value,
                        **kwargs
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0
                    #     +
                    #     exp_args.rtm_transition_amplification
                    #     *
                    #     transition_scale
                    # )

                    x = x * (
                        1.0
                        +
                        rtm.amplification
                        *
                        transition_scale
                    )

                    return x, attn
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────


                # =============================================================================
                # 📌📌 Replace encoder-attention forward
                # =============================================================================
                layer.encoder_attn.forward = wrapped_encoder_attn_forward
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────

                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 Cross-Attention Insertion | RTM inserted into decoder layer {i}") 

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────



    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 3️⃣ 🧩🧠 === FFN Depth-Weighted Insertion === ♻️♻️ =========================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 3️⃣ 🧩🧠 === FFN Depth-Weighted Insertion === ♻️♻️ =========================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    elif exp_args.rtm_insertion_type == "ffn_depth_weighted":

        # ================================================================================================
        # 1️⃣.0️⃣ 🧠 Build RTM_Former  
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: Depth-Weighted RTM_Former for insertion_typ:ffn_depth_weighted🧬🧬")
        print(f"♻️♻️ Loaded Model: Depth-Weighted RTM_Former for insertion_typ:ffn_depth_weighted🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # 🔒 IMPORTANT: reset RNG after importing RTM_Former, before the real training model is built (For Repeatable Initialization)
        set_seed_torch(seed1)
        set_seed_main(seed2)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print("\nMODEL INIT CHECK")
        print(model.decoder.layers[0].fc1.weight[0, :10])
        print()
        # ==========================================================
        # 🚦🔍 Select layers
        # ========================================================== 
        selected_layers = list(
            range(
                len(model.decoder.layers) - exp_args.rtm_num_layers,
                len(model.decoder.layers)
            )
        )
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────────────────────────────────       
        # =============================================================================
        # 1️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder FFNs 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i in selected_layers:

                original_activation_fn = layer.activation_fn

                layer_rank = selected_layers.index(i)


                # =====================================================
                # ⏭️⏭️ Forward direction Layer scale ⏭️⏭️
                # =====================================================               
                if exp_args.rtm_layer_scale_direction == "forward":
                    layer_scale = exp_args.rtm_layer_scale_min + (
                        1.0 - exp_args.rtm_layer_scale_min
                    ) * (
                        layer_rank / max(1, exp_args.rtm_num_layers - 1)
                    )
                # =====================================================
                # ⏪⏪ Reverse direction Layer scale ⏪⏪
                # =====================================================  
                elif exp_args.rtm_layer_scale_direction == "reverse":
                    layer_scale = 1.0 - (
                        1.0 - exp_args.rtm_layer_scale_min
                    ) * (
                        layer_rank / max(1, exp_args.rtm_num_layers - 1)
                    )
                # =====================================================
                # ⏪🔒 Reverse direction + Fixed Final Layer Scale ⏪🔒
                # =====================================================
                elif exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                    if layer_rank == exp_args.rtm_num_layers - 1:
                        layer_scale = exp_args.rtm_final_layer_scale
                    else:
                        layer_scale = 1.0 - (
                            1.0 - exp_args.rtm_layer_scale_min
                        ) * (
                            layer_rank / max(1, exp_args.rtm_num_layers - 2)
                        ) 
                # ────────────────────────────────────────────────────────────────────────────────────────
                else:
                    raise ValueError(f"Unknown RTM layer scale direction: {exp_args.rtm_layer_scale_direction}")
                # ────────────────────────────────────────────────────────────────────────────────────────

                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,  
                    gate_type=exp_args.rtm_gate_type,
                    rtm_rotation=exp_args.rtm_rotation                                     
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility + Parameter Tracking)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:
                    
                    # 🔒 Configure RTM learnable coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Activation Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ─────────────────────────────────────────────────────────────────────────────────────────────────                
                # ─────────────────────────────────────────────────────────────────────────────────────────────────

                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                def wrapped_activation(
                    x,
                    act_fn=original_activation_fn,
                    rtm_module=rtm,
                    layer_scale=layer_scale
                ):

                    # ==========================================================
                    # 1️⃣.2️⃣.1️⃣ Standard FFN activation
                    # ==========================================================
                    x = act_fn(x)

                    # ==========================================================
                    # 1️⃣.2️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 1️⃣.2️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm_module.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0 + exp_args.rtm_transition_amplification * transition_scale
                    # )

                    # x = x * (
                    #     1.0 + rtm_module.amplification * transition_scale
                    # )

                    global RTM_SCALE

                    x_base = x

                    x_mod = x * (
                        1.0 + rtm_module.amplification * transition_scale
                    )

                    x_mod = rtm_module(x_mod)
                    # ==========================================================
                    # 1️⃣.2️⃣.6️⃣ RTM refinement
                    # ==========================================================

                    # ----------------------------------------------------------
                    # 🏔️ Depth-Aware RTM Scaling (Layer-Adaptive Modulation)
                    # ---------------------------------------------------------- 
                    if exp_args.rtm_depth_weight:

                        x = x_base + RTM_SCALE * layer_scale * (x_mod - x_base)

                    # ----------------------------------------------------------
                    # ⚖️ Uniform RTM Scaling (Same Modulation Across Layers)
                    # ----------------------------------------------------------
                    else:

                        x = x_base + RTM_SCALE * (x_mod - x_base)
                    # ─────────────────────────────────────────────────────────────────────────────────────────────────

                    return x
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               

                # =============================================================================
                # 📌📌 Replace decoder activation function
                # =============================================================================                 
                layer.activation_fn = wrapped_activation
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  


                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 FFN Insertion | RTM inserted into decoder layer {i} | layer_scale={layer_scale:.3f}")              
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: Depth-Weighted RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    else:
        raise ValueError(f"Unknown RTM insertion type: {exp_args.rtm_insertion_type}")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print(f"📣📣 RTM_Former for insertion_typ={exp_args.rtm_insertion_type} | Loaded Model={exp_args.net }♻️♻️")
    print(f"📣📣 RTM_Former for insertion_typ={exp_args.rtm_insertion_type} | Loaded Model={exp_args.net }♻️♻️")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────



    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  ======== 🔖  5.1.1 Model Configuration Hyperparameter Summary ==================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 
    if exp_args.net == "RTM_Former":

        rtm_log = (
            f"📣 RTM Configuration             : "
            f"alpha={exp_args.rtm_alpha} | "
            f"amp={exp_args.rtm_transition_amplification} | "
            f"layers={exp_args.rtm_num_layers} | "
            f"insertion={exp_args.rtm_insertion_type} | "
            f"scale_init={exp_args.rtm_scale} | "
            f"start_epoch={exp_args.rtm_start_epoch} | "
            f"full_epoch={exp_args.rtm_full_epoch}"
        )

        # Depth-weighted RTM only
        if exp_args.rtm_insertion_type == "ffn_depth_weighted":

            rtm_log += (
                f" | layer_scale_min={exp_args.rtm_layer_scale_min}"
                f" | layer_direction={exp_args.rtm_layer_scale_direction}"
                f" | gate={exp_args.rtm_gate_type}"
            )

            # Only exists for reverse_final_fixed
            if exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                rtm_log += (
                    f" | final_layer_scale={exp_args.rtm_final_layer_scale}"
                )

        print(rtm_log)


    else:
        print(
            f"📣 Baseline Configuration        : "
            f"model={exp_args.net}"
        )
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ======== 🔖🔖  5.2  Insert model to device 🔖🔖================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

model = model.to(DEVICE)
 # ─────────────────────────────────────────────────────────────────────────────────────────────────
print(f"🚀🚀 Model moved to device: {DEVICE} ♻️♻️")
print(f"🚀🚀 Model moved to device: {DEVICE} ♻️♻️")
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ======== 🔖🔖 5.3  Model Validation & Sanity Checks 🔖🔖=======================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 🔍 Print custom activations to confirm they are used
for name, module in model.named_modules():
    if isinstance(module, nn.Module) and module.__class__.__name__ in ["TanhExp", "FFTGate"]:
        print(f"🔎 Found custom activation in {name}: {module}")


# ✅ Sanity check Device
if DEVICE.type == "cuda":
    print(f"🚀 Model moved to CUDA: {torch.cuda.get_device_name(DEVICE)}")
else:
    print("⚠️ Using CPU (CUDA not available)")
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 6. PATH DEFINATION | XXX --------------------------------------------------------#####################
########################################################################################################################


# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 1️⃣ 📌📌 ======== ENSURE DIRECTORY EXIST ========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# # ─────────────────────────────────────────────────────────────────────────────────────────────────
# # 🟡 === Checkpoint directories ===
# if not os.path.exists('checkpoint'):
#     os.makedirs('checkpoint')
# # ─────────────────────────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Results directories ===
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}')

if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === LR Log directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Training Log directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Model Summary directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')
# ────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣ 📌📌 ======== DEFINE DIRECTORY ==============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.1️⃣🔖 ==== Checkpoint Directory Selection | Normal Run vs Crash Run | save every epoch ===== 
# ────────────────────────────────────────────────────────────────────────────────────────────────

# 🅰️💾 Normal run: save checkpoints in standard folder
if not exp_args.crash_run:
    checkpoint_dir = f'./checkpoints/{exp_args.dataset_name}/{exp_args.net}/'

    # 📁 Create checkpoint folder if it does not exist
    os.makedirs(checkpoint_dir, exist_ok=True)

    print(f"🅰️💾 Normal Run Enabled | Saving checkpoints to: {checkpoint_dir}")
#---------------------------------------------------------------------------------------------------
# 🅱️♻️ Crash run: save checkpoints in recovery folder
else:
    checkpoint_dir = f'./checkpoints_crash_run/{exp_args.dataset_name}/{exp_args.net}/'

    # 📁 Create checkpoint folder if it does not exist    
    os.makedirs(checkpoint_dir, exist_ok=True)

    print(f"🅱️♻️ Crash Run Enabled | Saving recovery checkpoints to: {checkpoint_dir}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.2️⃣🔖 === Main Test & Train Results === 
train_results_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/Train_{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
test_results_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/Test_{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.3️⃣🔖  === LR, Training & Summary logs  === 
LR_save_paths = {
    "LR_history": f"./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_LR_log.txt"
}
save_paths = {
    "log_history": f"./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_training_logs.txt"
}
# ────────────────────────────────────────────────────────────────────────────────────────────────

# 2️⃣.4️⃣🔖 ===  Model summary path === 
model_summary_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_model_summary.txt'
model_summary_full_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_model_summary_full.txt'
# ────────────────────────────────────────────────────────────────────────────────────────────────

# # ✅🔖 === Path template for saving checkpoints at every epoch | Part2 (if scratch occured) ===
# checkpoint_dir = f'./checkpoints/{exp_args.dataset_name}/tanhexp_AfterCrash_41_49epochs/'
# os.makedirs(checkpoint_dir, exist_ok=True)
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 7. TRAINING INITIALIZATION & OPTIMIZATION SETUP | XXX ---------------------------#####################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ========  7.1 AMP GradScaler ===============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Initialize AMP GradScaler once globall
scaler = torch.cuda.amp.GradScaler()
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  Training Criterion ===============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

criterion = LabelSmoothingCrossEntropy(smoothing=exp_args.smoothing)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ========   Optimizer Configuration =========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ♻️ Adam optimizer (β1=0.9, β2=0.98, eps=1e-9) ♻️
# ─────────────────────────────────────────────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=exp_args.lr,
    betas=(0.9, 0.98),
    eps=exp_args.eps,
    weight_decay=exp_args.weight_decay,
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 4️⃣ ========  Learning Rate Scheduler ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ✅  Transformer LR schedule (warmup + inverse sqrt decay)
# --------------------------

warmup_updates = exp_args.warmup_updates   # configurable warmup steps
peak_lr = exp_args.lr

# ────────────────────────────────────────────────────────────────────────────────────────────────
def lr_lambda(step):
    step = max(1, step)
    if step == 1:
        print(f"🚀 Warmup started at step {step} (target peak LR = {peak_lr}).")
    if step == warmup_updates:
        actual_lr = optimizer.param_groups[0]["lr"]
        print(f"✅ Warmup finished at step {step}. Peak LR reached: {actual_lr:.8f}")
    if step == warmup_updates + 1:
        print(f"📉 Inverse square root decay started at step {step}.")
    
    if step <= warmup_updates:
        # Linear warmup
        scale = step / warmup_updates
    else:
        # Inverse square root decay
        scale = (warmup_updates ** 0.5) / (step ** 0.5)
    return scale

# ────────────────────────────────────────────────────────────────────────────────────────────────

lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 5️⃣ ========  Global Tracking Variables ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ global initialization
best_bleu = 0.0
total_test_duration = 0.0
RTM_SCALE_LOG = ""
# ─────────────────────────────────────────────────────────────────────────────────────────────────

Python: 3.10.16 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:19:12) [MSC v.1929 64 bit (AMD64)]
Torch: 2.2.0+cu121 | CUDA available: True
Fairseq: 0.12.2
OmegaConf: 2.0.6
Hydra-Core: 1.0.7
✅ Current working directory: C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH
✅ sys.path updated:
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\python310.zip
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\DLLs
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env
   📂 
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\win32
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\win32\lib
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\Pythonwin
   📂 C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH
   📂 C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH\models
   📂 C:\

c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\modules\module.py:1150: UserWarning: expandable_segments not supported on this platform (Triggered internally at ..\c10/cuda/CUDAAllocatorConfig.h:30.)
  return t.to(device, dtype if t.is_floating_point() or t.is_complex() else None, non_blocking)


In [ ]:
########################################################################################################################
####-------| NOTE 11. TEST ALL CHECKPOINTS (BLEU per epoch)  | XXX -----------------------------########################
########################################################################################################################
########################################################################################################################
####-------| NOTE 11. TEST ALL CHECKPOINTS (BLEU per epoch)  | XXX -----------------------------########################
########################################################################################################################


# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ======== Paths for checkpoint BLEU sweeps ===================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Paths for checkpoint BLEU sweeps
test_checkpoint_results_path = (
    f'./Results/{exp_args.dataset_name}/{exp_args.net}/'
    f'TestCheckpoints_{exp_args.net}_'
    f'{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
)
# ────────────────────────────────────────────────────────────────────────────────────────────────


# # ✅ Paths for checkpoint BLEU sweeps | Part2 (if Cratch occured)
# test_checkpoint_results_path = (
#     f'./Results/{exp_args.dataset_name}/{exp_args.act_name}/'
#     f'TestCheckpoints_{exp_args.act_name}_{exp_args.net}_'
#     f'{exp_args.dataset_name}_{exp_args.optimizer1}_tanhexp_AfterCrash_41_49epochs.txt'
# )
# ────────────────────────────────────────────────────────────────────────────────────────────────


# # ─────────────────────────────────────────────────────────────────────────────────────────────────
# #  2️⃣ ========  BLEU Detokenization (🔖 EN/FR/DE) ================================================
# # ───────────────────────────────────────────────────────────────────────────────────────────────── 
# # ────────────────────────────────────────────────────────────────────────────────────────────────
# def detokenize_for_bleu(s: str) -> str:
#     s = s.replace("@@ ", "").replace("@@", "")  # BPE
#     s = s.replace("▁", " ")                     # SentencePiece
#     s = " ".join(s.split())                     # normalize spaces
#     s = re.sub(r"\s+([?.!,])", r"\1", s)        # punctuation
#     return s.strip()
# # ────────────────────────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  BLEU Detokenization (🔖 Chinese) =================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
def detokenize_for_bleu(s: str) -> str:
    # For Chinese, don't remove SentencePiece boundaries or spaces | Only remove BPE continuation markers
    # sacreBLEU's "zh" tokenizer expects raw characters.
    return s.replace("@@ ", "").replace("@@", "").strip()
# ────────────────────────────────────────────────────────────────────────────────────────────────




# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ======== Test_Checkpoint Function ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ test_checkpoint function
def test_checkpoint_epochs(task, model, criterion, checkpoint_dir, test_checkpoint_results_path):
    """
    Evaluate BLEU (beam=5) for all checkpoints in checkpoint_dir.
    Save results to test_checkpoint_results_path in the same format as normal test logs.
    """
# ────────────────────────────────────────────────────────────────────────────────────────────────
    global RTM_SCALE, RTM_SCALE_LOG
    # ────────────────────────────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Collect checkpoints sorted by epoch number in filename
    def extract_epoch_num(filename):
        match = re.search(r"epoch_(\d+)", filename)
        return int(match.group(1)) if match else float("inf")  # inf if no epoch number

    ckpts = sorted(
        glob.glob(os.path.join(checkpoint_dir, "*.pt")),
        key=lambda x: extract_epoch_num(os.path.basename(x))
    )

    if not ckpts:
        print(f"⚠️ No checkpoints found in {checkpoint_dir}")
        return

    # # ────────────────────────────────────────────────────────────────────────────────────────────────
    # # ✅ Reset output file
    # os.makedirs(os.path.dirname(test_checkpoint_results_path), exist_ok=True)
    # with open(test_checkpoint_results_path, "w", encoding="utf-8") as f:
    #     f.write("")
    # # ────────────────────────────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 📦 Reset output file + write model status
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    #1️⃣ Create output folder if it does not exist
    os.makedirs(os.path.dirname(test_checkpoint_results_path), exist_ok=True)

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Normal run: reset file
    # ✅ Crash run : append to existing file
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣♻️ Normal run: reset file
    if not exp_args.crash_run:

        with open(test_checkpoint_results_path, "w", encoding="utf-8") as f:
            f.write("")

    # 3️⃣⏭️ Crash run : append to existing file
    else:

        with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
            f.write("\n♻️ Resuming checkpoint testing after crash\n\n")
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 4️⃣✍️  Write test result header (model information)
    with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
        f.write("")

        # ---------------------------------------------------------------------------
        # 4️⃣.1️⃣🏷️ Testing Model Status
        # ---------------------------------------------------------------------------
        if exp_args.net == "RTM_Former":

            f.write(
                f"Model: RTM_Former\n\n"
            )

        else:

            f.write(
                f"Model: Baseline Fairseq Transformer\n\n"
            )        
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ────────────────────────────────────────────────────────────────────────────────────────────────



    # ────────────────────────────────────────────────────────────────────────────────────────────────
    best_bleu = 0.0
    total_test_duration = 0.0
    # ────────────────────────────────────────────────────────────────────────────────────────────────


    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    for ckpt in ckpts:

        # # ────────────────────────────────────────────────────────────────────────────────
        # # ✅ 📌📌 Load checkpoint (OLD) 🅰️🔼 
        # # ────────────────────────────────────────────────────────────────────────────────        
        # print(f"🔄 Loading checkpoint: {ckpt}")
        # state = torch.load(ckpt, map_location=DEVICE)
        # model.load_state_dict(state["model_state"], strict=False)
        # model.to(DEVICE)
        # model.eval()
        # --------------------------------------------------------------------------------------------------
        # ────────────────────────────────────────────────────────────────────────────────
        # ✅📌📌 Load checkpoint (NEW) 🅱️🔼 Restore RTM schedule for checkpoint testing using same strategy as main loop
        # ────────────────────────────────────────────────────────────────────────────────
        if exp_args.net == "Transformer":
            print(f"🔄 Loading checkpoint Baseline Transformer : {ckpt}")
            state = torch.load(ckpt, map_location=DEVICE)
            model.load_state_dict(state["model_state"], strict=False)
            model.to(DEVICE)
            model.eval()
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        elif exp_args.net == "RTM_Former":
            print(f"🔄 Loading checkpoint RTM_Former : {ckpt}")
            state = torch.load(ckpt, map_location=DEVICE)
            model.load_state_dict(state["model_state"], strict=False)            
            

            RTM_START_EPOCH = exp_args.rtm_start_epoch
            RTM_FULL_EPOCH = exp_args.rtm_full_epoch
            ckpt_epoch = state["epoch"]

            if ckpt_epoch < RTM_START_EPOCH:
                RTM_SCALE = exp_args.rtm_scale
            else:
                progress = min(
                    1.0,
                    (ckpt_epoch - RTM_START_EPOCH) / (RTM_FULL_EPOCH - RTM_START_EPOCH)
                )

                RTM_SCALE = 0.5 * (1.0 - math.cos(math.pi * progress))

            print(f"📐 Checkpoint RTM_SCALE for epoch {ckpt_epoch}: {RTM_SCALE:.3f}")

            if exp_args.rtm_insertion_type == "ffn_depth_weighted":

                if exp_args.rtm_layer_scale_direction == "forward":
                    depth_scales = [
                        exp_args.rtm_layer_scale_min
                        + (1.0 - exp_args.rtm_layer_scale_min)
                        * (j / max(1, exp_args.rtm_num_layers - 1))
                        for j in range(exp_args.rtm_num_layers)
                    ]

                elif exp_args.rtm_layer_scale_direction == "reverse":
                    depth_scales = [
                        1.0
                        - (1.0 - exp_args.rtm_layer_scale_min)
                        * (j / max(1, exp_args.rtm_num_layers - 1))
                        for j in range(exp_args.rtm_num_layers)
                    ]

                elif exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                    depth_scales = []

                    for j in range(exp_args.rtm_num_layers):
                        if j == exp_args.rtm_num_layers - 1:
                            scale_j = exp_args.rtm_final_layer_scale
                        else:
                            scale_j = (
                                1.0
                                - (1.0 - exp_args.rtm_layer_scale_min)
                                * (j / max(1, exp_args.rtm_num_layers - 2))
                            )

                        depth_scales.append(scale_j)

                else:
                    raise ValueError(
                        f"Unknown RTM layer scale direction: {exp_args.rtm_layer_scale_direction}"
                    )

                final_scales = [RTM_SCALE * s for s in depth_scales]

                RTM_SCALE_LOG = f" | RTM_SCALE={RTM_SCALE:.3f}"
                RTM_SCALE_LOG += (
                    f" | RTM_layer_scales({exp_args.rtm_layer_scale_direction})="
                    f"{[round(s, 3) for s in final_scales]}"
                )

                print(
                    f"📐 Checkpoint final RTM layer scales "
                    f"({exp_args.rtm_layer_scale_direction}): "
                    f"{[round(s, 3) for s in final_scales]}"
                )
            model.to(DEVICE)
            model.eval()               
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        else:
            raise ValueError(f"Unknown model type: {exp_args.net}")
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ────────────────────────────────────────────────────────────────────────────────────────────────


        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Build test iterator
        test_iter = task.get_batch_iterator(
            dataset=task.dataset("test"),
            max_tokens=fairseq_args.max_tokens,   # cut in half option: fairseq_args.max_tokens // 2
            max_positions=(exp_args.max_source_positions, exp_args.max_target_positions),
            seed=seed1,
            num_shards=exp_args.num_shards,
            shard_id=exp_args.shard_id,
            num_workers=exp_args.num_workers,
            ignore_invalid_inputs=exp_args.ignore_invalid_inputs,
        ).next_epoch_itr(shuffle=False)
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ References
        true_references = []
        for sample in test_iter:
            targets = sample["target"]
            for t in targets:
                t_no_pad = t[t != tgt_dict.pad()]
                ref_str = tgt_dict.string(t_no_pad, bpe_symbol="@@")
                true_references.append(detokenize_for_bleu(ref_str))
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Rebuild iterator for decoding
        test_iter = task.get_batch_iterator(
            dataset=task.dataset("test"),
            max_tokens=fairseq_args.max_tokens,    # cut in half option: fairseq_args.max_tokens // 2
            max_positions=(exp_args.max_source_positions, exp_args.max_target_positions),
            seed=seed1,
            num_shards=exp_args.num_shards,
            shard_id=exp_args.shard_id,
            num_workers=exp_args.num_workers,
            ignore_invalid_inputs=exp_args.ignore_invalid_inputs,
        ).next_epoch_itr(shuffle=False)
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Build generator with beam=5
        gen_cfg = deepcopy(cfg.generation)
        gen_cfg.beam = exp_args.beam         # 🎀🔖 5
        gen_cfg.lenpen = exp_args.lenpen     # 🎀🔖 1.0 
        generator = task.build_generator([model], gen_cfg)
        # ────────────────────────────────────────────────────────────────────────────────────────────────


 

        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Run evaluation
        sys_outputs, test_loss = [], 0.0
        start_time = time.time()

        with torch.no_grad():
            with tqdm(enumerate(test_iter), total=len(test_iter), desc=f"Testing {os.path.basename(ckpt)}") as progress:
                for i, sample in progress:   # 👈 keep i!
                    sample = utils.move_to_cuda(sample) if torch.cuda.is_available() else sample
                    src_tokens = sample["net_input"]["src_tokens"]
                    prev_output_tokens = sample["net_input"]["prev_output_tokens"]
                    target = sample["target"]
                    src_lengths = (src_tokens != src_dict.pad()).sum(dim=1)

                    with torch.cuda.amp.autocast():
                        logits, _ = model(src_tokens, src_lengths, prev_output_tokens)
                        pad = tgt_dict.pad()
                        logits2d = logits.view(-1, logits.size(-1))
                        target1d = target.view(-1)
                        mask = target1d != pad
                        loss = criterion(logits2d[mask], target1d[mask]) if mask.any() else logits2d.sum() * 0.0
                    test_loss += loss.item()

                    hypos = task.inference_step(generator, [model], sample)
                    for j, hypos_j in enumerate(hypos):
                        hypo_tokens = [tok for tok in hypos_j[0]["tokens"].tolist() if tok != tgt_dict.pad()]
                        sys_str = tgt_dict.string(hypo_tokens, bpe_symbol="@@")
                        sys_outputs.append(detokenize_for_bleu(sys_str))

                    # ✅ Update progress bar postfix with running average
                    progress.set_postfix(loss=round(test_loss / (i + 1), 3))
       # ────────────────────────────────────────────────────────────────────────────────────────────────




       # ────────────────────────────────────────────────────────────────────────────────────────────────
        avg_loss = test_loss / max(1, len(test_iter))
        # bleu = sacrebleu.corpus_bleu(sys_outputs, [true_references])                 #🔖 EN/FR/DE
        bleu = sacrebleu.corpus_bleu(sys_outputs, [true_references], tokenize="zh")    #🔖 Chinese

        duration = time.time() - start_time
        total_test_duration += duration
       # ────────────────────────────────────────────────────────────────────────────────────────────────


       # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Write results
        with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
            f.write(f"Checkpoint {os.path.basename(ckpt)} | Test Loss: {avg_loss:.3f} | sacreBLEU: {bleu.score:.2f} | Beam=5\n")

        print(f"✅ {os.path.basename(ckpt)} | Loss {avg_loss:.3f} | sacreBLEU {bleu.score:.2f}")


        # 🧹 add visual spacing between runs
        tqdm.write("")  
       # ────────────────────────────────────────────────────────────────────────────────────────────────

       # ────────────────────────────────────────────────────────────────────────────────────────────────
        if bleu.score > best_bleu:
            best_bleu = bleu.score

    with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
        f.write(f"\n🏆 Best BLEU Score: {best_bleu:.2f}\n")
        total_mins, total_secs = divmod(total_test_duration, 60)
        f.write(f"\n🕒 Total Test Time over all checkpoints: {int(total_mins)} min {total_secs:.2f} sec\n")

    print(f"🏆 Best BLEU over checkpoints: {best_bleu:.2f}")
    # ────────────────────────────────────────────────────────────────────────────────────────────────



# ────────────────────────────────────────────────────────────────────────────────────────────────
# # ✅ Free unused GPU memory BEFORE training starts
# torch.cuda.empty_cache()
# torch.cuda.reset_peak_memory_stats()

# 4️⃣ Call the function
test_checkpoint_epochs(task, model, criterion, checkpoint_dir, test_checkpoint_results_path)
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────────────────────────

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_9_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 9: 0.345


Testing epoch_9_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt:   0%|          | 0/82 [00:00<?, ?it/s]c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)
c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\functional.py:5109: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
Testing epoch_9_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:57<00:00,  1.43it/s, loss=4.8] 


✅ epoch_9_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.800 | sacreBLEU 15.44

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_10_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 10: 0.500


Testing epoch_10_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [01:09<00:00,  1.18it/s, loss=4.74]


✅ epoch_10_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.740 | sacreBLEU 14.69

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_11_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 11: 0.655


...

Testing epoch_16_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:52<00:00,  1.58it/s, loss=4.56]


✅ epoch_16_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.560 | sacreBLEU 18.19

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_17_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 17: 1.000


Testing epoch_17_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:44<00:00,  1.84it/s, loss=4.53]


✅ epoch_17_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.528 | sacreBLEU 17.95

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_18_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 18: 1.000


: 

Restarted pytorch_env (Python 3.10.16)

In [ ]:
#####-------------------------------- NOTE MAIN IWSLT17 EN-ZH NOTE --------------------------------------------------#####
##########################################################################################################################
######################|--------------------------------------------------------------|####################################
###################################🔗 MAIN | TRAIN | TEST LOOP 🔗########################################################
######################|--------------------------------------------------------------|####################################
##########################################################################################################################
#####-------------------------------- NOTE MAIN IWSLT17 EN-ZH NOTE --------------------------------------------------#####



# 📄  main_IWSLT17En_Zh.py
########################################################################################################################
####-------| NOTE 1.1 IMPORTS LIBRARIES  | XXX -------------------------------------------------------##################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 📜 === Enable flexible CUDA memory allocation to reduce fragmentation ===
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ======================================================================================================
# 📜 === Core Libraries ===
# ======================================================================================================
import sys
import numpy as np
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import time
import random
import math
import re
import glob
from copy import deepcopy
import sacrebleu  # ✅ for BLEU evaluation
from timm.loss import LabelSmoothingCrossEntropy
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ======================================================================================================
# 📜 === Fairseq & Supporting Imports ===
# ======================================================================================================
import importlib.metadata as importlib_metadata
import fairseq  # type: ignore
from fairseq.data import Dictionary, data_utils, iterators          # type: ignore
from fairseq.tasks.translation import TranslationTask               # type: ignore
from fairseq import options                                         # type: ignore
from fairseq.dataclass.utils import convert_namespace_to_omegaconf  # type: ignore                                             
from fairseq import utils                                           # type: ignore 
import hydra                                                        # type: ignore  
# ────────────────────────────────────────────────────────────────────────────────────────────────
from ptflops import get_model_complexity_info
from calflops import calculate_flops
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ======================================================================================================
# ♻️ === Print environment summary for sanity check ===
# ======================================================================================================
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Fairseq:", fairseq.__version__)
print("OmegaConf:", importlib_metadata.version("omegaconf"))
print("Hydra-Core:", importlib_metadata.version("hydra-core"))
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 1.2. DEFINE PROJECT PATH | XXX ---------------------------------------------------####################
########################################################################################################################

# ✅ Define working directory
Project_PATH = r"C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH"
if os.getcwd() != Project_PATH:
    os.chdir(Project_PATH)
print(f"✅ Current working directory: {os.getcwd()}")


# ✅ Define absolute paths
PROJECT_PATH = Project_PATH
MODELS_PATH = os.path.join(Project_PATH, "models")
ACTIVATION_PATH = os.path.join(Project_PATH, "activation")


# ✅ Ensure necessary paths are in sys.path
for path in [PROJECT_PATH, MODELS_PATH, ACTIVATION_PATH]:
    if path not in sys.path:
        sys.path.append(path)

# ✅ Print updated sys.path for debugging
print("✅ sys.path updated:")
for path in sys.path:
    print("   📂", path)
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 1.3. OTHER IMPORTS | XXX ---------------------------------------------------------####################
########################################################################################################################

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 📜 ============  Imput parser safe for afno because of --fno-bias, etc  =======================
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 1️⃣✅ Import parser from parser_IWSLT17En_Zh.py
from parser_IWSLT17En_Zh import get_parser

# ✅ Create parser and parse arguments
parser = get_parser()
# args, unknown = parser.parse_known_args()

# ✅ IMPORTANT: Do NOT read Jupyter / VSCode kernel arguments
# This prevents the "--f" ambiguity issue
exp_args = parser.parse_args(args=[])

num_aug_splits = exp_args.aug_splits

print(f"✅ Parser imported successfully | num_aug_splits = {num_aug_splits}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣✅ Import Fairseq parser/config
from fairseq_config_IWSLT17En_Zh import get_fairseq_parser
# ✅ Build Fairseq configuration
fairseq_args, cfg = get_fairseq_parser(exp_args)


num_aug_splits = exp_args.aug_splits

print(f"✅ Parser imported successfully | num_aug_splits = {num_aug_splits}")

print(f"✅ Encoder Embed Dim: {fairseq_args.encoder_embed_dim}")
print(f"✅ Decoder Layers: {fairseq_args.decoder_layers}")
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 3️⃣✅ Import model summary utility
from models.model_summary.model_summary import save_model_summary
from models.model_summary.model_summary_full import save_model_summary_full

print("✅ model_summary.py loaded successfully")
print("✅ model_summary_full.py loaded successfully")
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 2. SEEDING FOR REPRODUCIBILITY | XXX ---------------------------------------------####################
########################################################################################################################

# ────────────────────────────────────────────────────────────────────────────────────────────────
def set_seed_torch(seed):
    torch.manual_seed(seed)
# ────────────────────────────────────────────────────────────────────────────────────────────────
def set_seed_main(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True      # ✅ Default: "False" for Faster, non-deterministic kernels | "True" to Ensure deterministic behavior for CuDNN (Slower)
    torch.backends.cudnn.benchmark = False         # ✅ Default: "True" for Autotune kernels for performance   | "False" Disable CuDNN's autotuning for reproducibility (Slower)

    torch.backends.cuda.matmul.allow_tf32 = False  # ✅ Disable TF32 (strict reproducibility)
    torch.backends.cudnn.allow_tf32 = False        # ✅ Disable TF32 (strict reproducibility)

    # torch.use_deterministic_algorithms(True, warn_only=True)

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ ============= Define Seed =============
seed1, seed2 = exp_args.seed1, exp_args.seed2
set_seed_torch(seed1)  
set_seed_main(seed2)  
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 3.1. PROCESSING DATASET 1 | XXX ---- TOKENIZATION & DATASET PREPARATION |---------####################
########################################################################################################################
########################################################################################################################
####-------| NOTE 3.1. PROCESSING DATASET 1 | XXX ------------| ENGLISH→CHINESE |-------------------####################
########################################################################################################################

"""
Stage 1:
Train SentencePiece tokenizer, encode the raw English→Chinese corpus,
and create aligned train/test text splits for neural machine translation.
"""

# ─────────────────────────────────────────────────────────────────────────────────────────────────
print(
    f"🔎🔎 run_dataset_tokenization: {exp_args.run_dataset_tokenization} "
    f"{'(RUNNING)✔️✔️' if exp_args.run_dataset_tokenization else '(SKIPPED)❌❌'}"
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if exp_args.run_dataset_tokenization:
    from datasets import load_dataset    
    import sentencepiece as spm

    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Path for EN→ZH dataset
    DATA_PATH = r"C:\Users\emeka\Research\ModelCUDA\Big_Data_Journal\Comparison\Code\Paper\github3\IWSLT\Transformer_Baselines_EN_ZH\Dataset"
    os.makedirs(DATA_PATH, exist_ok=True)
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  1️⃣ ======== Download and export official IWSLT17 dataset =======================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Download to your folder (as the Hugging Face cache)
    en_zh = load_dataset(
        "IWSLT/iwslt2017", "iwslt2017-en-zh",
        cache_dir=DATA_PATH,
        trust_remote_code=True
    )
    zh_en = load_dataset(
        "IWSLT/iwslt2017", "iwslt2017-zh-en",
        cache_dir=DATA_PATH,
        trust_remote_code=True
    )  # optional
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  2️⃣ ======== Dataset verification ===============================================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ sanity check (should show ~231k train)
    print(f"EN→ZH dataset: {en_zh}")
    print(f"ZH→EN dataset: {zh_en}")

    print("EN→ZH splits:", list(en_zh.keys()), {k: len(en_zh[k]) for k in en_zh.keys()})
    print("ZH→EN splits:", list(zh_en.keys()), {k: len(zh_en[k]) for k in zh_en.keys()})

    # optional assertions
    assert set(en_zh.keys()) == {"train", "validation", "test"}
    assert set(zh_en.keys()) == {"train", "validation", "test"}
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  3️⃣ ======== Export official train/validation/test splits ======================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Export official IWSLT17 splits to plain text (EN/ZH) under raw_zh_en_dataset_path
    def dump_split(ds_dict, split, src, tgt, out_dir):
        src_fp = os.path.join(out_dir, f"{split}.{src}")
        tgt_fp = os.path.join(out_dir, f"{split}.{tgt}")
        with open(src_fp, "w", encoding="utf-8") as fs, open(tgt_fp, "w", encoding="utf-8") as ft:
            for row in ds_dict[split]:
                tr = row["translation"]
                fs.write((tr[src] or "").strip() + "\n")
                ft.write((tr[tgt] or "").strip() + "\n")
        print(f"Wrote {split}: {src_fp} | {tgt_fp}")

    for split in ["train", "validation", "test"]:
        dump_split(en_zh, split, "en", "zh", DATA_PATH)
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  4️⃣ ========  Train SentencePiece model =========================================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────     

    print("🧠 Training SentencePiece model for EN→ZH...")

    # Use TRAIN split for SPM training
    SRC_EN = os.path.join(DATA_PATH, "train.en")
    TGT_ZH = os.path.join(DATA_PATH, "train.zh")

    spm_model_prefix = os.path.join(DATA_PATH, "spm_enzh")

    spm.SentencePieceTrainer.Train(
        input=f"{SRC_EN},{TGT_ZH}",
        model_prefix=spm_model_prefix,
        vocab_size=10000,
        character_coverage=0.9995,   # ✅ --character_coverage for rich charcater langauage: Japanese or Chinese = 0.9995 | other languages with small character set: 1.0
        model_type="bpe"
    )
    print(f"✅ SentencePiece model saved to: {spm_model_prefix}.model / .vocab")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  5️⃣ ========  Encode files ======================================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    print("⚙️ Encoding dataset...")

    # ✅ Load the trained SentencePiece model
    sp = spm.SentencePieceProcessor(model_file=f"{spm_model_prefix}.model")

    def encode_file(in_file, out_file):
        with open(in_file, encoding="utf-8") as fin, open(out_file, "w", encoding="utf-8") as fout:
            for line in fin:
                fout.write(" ".join(sp.encode(line.strip(), out_type=str)) + "\n")

    # ✅ Encode all official splits (no custom re-split)
    for split in ["train", "validation", "test"]:
        encode_file(os.path.join(DATA_PATH, f"{split}.en"), os.path.join(DATA_PATH, f"{split}.spm.en"))
        encode_file(os.path.join(DATA_PATH, f"{split}.zh"), os.path.join(DATA_PATH, f"{split}.spm.zh"))
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  6️⃣ ======== Use official train/validation/test splits =========================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print(f"✅ Using official IWSLT17 train/validation/test splits.")

    print(f"✅ Encoded EN→ZH data saved under: {DATA_PATH}")
    print(f"✅ Next: run fairseq-preprocess to binarize (on *.spm.en/*.spm.zh).")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 3.2. PROCESSING DATASET 2 | XXX ----- FAIRSEQ PREPROCESSING & BINARIZATION |------####################
########################################################################################################################
########################################################################################################################
####-------| NOTE 3.2. PROCESSING DATASET 2 | XXX ------------| ENGLISH→CHINESE |--------------------####################
########################################################################################################################

"""
Stage 2:
Preprocess the tokenized English→Chinese dataset using fairseq-preprocess
to build vocabularies and generate Fairseq binary training files.
"""

# ─────────────────────────────────────────────────────────────────────────────────────────────────
print(
    f"🔎🔎 run_fairseq_preprocessing: {exp_args.run_fairseq_preprocessing} "
    f"{'(RUNNING)✔️✔️' if exp_args.run_fairseq_preprocessing else '(SKIPPED)❌❌'}"
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if exp_args.run_fairseq_preprocessing:
        
    import subprocess


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  1️⃣ ======== Check dataset files before preprocessing ==========================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    # ✅ Same folder you used above
    DATA_DIR = r"C:\Users\emeka\Research\ModelCUDA\Big_Data_Journal\Comparison\Code\Paper\github3\IWSLT\Transformer_Baselines_EN_ZH\Dataset"
    DESTDIR  = r"data-bin\iwslt17.en-zh"  # name it how you like
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    required_files = [
        os.path.join(DATA_DIR, "train.spm.en"),
        os.path.join(DATA_DIR, "train.spm.zh"),
        os.path.join(DATA_DIR, "validation.spm.en"),
        os.path.join(DATA_DIR, "validation.spm.zh"),
        os.path.join(DATA_DIR, "test.spm.en"),
        os.path.join(DATA_DIR, "test.spm.zh"),
    ]

    for f in required_files:
        assert os.path.exists(f), f"❌ Missing file: {f}"

    print("✅ All SentencePiece files found.")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    
    
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  2️⃣ ======== Configure preprocessing ============================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print("🚀 Running fairseq-preprocess for EN→ZH...")

    cmd = [
        "fairseq-preprocess",
        "--source-lang", "en",             # ✅Source = English
        "--target-lang", "zh",             # ✅Target = Chinese
        # use the SPM-encoded prefixes you just created
        "--trainpref", os.path.join(DATA_DIR, "train.spm"),
        "--validpref", os.path.join(DATA_DIR, "validation.spm"),
        "--testpref",  os.path.join(DATA_DIR, "test.spm"),
        "--destdir", DESTDIR,
        "--workers", "8"
    ]
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  3️⃣ ======== Run preprocessing ==================================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    subprocess.run(cmd, check=True)
    print(f"✅ Preprocessing complete! Binarized EN→ZH data saved in {DESTDIR}")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 4. FAIRSEQ SETUP, ACTIVATION PATCHING | XXX --------------------------------------####################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ========  4.1 Device Configuration =========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  4.2 Custom Activation Registration & Patching ====================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# Ensure activation folder is in sys.path
if ACTIVATION_PATH not in sys.path:
    sys.path.append(ACTIVATION_PATH)

# Backup original get_activation_fn
orig_get_activation_fn = utils.get_activation_fn
# ────────────────────────────────────────────────────────────────────────────────────────────────
def custom_get_activation_fn(activation: str):
    activation = activation.lower()
    if activation == "tanhexp":
        from activation.TanhExp import TanhExp
        print("✅ Registered custom activation: TanhExp")
        return TanhExp()
    
    elif activation == "fftgate":
        from activation.FFTGate import FFTGate
        print("✅ Registered custom activation: FFTGate")
        return FFTGate()
    
    elif activation == "geglu":        
        from activation.GEGLU import GEGLU
        print("✅ Registered custom activation: GEGLU")
        return GEGLU()

    else:
        # ✅ fallback to original fairseq activations
        return orig_get_activation_fn(activation)
# ────────────────────────────────────────────────────────────────────────────────────────────────
# Patch fairseq utils
utils.get_activation_fn = custom_get_activation_fn

# ────────────────────────────────────────────────────────────────────────────────────────────────
print(f"⚡ Using activation function: {exp_args.act_name.lower()}")

# Test-call the patched activation to confirm it works
try:
    act_fn = utils.get_activation_fn(exp_args.act_name.lower())
    print(f"🔍 Activation function resolved: {act_fn}")
except Exception as e:
    print(f"❌ Failed to resolve activation '{exp_args.act_name.lower()}': {e}")
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Manually inject custom activation into config
cfg.model.activation_fn = exp_args.act_name.lower()
print(f"⚡ Overwrote cfg.model.activation_fn = {cfg.model.activation_fn}")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ========  4.3 Fairseq Task & Dataset Initialization ========================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Load dictionaries
src_dict = Dictionary.load(os.path.join(fairseq_args.data, f"dict.{fairseq_args.source_lang}.txt"))
tgt_dict = Dictionary.load(os.path.join(fairseq_args.data, f"dict.{fairseq_args.target_lang}.txt"))

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Setup task
task = TranslationTask.setup_task(cfg.task)
task.load_dataset("train")
task.load_dataset("test")
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ✅✅ Print translation direction (EN→ZH confirmation)
print(f"🌍 Translation direction: {fairseq_args.source_lang} → {fairseq_args.target_lang}")
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ✅✅ Inspect dataset lengths after loading
train_dataset = task.dataset("train")
test_dataset  = task.dataset("test")
# ────────────────────────────────────────────────────────────────────────────────────────────────

print(f"📏 Train set - Max source length: {train_dataset.src_sizes.max()}, Max target length: {train_dataset.tgt_sizes.max()}")
print(f"📏 Test set  - Max source length: {test_dataset.src_sizes.max()}, Max target length: {test_dataset.tgt_sizes.max()}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Inspect data you’d lose if you lower max_positions
def check_percentage(dataset, N):
    src_long = (dataset.src_sizes > N).sum()
    tgt_long = (dataset.tgt_sizes > N).sum()
    total = len(dataset)

    print(f"Dataset size: {total}")
    print(f"  > {N} tokens (source): {src_long} ({100*src_long/total:.2f}%)")
    print(f"  > {N} tokens (target): {tgt_long} ({100*tgt_long/total:.2f}%)")
    print()

# ✅✅ Check on train + test
check_percentage(train_dataset, exp_args.max_source_positions)
check_percentage(test_dataset, exp_args.max_source_positions)
# ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 5. MODEL INITIALIZATION | XXX ----------------------------------------------------####################
########################################################################################################################

# ================================================================================================
# 🏷️========================= MODEL INITIALIZATION===============================================
# ================================================================================================
# ================================================================================================
####------------------ 0️⃣ 1️⃣ 2️⃣ 3️⃣ 4️⃣ 5️⃣ 6️⃣ 7️⃣ 8️⃣ 9️⃣ ------------------------------------####



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  ======== 🔖🔖  5.1 Model Construction 🔖🔖====================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 


# ================================================================================================
# 1️⃣ 🧠 Build Fairseq Baseline Transformer
# ================================================================================================
# ================================================================================================
# 1️⃣ 🧠 Build Fairseq Baseline Transformer
# ================================================================================================

if exp_args.net == "Transformer":

    model = task.build_model(cfg.model)

    print(f"✅ Loaded Model: {model.__class__.__name__}")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ================================================================================================
# 2️⃣ 🧠 Build RTM_Former 📐🔑🏷️
# ================================================================================================
# ================================================================================================
# 2️⃣ 🧠 Build RTM_Former 📐🔑🏷️
# ================================================================================================
elif exp_args.net == "RTM_Former":


    rtm_layers = []

    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 1️⃣ 🧩🧠 === FFN Insertion === ♻️♻️ ========================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 1️⃣ 🧩🧠 === FFN Insertion === ♻️♻️ ========================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    if exp_args.rtm_insertion_type == "ffn":

        # ================================================================================================
        # 1️⃣.0️⃣ 🧠 Build RTM_Former  
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:ffn🧬🧬")
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:ffn🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print("\nMODEL INIT CHECK")
        print(model.decoder.layers[0].fc1.weight[0, :10])
        print()
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────────────────────────────────       
        # =============================================================================
        # 1️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder FFNs 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i >= len(model.decoder.layers) - exp_args.rtm_num_layers:

                original_activation_fn = layer.activation_fn

                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,
                    rtm_rotation=exp_args.rtm_rotation                 
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:

                    # 🔒 Configure RTM coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Activation Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ───────────────────────────────────────────────────────────────────────────────────────────────── 
                # ─────────────────────────────────────────────────────────────────────────────────────────────────           


                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                def wrapped_activation(
                    x,
                    act_fn=original_activation_fn,
                    rtm_module=rtm
                ):

                    # ==========================================================
                    # 1️⃣.2️⃣.1️⃣ Standard FFN activation
                    # ==========================================================
                    x = act_fn(x)

                    # ==========================================================
                    # 1️⃣.2️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 1️⃣.2️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm_module.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0 + exp_args.rtm_transition_amplification * transition_scale
                    # )

                    # x = x * (
                    #     1.0 + rtm_module.amplification * transition_scale
                    # )

                    global RTM_SCALE

                    x_base = x

                    x_mod = x * (
                        1.0 + rtm_module.amplification * transition_scale
                    )

                    x_mod = rtm_module(x_mod)
                    # ==========================================================
                    # 1️⃣.2️⃣.6️⃣ RTM refinement
                    # ==========================================================
                    # x = rtm_module(x)
                    x = x_base + RTM_SCALE * (x_mod - x_base)

                    return x
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               

                # =============================================================================
                # 📌📌 Replace decoder activation function
                # =============================================================================                 
                layer.activation_fn = wrapped_activation
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  


                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 FFN Insertion | RTM inserted into decoder layer {i}")                
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────




    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣ 🧩🧠 === Cross-Attention Insertion ♻️♻️ ================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣ 🧩🧠 === Cross-Attention Insertion ♻️♻️ ================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────

    elif exp_args.rtm_insertion_type == "cross_attn":
    
        # ================================================================================================
        # 2️⃣.0️⃣ 🧠 Build RTM_Former 🅰️🔼 
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:cross_attn🧬🧬")
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:cross_attn🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder Cross-Attention 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i >= len(model.decoder.layers) - exp_args.rtm_num_layers:

                # =============================================================================
                # 2️⃣.2️⃣.1️⃣📌📌 RTM module
                # =============================================================================
                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,
                    rtm_rotation=exp_args.rtm_rotation                    
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:

                    # 🔒 Configure RTM coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Attention Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────


                # =============================================================================
                # 2️⃣.2️⃣.2️⃣📌📌 Store original encoder-attention forward
                # =============================================================================
                original_encoder_attn_forward = layer.encoder_attn.forward

                # =============================================================================
                # 2️⃣.2️⃣.3️⃣📌📌 Wrapped Cross-Attention forward
                # =============================================================================
                def wrapped_encoder_attn_forward(
                    query,
                    key,
                    value,
                    **kwargs
                ):

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.1️⃣ Original Cross-Attention
                    # ==========================================================
                    x, attn = original_encoder_attn_forward(
                        query=query,
                        key=key,
                        value=value,
                        **kwargs
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0
                    #     +
                    #     exp_args.rtm_transition_amplification
                    #     *
                    #     transition_scale
                    # )

                    x = x * (
                        1.0
                        +
                        rtm.amplification
                        *
                        transition_scale
                    )

                    return x, attn
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────


                # =============================================================================
                # 📌📌 Replace encoder-attention forward
                # =============================================================================
                layer.encoder_attn.forward = wrapped_encoder_attn_forward
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────

                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 Cross-Attention Insertion | RTM inserted into decoder layer {i}") 

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────



    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 3️⃣ 🧩🧠 === FFN Depth-Weighted Insertion === ♻️♻️ =========================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 3️⃣ 🧩🧠 === FFN Depth-Weighted Insertion === ♻️♻️ =========================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    elif exp_args.rtm_insertion_type == "ffn_depth_weighted":

        # ================================================================================================
        # 1️⃣.0️⃣ 🧠 Build RTM_Former  
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: Depth-Weighted RTM_Former for insertion_typ:ffn_depth_weighted🧬🧬")
        print(f"♻️♻️ Loaded Model: Depth-Weighted RTM_Former for insertion_typ:ffn_depth_weighted🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # 🔒 IMPORTANT: reset RNG after importing RTM_Former, before the real training model is built (For Repeatable Initialization)
        set_seed_torch(seed1)
        set_seed_main(seed2)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print("\nMODEL INIT CHECK")
        print(model.decoder.layers[0].fc1.weight[0, :10])
        print()
        # ==========================================================
        # 🚦🔍 Select layers
        # ========================================================== 
        selected_layers = list(
            range(
                len(model.decoder.layers) - exp_args.rtm_num_layers,
                len(model.decoder.layers)
            )
        )
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────────────────────────────────       
        # =============================================================================
        # 1️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder FFNs 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i in selected_layers:

                original_activation_fn = layer.activation_fn

                layer_rank = selected_layers.index(i)


                # =====================================================
                # ⏭️⏭️ Forward direction Layer scale ⏭️⏭️
                # =====================================================               
                if exp_args.rtm_layer_scale_direction == "forward":
                    layer_scale = exp_args.rtm_layer_scale_min + (
                        1.0 - exp_args.rtm_layer_scale_min
                    ) * (
                        layer_rank / max(1, exp_args.rtm_num_layers - 1)
                    )
                # =====================================================
                # ⏪⏪ Reverse direction Layer scale ⏪⏪
                # =====================================================  
                elif exp_args.rtm_layer_scale_direction == "reverse":
                    layer_scale = 1.0 - (
                        1.0 - exp_args.rtm_layer_scale_min
                    ) * (
                        layer_rank / max(1, exp_args.rtm_num_layers - 1)
                    )
                # =====================================================
                # ⏪🔒 Reverse direction + Fixed Final Layer Scale ⏪🔒
                # =====================================================
                elif exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                    if layer_rank == exp_args.rtm_num_layers - 1:
                        layer_scale = exp_args.rtm_final_layer_scale
                    else:
                        layer_scale = 1.0 - (
                            1.0 - exp_args.rtm_layer_scale_min
                        ) * (
                            layer_rank / max(1, exp_args.rtm_num_layers - 2)
                        ) 
                # ────────────────────────────────────────────────────────────────────────────────────────
                else:
                    raise ValueError(f"Unknown RTM layer scale direction: {exp_args.rtm_layer_scale_direction}")
                # ────────────────────────────────────────────────────────────────────────────────────────

                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,  
                    gate_type=exp_args.rtm_gate_type,
                    rtm_rotation=exp_args.rtm_rotation                                     
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility + Parameter Tracking)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:
                    
                    # 🔒 Configure RTM learnable coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Activation Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ─────────────────────────────────────────────────────────────────────────────────────────────────                
                # ─────────────────────────────────────────────────────────────────────────────────────────────────

                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                def wrapped_activation(
                    x,
                    act_fn=original_activation_fn,
                    rtm_module=rtm,
                    layer_scale=layer_scale
                ):

                    # ==========================================================
                    # 1️⃣.2️⃣.1️⃣ Standard FFN activation
                    # ==========================================================
                    x = act_fn(x)

                    # ==========================================================
                    # 1️⃣.2️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 1️⃣.2️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm_module.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0 + exp_args.rtm_transition_amplification * transition_scale
                    # )

                    # x = x * (
                    #     1.0 + rtm_module.amplification * transition_scale
                    # )

                    global RTM_SCALE

                    x_base = x

                    x_mod = x * (
                        1.0 + rtm_module.amplification * transition_scale
                    )

                    x_mod = rtm_module(x_mod)
                    # ==========================================================
                    # 1️⃣.2️⃣.6️⃣ RTM refinement
                    # ==========================================================

                    # ----------------------------------------------------------
                    # 🏔️ Depth-Aware RTM Scaling (Layer-Adaptive Modulation)
                    # ---------------------------------------------------------- 
                    if exp_args.rtm_depth_weight:

                        x = x_base + RTM_SCALE * layer_scale * (x_mod - x_base)

                    # ----------------------------------------------------------
                    # ⚖️ Uniform RTM Scaling (Same Modulation Across Layers)
                    # ----------------------------------------------------------
                    else:

                        x = x_base + RTM_SCALE * (x_mod - x_base)
                    # ─────────────────────────────────────────────────────────────────────────────────────────────────

                    return x
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               

                # =============================================================================
                # 📌📌 Replace decoder activation function
                # =============================================================================                 
                layer.activation_fn = wrapped_activation
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  


                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 FFN Insertion | RTM inserted into decoder layer {i} | layer_scale={layer_scale:.3f}")              
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: Depth-Weighted RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    else:
        raise ValueError(f"Unknown RTM insertion type: {exp_args.rtm_insertion_type}")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print(f"📣📣 RTM_Former for insertion_typ={exp_args.rtm_insertion_type} | Loaded Model={exp_args.net }♻️♻️")
    print(f"📣📣 RTM_Former for insertion_typ={exp_args.rtm_insertion_type} | Loaded Model={exp_args.net }♻️♻️")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────



    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  ======== 🔖  5.1.1 Model Configuration Hyperparameter Summary ==================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 
    if exp_args.net == "RTM_Former":

        rtm_log = (
            f"📣 RTM Configuration             : "
            f"alpha={exp_args.rtm_alpha} | "
            f"amp={exp_args.rtm_transition_amplification} | "
            f"layers={exp_args.rtm_num_layers} | "
            f"insertion={exp_args.rtm_insertion_type} | "
            f"scale_init={exp_args.rtm_scale} | "
            f"start_epoch={exp_args.rtm_start_epoch} | "
            f"full_epoch={exp_args.rtm_full_epoch}"
        )

        # Depth-weighted RTM only
        if exp_args.rtm_insertion_type == "ffn_depth_weighted":

            rtm_log += (
                f" | layer_scale_min={exp_args.rtm_layer_scale_min}"
                f" | layer_direction={exp_args.rtm_layer_scale_direction}"
                f" | gate={exp_args.rtm_gate_type}"
            )

            # Only exists for reverse_final_fixed
            if exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                rtm_log += (
                    f" | final_layer_scale={exp_args.rtm_final_layer_scale}"
                )

        print(rtm_log)


    else:
        print(
            f"📣 Baseline Configuration        : "
            f"model={exp_args.net}"
        )
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ======== 🔖🔖  5.2  Insert model to device 🔖🔖================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

model = model.to(DEVICE)
 # ─────────────────────────────────────────────────────────────────────────────────────────────────
print(f"🚀🚀 Model moved to device: {DEVICE} ♻️♻️")
print(f"🚀🚀 Model moved to device: {DEVICE} ♻️♻️")
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ======== 🔖🔖 5.3  Model Validation & Sanity Checks 🔖🔖=======================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 🔍 Print custom activations to confirm they are used
for name, module in model.named_modules():
    if isinstance(module, nn.Module) and module.__class__.__name__ in ["TanhExp", "FFTGate"]:
        print(f"🔎 Found custom activation in {name}: {module}")


# ✅ Sanity check Device
if DEVICE.type == "cuda":
    print(f"🚀 Model moved to CUDA: {torch.cuda.get_device_name(DEVICE)}")
else:
    print("⚠️ Using CPU (CUDA not available)")
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 6. PATH DEFINATION | XXX --------------------------------------------------------#####################
########################################################################################################################


# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 1️⃣ 📌📌 ======== ENSURE DIRECTORY EXIST ========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# # ─────────────────────────────────────────────────────────────────────────────────────────────────
# # 🟡 === Checkpoint directories ===
# if not os.path.exists('checkpoint'):
#     os.makedirs('checkpoint')
# # ─────────────────────────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Results directories ===
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}')

if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === LR Log directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Training Log directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Model Summary directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')
# ────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣ 📌📌 ======== DEFINE DIRECTORY ==============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.1️⃣🔖 ==== Checkpoint Directory Selection | Normal Run vs Crash Run | save every epoch ===== 
# ────────────────────────────────────────────────────────────────────────────────────────────────

# 🅰️💾 Normal run: save checkpoints in standard folder
if not exp_args.crash_run:
    checkpoint_dir = f'./checkpoints/{exp_args.dataset_name}/{exp_args.net}/'

    # 📁 Create checkpoint folder if it does not exist
    os.makedirs(checkpoint_dir, exist_ok=True)

    print(f"🅰️💾 Normal Run Enabled | Saving checkpoints to: {checkpoint_dir}")
#---------------------------------------------------------------------------------------------------
# 🅱️♻️ Crash run: save checkpoints in recovery folder
else:
    checkpoint_dir = f'./checkpoints_crash_run/{exp_args.dataset_name}/{exp_args.net}/'

    # 📁 Create checkpoint folder if it does not exist    
    os.makedirs(checkpoint_dir, exist_ok=True)

    print(f"🅱️♻️ Crash Run Enabled | Saving recovery checkpoints to: {checkpoint_dir}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.2️⃣🔖 === Main Test & Train Results === 
train_results_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/Train_{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
test_results_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/Test_{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.3️⃣🔖  === LR, Training & Summary logs  === 
LR_save_paths = {
    "LR_history": f"./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_LR_log.txt"
}
save_paths = {
    "log_history": f"./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_training_logs.txt"
}
# ────────────────────────────────────────────────────────────────────────────────────────────────

# 2️⃣.4️⃣🔖 ===  Model summary path === 
model_summary_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_model_summary.txt'
model_summary_full_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_model_summary_full.txt'
# ────────────────────────────────────────────────────────────────────────────────────────────────

# # ✅🔖 === Path template for saving checkpoints at every epoch | Part2 (if scratch occured) ===
# checkpoint_dir = f'./checkpoints/{exp_args.dataset_name}/tanhexp_AfterCrash_41_49epochs/'
# os.makedirs(checkpoint_dir, exist_ok=True)
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 7. TRAINING INITIALIZATION & OPTIMIZATION SETUP | XXX ---------------------------#####################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ========  7.1 AMP GradScaler ===============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Initialize AMP GradScaler once globall
scaler = torch.cuda.amp.GradScaler()
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  Training Criterion ===============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

criterion = LabelSmoothingCrossEntropy(smoothing=exp_args.smoothing)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ========   Optimizer Configuration =========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ♻️ Adam optimizer (β1=0.9, β2=0.98, eps=1e-9) ♻️
# ─────────────────────────────────────────────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=exp_args.lr,
    betas=(0.9, 0.98),
    eps=exp_args.eps,
    weight_decay=exp_args.weight_decay,
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 4️⃣ ========  Learning Rate Scheduler ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ✅  Transformer LR schedule (warmup + inverse sqrt decay)
# --------------------------

warmup_updates = exp_args.warmup_updates   # configurable warmup steps
peak_lr = exp_args.lr

# ────────────────────────────────────────────────────────────────────────────────────────────────
def lr_lambda(step):
    step = max(1, step)
    if step == 1:
        print(f"🚀 Warmup started at step {step} (target peak LR = {peak_lr}).")
    if step == warmup_updates:
        actual_lr = optimizer.param_groups[0]["lr"]
        print(f"✅ Warmup finished at step {step}. Peak LR reached: {actual_lr:.8f}")
    if step == warmup_updates + 1:
        print(f"📉 Inverse square root decay started at step {step}.")
    
    if step <= warmup_updates:
        # Linear warmup
        scale = step / warmup_updates
    else:
        # Inverse square root decay
        scale = (warmup_updates ** 0.5) / (step ** 0.5)
    return scale

# ────────────────────────────────────────────────────────────────────────────────────────────────

lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 5️⃣ ========  Global Tracking Variables ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ global initialization
best_bleu = 0.0
total_test_duration = 0.0
RTM_SCALE_LOG = ""
# ─────────────────────────────────────────────────────────────────────────────────────────────────

Python: 3.10.16 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:19:12) [MSC v.1929 64 bit (AMD64)]
Torch: 2.2.0+cu121 | CUDA available: True
Fairseq: 0.12.2
OmegaConf: 2.0.6
Hydra-Core: 1.0.7
✅ Current working directory: C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH
✅ sys.path updated:
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\python310.zip
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\DLLs
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env
   📂 
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\win32
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\win32\lib
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\Pythonwin
   📂 C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH
   📂 C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH\models
   📂 C:\

c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\modules\module.py:1150: UserWarning: expandable_segments not supported on this platform (Triggered internally at ..\c10/cuda/CUDAAllocatorConfig.h:30.)
  return t.to(device, dtype if t.is_floating_point() or t.is_complex() else None, non_blocking)


In [ ]:
########################################################################################################################
####-------| NOTE 11. TEST ALL CHECKPOINTS (BLEU per epoch)  | XXX -----------------------------########################
########################################################################################################################
########################################################################################################################
####-------| NOTE 11. TEST ALL CHECKPOINTS (BLEU per epoch)  | XXX -----------------------------########################
########################################################################################################################


# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ======== Paths for checkpoint BLEU sweeps ===================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Paths for checkpoint BLEU sweeps
test_checkpoint_results_path = (
    f'./Results/{exp_args.dataset_name}/{exp_args.net}/'
    f'TestCheckpoints_{exp_args.net}_'
    f'{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
)
# ────────────────────────────────────────────────────────────────────────────────────────────────


# # ✅ Paths for checkpoint BLEU sweeps | Part2 (if Cratch occured)
# test_checkpoint_results_path = (
#     f'./Results/{exp_args.dataset_name}/{exp_args.act_name}/'
#     f'TestCheckpoints_{exp_args.act_name}_{exp_args.net}_'
#     f'{exp_args.dataset_name}_{exp_args.optimizer1}_tanhexp_AfterCrash_41_49epochs.txt'
# )
# ────────────────────────────────────────────────────────────────────────────────────────────────


# # ─────────────────────────────────────────────────────────────────────────────────────────────────
# #  2️⃣ ========  BLEU Detokenization (🔖 EN/FR/DE) ================================================
# # ───────────────────────────────────────────────────────────────────────────────────────────────── 
# # ────────────────────────────────────────────────────────────────────────────────────────────────
# def detokenize_for_bleu(s: str) -> str:
#     s = s.replace("@@ ", "").replace("@@", "")  # BPE
#     s = s.replace("▁", " ")                     # SentencePiece
#     s = " ".join(s.split())                     # normalize spaces
#     s = re.sub(r"\s+([?.!,])", r"\1", s)        # punctuation
#     return s.strip()
# # ────────────────────────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  BLEU Detokenization (🔖 Chinese) =================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
def detokenize_for_bleu(s: str) -> str:
    # For Chinese, don't remove SentencePiece boundaries or spaces | Only remove BPE continuation markers
    # sacreBLEU's "zh" tokenizer expects raw characters.
    return s.replace("@@ ", "").replace("@@", "").strip()
# ────────────────────────────────────────────────────────────────────────────────────────────────




# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ======== Test_Checkpoint Function ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ test_checkpoint function
def test_checkpoint_epochs(task, model, criterion, checkpoint_dir, test_checkpoint_results_path):
    """
    Evaluate BLEU (beam=5) for all checkpoints in checkpoint_dir.
    Save results to test_checkpoint_results_path in the same format as normal test logs.
    """
# ────────────────────────────────────────────────────────────────────────────────────────────────
    global RTM_SCALE, RTM_SCALE_LOG
    # ────────────────────────────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Collect checkpoints sorted by epoch number in filename
    def extract_epoch_num(filename):
        match = re.search(r"epoch_(\d+)", filename)
        return int(match.group(1)) if match else float("inf")  # inf if no epoch number

    ckpts = sorted(
        glob.glob(os.path.join(checkpoint_dir, "*.pt")),
        key=lambda x: extract_epoch_num(os.path.basename(x))
    )

    if not ckpts:
        print(f"⚠️ No checkpoints found in {checkpoint_dir}")
        return

    # # ────────────────────────────────────────────────────────────────────────────────────────────────
    # # ✅ Reset output file
    # os.makedirs(os.path.dirname(test_checkpoint_results_path), exist_ok=True)
    # with open(test_checkpoint_results_path, "w", encoding="utf-8") as f:
    #     f.write("")
    # # ────────────────────────────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 📦 Reset output file + write model status
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    #1️⃣ Create output folder if it does not exist
    os.makedirs(os.path.dirname(test_checkpoint_results_path), exist_ok=True)

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Normal run: reset file
    # ✅ Crash run : append to existing file
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣♻️ Normal run: reset file
    if not exp_args.crash_run:

        with open(test_checkpoint_results_path, "w", encoding="utf-8") as f:
            f.write("")

    # 3️⃣⏭️ Crash run : append to existing file
    else:

        with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
            f.write("\n♻️ Resuming checkpoint testing after crash\n\n")
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 4️⃣✍️  Write test result header (model information)
    with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
        f.write("")

        # ---------------------------------------------------------------------------
        # 4️⃣.1️⃣🏷️ Testing Model Status
        # ---------------------------------------------------------------------------
        if exp_args.net == "RTM_Former":

            f.write(
                f"Model: RTM_Former\n\n"
            )

        else:

            f.write(
                f"Model: Baseline Fairseq Transformer\n\n"
            )        
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ────────────────────────────────────────────────────────────────────────────────────────────────



    # ────────────────────────────────────────────────────────────────────────────────────────────────
    best_bleu = 0.0
    total_test_duration = 0.0
    # ────────────────────────────────────────────────────────────────────────────────────────────────


    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    for ckpt in ckpts:

        # # ────────────────────────────────────────────────────────────────────────────────
        # # ✅ 📌📌 Load checkpoint (OLD) 🅰️🔼 
        # # ────────────────────────────────────────────────────────────────────────────────        
        # print(f"🔄 Loading checkpoint: {ckpt}")
        # state = torch.load(ckpt, map_location=DEVICE)
        # model.load_state_dict(state["model_state"], strict=False)
        # model.to(DEVICE)
        # model.eval()
        # --------------------------------------------------------------------------------------------------
        # ────────────────────────────────────────────────────────────────────────────────
        # ✅📌📌 Load checkpoint (NEW) 🅱️🔼 Restore RTM schedule for checkpoint testing using same strategy as main loop
        # ────────────────────────────────────────────────────────────────────────────────
        if exp_args.net == "Transformer":
            print(f"🔄 Loading checkpoint Baseline Transformer : {ckpt}")
            state = torch.load(ckpt, map_location=DEVICE)
            model.load_state_dict(state["model_state"], strict=False)
            model.to(DEVICE)
            model.eval()
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        elif exp_args.net == "RTM_Former":
            print(f"🔄 Loading checkpoint RTM_Former : {ckpt}")
            state = torch.load(ckpt, map_location=DEVICE)
            model.load_state_dict(state["model_state"], strict=False)            
            

            RTM_START_EPOCH = exp_args.rtm_start_epoch
            RTM_FULL_EPOCH = exp_args.rtm_full_epoch
            ckpt_epoch = state["epoch"]

            if ckpt_epoch < RTM_START_EPOCH:
                RTM_SCALE = exp_args.rtm_scale
            else:
                progress = min(
                    1.0,
                    (ckpt_epoch - RTM_START_EPOCH) / (RTM_FULL_EPOCH - RTM_START_EPOCH)
                )

                RTM_SCALE = 0.5 * (1.0 - math.cos(math.pi * progress))

            print(f"📐 Checkpoint RTM_SCALE for epoch {ckpt_epoch}: {RTM_SCALE:.3f}")

            if exp_args.rtm_insertion_type == "ffn_depth_weighted":

                if exp_args.rtm_layer_scale_direction == "forward":
                    depth_scales = [
                        exp_args.rtm_layer_scale_min
                        + (1.0 - exp_args.rtm_layer_scale_min)
                        * (j / max(1, exp_args.rtm_num_layers - 1))
                        for j in range(exp_args.rtm_num_layers)
                    ]

                elif exp_args.rtm_layer_scale_direction == "reverse":
                    depth_scales = [
                        1.0
                        - (1.0 - exp_args.rtm_layer_scale_min)
                        * (j / max(1, exp_args.rtm_num_layers - 1))
                        for j in range(exp_args.rtm_num_layers)
                    ]

                elif exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                    depth_scales = []

                    for j in range(exp_args.rtm_num_layers):
                        if j == exp_args.rtm_num_layers - 1:
                            scale_j = exp_args.rtm_final_layer_scale
                        else:
                            scale_j = (
                                1.0
                                - (1.0 - exp_args.rtm_layer_scale_min)
                                * (j / max(1, exp_args.rtm_num_layers - 2))
                            )

                        depth_scales.append(scale_j)

                else:
                    raise ValueError(
                        f"Unknown RTM layer scale direction: {exp_args.rtm_layer_scale_direction}"
                    )

                final_scales = [RTM_SCALE * s for s in depth_scales]

                RTM_SCALE_LOG = f" | RTM_SCALE={RTM_SCALE:.3f}"
                RTM_SCALE_LOG += (
                    f" | RTM_layer_scales({exp_args.rtm_layer_scale_direction})="
                    f"{[round(s, 3) for s in final_scales]}"
                )

                print(
                    f"📐 Checkpoint final RTM layer scales "
                    f"({exp_args.rtm_layer_scale_direction}): "
                    f"{[round(s, 3) for s in final_scales]}"
                )
            model.to(DEVICE)
            model.eval()               
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        else:
            raise ValueError(f"Unknown model type: {exp_args.net}")
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ────────────────────────────────────────────────────────────────────────────────────────────────


        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Build test iterator
        test_iter = task.get_batch_iterator(
            dataset=task.dataset("test"),
            max_tokens=fairseq_args.max_tokens,   # cut in half option: fairseq_args.max_tokens // 2
            max_positions=(exp_args.max_source_positions, exp_args.max_target_positions),
            seed=seed1,
            num_shards=exp_args.num_shards,
            shard_id=exp_args.shard_id,
            num_workers=exp_args.num_workers,
            ignore_invalid_inputs=exp_args.ignore_invalid_inputs,
        ).next_epoch_itr(shuffle=False)
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ References
        true_references = []
        for sample in test_iter:
            targets = sample["target"]
            for t in targets:
                t_no_pad = t[t != tgt_dict.pad()]
                ref_str = tgt_dict.string(t_no_pad, bpe_symbol="@@")
                true_references.append(detokenize_for_bleu(ref_str))
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Rebuild iterator for decoding
        test_iter = task.get_batch_iterator(
            dataset=task.dataset("test"),
            max_tokens=fairseq_args.max_tokens,    # cut in half option: fairseq_args.max_tokens // 2
            max_positions=(exp_args.max_source_positions, exp_args.max_target_positions),
            seed=seed1,
            num_shards=exp_args.num_shards,
            shard_id=exp_args.shard_id,
            num_workers=exp_args.num_workers,
            ignore_invalid_inputs=exp_args.ignore_invalid_inputs,
        ).next_epoch_itr(shuffle=False)
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Build generator with beam=5
        gen_cfg = deepcopy(cfg.generation)
        gen_cfg.beam = exp_args.beam         # 🎀🔖 5
        gen_cfg.lenpen = exp_args.lenpen     # 🎀🔖 1.0 
        generator = task.build_generator([model], gen_cfg)
        # ────────────────────────────────────────────────────────────────────────────────────────────────


 

        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Run evaluation
        sys_outputs, test_loss = [], 0.0
        start_time = time.time()

        with torch.no_grad():
            with tqdm(enumerate(test_iter), total=len(test_iter), desc=f"Testing {os.path.basename(ckpt)}") as progress:
                for i, sample in progress:   # 👈 keep i!
                    sample = utils.move_to_cuda(sample) if torch.cuda.is_available() else sample
                    src_tokens = sample["net_input"]["src_tokens"]
                    prev_output_tokens = sample["net_input"]["prev_output_tokens"]
                    target = sample["target"]
                    src_lengths = (src_tokens != src_dict.pad()).sum(dim=1)

                    with torch.cuda.amp.autocast():
                        logits, _ = model(src_tokens, src_lengths, prev_output_tokens)
                        pad = tgt_dict.pad()
                        logits2d = logits.view(-1, logits.size(-1))
                        target1d = target.view(-1)
                        mask = target1d != pad
                        loss = criterion(logits2d[mask], target1d[mask]) if mask.any() else logits2d.sum() * 0.0
                    test_loss += loss.item()

                    hypos = task.inference_step(generator, [model], sample)
                    for j, hypos_j in enumerate(hypos):
                        hypo_tokens = [tok for tok in hypos_j[0]["tokens"].tolist() if tok != tgt_dict.pad()]
                        sys_str = tgt_dict.string(hypo_tokens, bpe_symbol="@@")
                        sys_outputs.append(detokenize_for_bleu(sys_str))

                    # ✅ Update progress bar postfix with running average
                    progress.set_postfix(loss=round(test_loss / (i + 1), 3))
       # ────────────────────────────────────────────────────────────────────────────────────────────────




       # ────────────────────────────────────────────────────────────────────────────────────────────────
        avg_loss = test_loss / max(1, len(test_iter))
        # bleu = sacrebleu.corpus_bleu(sys_outputs, [true_references])                 #🔖 EN/FR/DE
        bleu = sacrebleu.corpus_bleu(sys_outputs, [true_references], tokenize="zh")    #🔖 Chinese

        duration = time.time() - start_time
        total_test_duration += duration
       # ────────────────────────────────────────────────────────────────────────────────────────────────


       # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Write results
        with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
            f.write(f"Checkpoint {os.path.basename(ckpt)} | Test Loss: {avg_loss:.3f} | sacreBLEU: {bleu.score:.2f} | Beam=5\n")

        print(f"✅ {os.path.basename(ckpt)} | Loss {avg_loss:.3f} | sacreBLEU {bleu.score:.2f}")


        # 🧹 add visual spacing between runs
        tqdm.write("")  
       # ────────────────────────────────────────────────────────────────────────────────────────────────

       # ────────────────────────────────────────────────────────────────────────────────────────────────
        if bleu.score > best_bleu:
            best_bleu = bleu.score

    with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
        f.write(f"\n🏆 Best BLEU Score: {best_bleu:.2f}\n")
        total_mins, total_secs = divmod(total_test_duration, 60)
        f.write(f"\n🕒 Total Test Time over all checkpoints: {int(total_mins)} min {total_secs:.2f} sec\n")

    print(f"🏆 Best BLEU over checkpoints: {best_bleu:.2f}")
    # ────────────────────────────────────────────────────────────────────────────────────────────────



# ────────────────────────────────────────────────────────────────────────────────────────────────
# # ✅ Free unused GPU memory BEFORE training starts
# torch.cuda.empty_cache()
# torch.cuda.reset_peak_memory_stats()

# 4️⃣ Call the function
test_checkpoint_epochs(task, model, criterion, checkpoint_dir, test_checkpoint_results_path)
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────────────────────────

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_18_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 18: 1.000


Testing epoch_18_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt:   0%|          | 0/82 [00:00<?, ?it/s]c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)
c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\functional.py:5109: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
Testing epoch_18_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:46<00:00,  1.75it/s, loss=4.51]


✅ epoch_18_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.515 | sacreBLEU 18.48

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_19_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 19: 1.000


Testing epoch_19_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:40<00:00,  2.03it/s, loss=4.47]


✅ epoch_19_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.473 | sacreBLEU 18.00

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_20_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 20: 1.000


...

Testing epoch_23_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:39<00:00,  2.06it/s, loss=4.42]


✅ epoch_23_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.417 | sacreBLEU 18.57

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_24_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 24: 1.000


Testing epoch_24_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:43<00:00,  1.87it/s, loss=4.41]


✅ epoch_24_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.409 | sacreBLEU 18.61

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_25_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 25: 1.000


: 

Restarted pytorch_env (Python 3.10.16)

In [ ]:
#####-------------------------------- NOTE MAIN IWSLT17 EN-ZH NOTE --------------------------------------------------#####
##########################################################################################################################
######################|--------------------------------------------------------------|####################################
###################################🔗 MAIN | TRAIN | TEST LOOP 🔗########################################################
######################|--------------------------------------------------------------|####################################
##########################################################################################################################
#####-------------------------------- NOTE MAIN IWSLT17 EN-ZH NOTE --------------------------------------------------#####



# 📄  main_IWSLT17En_Zh.py
########################################################################################################################
####-------| NOTE 1.1 IMPORTS LIBRARIES  | XXX -------------------------------------------------------##################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 📜 === Enable flexible CUDA memory allocation to reduce fragmentation ===
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ======================================================================================================
# 📜 === Core Libraries ===
# ======================================================================================================
import sys
import numpy as np
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import time
import random
import math
import re
import glob
from copy import deepcopy
import sacrebleu  # ✅ for BLEU evaluation
from timm.loss import LabelSmoothingCrossEntropy
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ======================================================================================================
# 📜 === Fairseq & Supporting Imports ===
# ======================================================================================================
import importlib.metadata as importlib_metadata
import fairseq  # type: ignore
from fairseq.data import Dictionary, data_utils, iterators          # type: ignore
from fairseq.tasks.translation import TranslationTask               # type: ignore
from fairseq import options                                         # type: ignore
from fairseq.dataclass.utils import convert_namespace_to_omegaconf  # type: ignore                                             
from fairseq import utils                                           # type: ignore 
import hydra                                                        # type: ignore  
# ────────────────────────────────────────────────────────────────────────────────────────────────
from ptflops import get_model_complexity_info
from calflops import calculate_flops
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ======================================================================================================
# ♻️ === Print environment summary for sanity check ===
# ======================================================================================================
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Fairseq:", fairseq.__version__)
print("OmegaConf:", importlib_metadata.version("omegaconf"))
print("Hydra-Core:", importlib_metadata.version("hydra-core"))
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 1.2. DEFINE PROJECT PATH | XXX ---------------------------------------------------####################
########################################################################################################################

# ✅ Define working directory
Project_PATH = r"C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH"
if os.getcwd() != Project_PATH:
    os.chdir(Project_PATH)
print(f"✅ Current working directory: {os.getcwd()}")


# ✅ Define absolute paths
PROJECT_PATH = Project_PATH
MODELS_PATH = os.path.join(Project_PATH, "models")
ACTIVATION_PATH = os.path.join(Project_PATH, "activation")


# ✅ Ensure necessary paths are in sys.path
for path in [PROJECT_PATH, MODELS_PATH, ACTIVATION_PATH]:
    if path not in sys.path:
        sys.path.append(path)

# ✅ Print updated sys.path for debugging
print("✅ sys.path updated:")
for path in sys.path:
    print("   📂", path)
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 1.3. OTHER IMPORTS | XXX ---------------------------------------------------------####################
########################################################################################################################

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 📜 ============  Imput parser safe for afno because of --fno-bias, etc  =======================
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 1️⃣✅ Import parser from parser_IWSLT17En_Zh.py
from parser_IWSLT17En_Zh import get_parser

# ✅ Create parser and parse arguments
parser = get_parser()
# args, unknown = parser.parse_known_args()

# ✅ IMPORTANT: Do NOT read Jupyter / VSCode kernel arguments
# This prevents the "--f" ambiguity issue
exp_args = parser.parse_args(args=[])

num_aug_splits = exp_args.aug_splits

print(f"✅ Parser imported successfully | num_aug_splits = {num_aug_splits}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣✅ Import Fairseq parser/config
from fairseq_config_IWSLT17En_Zh import get_fairseq_parser
# ✅ Build Fairseq configuration
fairseq_args, cfg = get_fairseq_parser(exp_args)


num_aug_splits = exp_args.aug_splits

print(f"✅ Parser imported successfully | num_aug_splits = {num_aug_splits}")

print(f"✅ Encoder Embed Dim: {fairseq_args.encoder_embed_dim}")
print(f"✅ Decoder Layers: {fairseq_args.decoder_layers}")
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 3️⃣✅ Import model summary utility
from models.model_summary.model_summary import save_model_summary
from models.model_summary.model_summary_full import save_model_summary_full

print("✅ model_summary.py loaded successfully")
print("✅ model_summary_full.py loaded successfully")
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 2. SEEDING FOR REPRODUCIBILITY | XXX ---------------------------------------------####################
########################################################################################################################

# ────────────────────────────────────────────────────────────────────────────────────────────────
def set_seed_torch(seed):
    torch.manual_seed(seed)
# ────────────────────────────────────────────────────────────────────────────────────────────────
def set_seed_main(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True      # ✅ Default: "False" for Faster, non-deterministic kernels | "True" to Ensure deterministic behavior for CuDNN (Slower)
    torch.backends.cudnn.benchmark = False         # ✅ Default: "True" for Autotune kernels for performance   | "False" Disable CuDNN's autotuning for reproducibility (Slower)

    torch.backends.cuda.matmul.allow_tf32 = False  # ✅ Disable TF32 (strict reproducibility)
    torch.backends.cudnn.allow_tf32 = False        # ✅ Disable TF32 (strict reproducibility)

    # torch.use_deterministic_algorithms(True, warn_only=True)

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ ============= Define Seed =============
seed1, seed2 = exp_args.seed1, exp_args.seed2
set_seed_torch(seed1)  
set_seed_main(seed2)  
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 3.1. PROCESSING DATASET 1 | XXX ---- TOKENIZATION & DATASET PREPARATION |---------####################
########################################################################################################################
########################################################################################################################
####-------| NOTE 3.1. PROCESSING DATASET 1 | XXX ------------| ENGLISH→CHINESE |-------------------####################
########################################################################################################################

"""
Stage 1:
Train SentencePiece tokenizer, encode the raw English→Chinese corpus,
and create aligned train/test text splits for neural machine translation.
"""

# ─────────────────────────────────────────────────────────────────────────────────────────────────
print(
    f"🔎🔎 run_dataset_tokenization: {exp_args.run_dataset_tokenization} "
    f"{'(RUNNING)✔️✔️' if exp_args.run_dataset_tokenization else '(SKIPPED)❌❌'}"
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if exp_args.run_dataset_tokenization:
    from datasets import load_dataset    
    import sentencepiece as spm

    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Path for EN→ZH dataset
    DATA_PATH = r"C:\Users\emeka\Research\ModelCUDA\Big_Data_Journal\Comparison\Code\Paper\github3\IWSLT\Transformer_Baselines_EN_ZH\Dataset"
    os.makedirs(DATA_PATH, exist_ok=True)
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  1️⃣ ======== Download and export official IWSLT17 dataset =======================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Download to your folder (as the Hugging Face cache)
    en_zh = load_dataset(
        "IWSLT/iwslt2017", "iwslt2017-en-zh",
        cache_dir=DATA_PATH,
        trust_remote_code=True
    )
    zh_en = load_dataset(
        "IWSLT/iwslt2017", "iwslt2017-zh-en",
        cache_dir=DATA_PATH,
        trust_remote_code=True
    )  # optional
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  2️⃣ ======== Dataset verification ===============================================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ sanity check (should show ~231k train)
    print(f"EN→ZH dataset: {en_zh}")
    print(f"ZH→EN dataset: {zh_en}")

    print("EN→ZH splits:", list(en_zh.keys()), {k: len(en_zh[k]) for k in en_zh.keys()})
    print("ZH→EN splits:", list(zh_en.keys()), {k: len(zh_en[k]) for k in zh_en.keys()})

    # optional assertions
    assert set(en_zh.keys()) == {"train", "validation", "test"}
    assert set(zh_en.keys()) == {"train", "validation", "test"}
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  3️⃣ ======== Export official train/validation/test splits ======================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Export official IWSLT17 splits to plain text (EN/ZH) under raw_zh_en_dataset_path
    def dump_split(ds_dict, split, src, tgt, out_dir):
        src_fp = os.path.join(out_dir, f"{split}.{src}")
        tgt_fp = os.path.join(out_dir, f"{split}.{tgt}")
        with open(src_fp, "w", encoding="utf-8") as fs, open(tgt_fp, "w", encoding="utf-8") as ft:
            for row in ds_dict[split]:
                tr = row["translation"]
                fs.write((tr[src] or "").strip() + "\n")
                ft.write((tr[tgt] or "").strip() + "\n")
        print(f"Wrote {split}: {src_fp} | {tgt_fp}")

    for split in ["train", "validation", "test"]:
        dump_split(en_zh, split, "en", "zh", DATA_PATH)
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  4️⃣ ========  Train SentencePiece model =========================================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────     

    print("🧠 Training SentencePiece model for EN→ZH...")

    # Use TRAIN split for SPM training
    SRC_EN = os.path.join(DATA_PATH, "train.en")
    TGT_ZH = os.path.join(DATA_PATH, "train.zh")

    spm_model_prefix = os.path.join(DATA_PATH, "spm_enzh")

    spm.SentencePieceTrainer.Train(
        input=f"{SRC_EN},{TGT_ZH}",
        model_prefix=spm_model_prefix,
        vocab_size=10000,
        character_coverage=0.9995,   # ✅ --character_coverage for rich charcater langauage: Japanese or Chinese = 0.9995 | other languages with small character set: 1.0
        model_type="bpe"
    )
    print(f"✅ SentencePiece model saved to: {spm_model_prefix}.model / .vocab")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  5️⃣ ========  Encode files ======================================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    print("⚙️ Encoding dataset...")

    # ✅ Load the trained SentencePiece model
    sp = spm.SentencePieceProcessor(model_file=f"{spm_model_prefix}.model")

    def encode_file(in_file, out_file):
        with open(in_file, encoding="utf-8") as fin, open(out_file, "w", encoding="utf-8") as fout:
            for line in fin:
                fout.write(" ".join(sp.encode(line.strip(), out_type=str)) + "\n")

    # ✅ Encode all official splits (no custom re-split)
    for split in ["train", "validation", "test"]:
        encode_file(os.path.join(DATA_PATH, f"{split}.en"), os.path.join(DATA_PATH, f"{split}.spm.en"))
        encode_file(os.path.join(DATA_PATH, f"{split}.zh"), os.path.join(DATA_PATH, f"{split}.spm.zh"))
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  6️⃣ ======== Use official train/validation/test splits =========================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print(f"✅ Using official IWSLT17 train/validation/test splits.")

    print(f"✅ Encoded EN→ZH data saved under: {DATA_PATH}")
    print(f"✅ Next: run fairseq-preprocess to binarize (on *.spm.en/*.spm.zh).")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 3.2. PROCESSING DATASET 2 | XXX ----- FAIRSEQ PREPROCESSING & BINARIZATION |------####################
########################################################################################################################
########################################################################################################################
####-------| NOTE 3.2. PROCESSING DATASET 2 | XXX ------------| ENGLISH→CHINESE |--------------------####################
########################################################################################################################

"""
Stage 2:
Preprocess the tokenized English→Chinese dataset using fairseq-preprocess
to build vocabularies and generate Fairseq binary training files.
"""

# ─────────────────────────────────────────────────────────────────────────────────────────────────
print(
    f"🔎🔎 run_fairseq_preprocessing: {exp_args.run_fairseq_preprocessing} "
    f"{'(RUNNING)✔️✔️' if exp_args.run_fairseq_preprocessing else '(SKIPPED)❌❌'}"
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if exp_args.run_fairseq_preprocessing:
        
    import subprocess


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  1️⃣ ======== Check dataset files before preprocessing ==========================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    # ✅ Same folder you used above
    DATA_DIR = r"C:\Users\emeka\Research\ModelCUDA\Big_Data_Journal\Comparison\Code\Paper\github3\IWSLT\Transformer_Baselines_EN_ZH\Dataset"
    DESTDIR  = r"data-bin\iwslt17.en-zh"  # name it how you like
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    required_files = [
        os.path.join(DATA_DIR, "train.spm.en"),
        os.path.join(DATA_DIR, "train.spm.zh"),
        os.path.join(DATA_DIR, "validation.spm.en"),
        os.path.join(DATA_DIR, "validation.spm.zh"),
        os.path.join(DATA_DIR, "test.spm.en"),
        os.path.join(DATA_DIR, "test.spm.zh"),
    ]

    for f in required_files:
        assert os.path.exists(f), f"❌ Missing file: {f}"

    print("✅ All SentencePiece files found.")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    
    
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  2️⃣ ======== Configure preprocessing ============================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print("🚀 Running fairseq-preprocess for EN→ZH...")

    cmd = [
        "fairseq-preprocess",
        "--source-lang", "en",             # ✅Source = English
        "--target-lang", "zh",             # ✅Target = Chinese
        # use the SPM-encoded prefixes you just created
        "--trainpref", os.path.join(DATA_DIR, "train.spm"),
        "--validpref", os.path.join(DATA_DIR, "validation.spm"),
        "--testpref",  os.path.join(DATA_DIR, "test.spm"),
        "--destdir", DESTDIR,
        "--workers", "8"
    ]
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  3️⃣ ======== Run preprocessing ==================================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    subprocess.run(cmd, check=True)
    print(f"✅ Preprocessing complete! Binarized EN→ZH data saved in {DESTDIR}")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 4. FAIRSEQ SETUP, ACTIVATION PATCHING | XXX --------------------------------------####################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ========  4.1 Device Configuration =========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  4.2 Custom Activation Registration & Patching ====================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# Ensure activation folder is in sys.path
if ACTIVATION_PATH not in sys.path:
    sys.path.append(ACTIVATION_PATH)

# Backup original get_activation_fn
orig_get_activation_fn = utils.get_activation_fn
# ────────────────────────────────────────────────────────────────────────────────────────────────
def custom_get_activation_fn(activation: str):
    activation = activation.lower()
    if activation == "tanhexp":
        from activation.TanhExp import TanhExp
        print("✅ Registered custom activation: TanhExp")
        return TanhExp()
    
    elif activation == "fftgate":
        from activation.FFTGate import FFTGate
        print("✅ Registered custom activation: FFTGate")
        return FFTGate()
    
    elif activation == "geglu":        
        from activation.GEGLU import GEGLU
        print("✅ Registered custom activation: GEGLU")
        return GEGLU()

    else:
        # ✅ fallback to original fairseq activations
        return orig_get_activation_fn(activation)
# ────────────────────────────────────────────────────────────────────────────────────────────────
# Patch fairseq utils
utils.get_activation_fn = custom_get_activation_fn

# ────────────────────────────────────────────────────────────────────────────────────────────────
print(f"⚡ Using activation function: {exp_args.act_name.lower()}")

# Test-call the patched activation to confirm it works
try:
    act_fn = utils.get_activation_fn(exp_args.act_name.lower())
    print(f"🔍 Activation function resolved: {act_fn}")
except Exception as e:
    print(f"❌ Failed to resolve activation '{exp_args.act_name.lower()}': {e}")
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Manually inject custom activation into config
cfg.model.activation_fn = exp_args.act_name.lower()
print(f"⚡ Overwrote cfg.model.activation_fn = {cfg.model.activation_fn}")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ========  4.3 Fairseq Task & Dataset Initialization ========================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Load dictionaries
src_dict = Dictionary.load(os.path.join(fairseq_args.data, f"dict.{fairseq_args.source_lang}.txt"))
tgt_dict = Dictionary.load(os.path.join(fairseq_args.data, f"dict.{fairseq_args.target_lang}.txt"))

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Setup task
task = TranslationTask.setup_task(cfg.task)
task.load_dataset("train")
task.load_dataset("test")
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ✅✅ Print translation direction (EN→ZH confirmation)
print(f"🌍 Translation direction: {fairseq_args.source_lang} → {fairseq_args.target_lang}")
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ✅✅ Inspect dataset lengths after loading
train_dataset = task.dataset("train")
test_dataset  = task.dataset("test")
# ────────────────────────────────────────────────────────────────────────────────────────────────

print(f"📏 Train set - Max source length: {train_dataset.src_sizes.max()}, Max target length: {train_dataset.tgt_sizes.max()}")
print(f"📏 Test set  - Max source length: {test_dataset.src_sizes.max()}, Max target length: {test_dataset.tgt_sizes.max()}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Inspect data you’d lose if you lower max_positions
def check_percentage(dataset, N):
    src_long = (dataset.src_sizes > N).sum()
    tgt_long = (dataset.tgt_sizes > N).sum()
    total = len(dataset)

    print(f"Dataset size: {total}")
    print(f"  > {N} tokens (source): {src_long} ({100*src_long/total:.2f}%)")
    print(f"  > {N} tokens (target): {tgt_long} ({100*tgt_long/total:.2f}%)")
    print()

# ✅✅ Check on train + test
check_percentage(train_dataset, exp_args.max_source_positions)
check_percentage(test_dataset, exp_args.max_source_positions)
# ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 5. MODEL INITIALIZATION | XXX ----------------------------------------------------####################
########################################################################################################################

# ================================================================================================
# 🏷️========================= MODEL INITIALIZATION===============================================
# ================================================================================================
# ================================================================================================
####------------------ 0️⃣ 1️⃣ 2️⃣ 3️⃣ 4️⃣ 5️⃣ 6️⃣ 7️⃣ 8️⃣ 9️⃣ ------------------------------------####



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  ======== 🔖🔖  5.1 Model Construction 🔖🔖====================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 


# ================================================================================================
# 1️⃣ 🧠 Build Fairseq Baseline Transformer
# ================================================================================================
# ================================================================================================
# 1️⃣ 🧠 Build Fairseq Baseline Transformer
# ================================================================================================

if exp_args.net == "Transformer":

    model = task.build_model(cfg.model)

    print(f"✅ Loaded Model: {model.__class__.__name__}")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ================================================================================================
# 2️⃣ 🧠 Build RTM_Former 📐🔑🏷️
# ================================================================================================
# ================================================================================================
# 2️⃣ 🧠 Build RTM_Former 📐🔑🏷️
# ================================================================================================
elif exp_args.net == "RTM_Former":


    rtm_layers = []

    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 1️⃣ 🧩🧠 === FFN Insertion === ♻️♻️ ========================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 1️⃣ 🧩🧠 === FFN Insertion === ♻️♻️ ========================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    if exp_args.rtm_insertion_type == "ffn":

        # ================================================================================================
        # 1️⃣.0️⃣ 🧠 Build RTM_Former  
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:ffn🧬🧬")
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:ffn🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print("\nMODEL INIT CHECK")
        print(model.decoder.layers[0].fc1.weight[0, :10])
        print()
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────────────────────────────────       
        # =============================================================================
        # 1️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder FFNs 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i >= len(model.decoder.layers) - exp_args.rtm_num_layers:

                original_activation_fn = layer.activation_fn

                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,
                    rtm_rotation=exp_args.rtm_rotation                 
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:

                    # 🔒 Configure RTM coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Activation Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ───────────────────────────────────────────────────────────────────────────────────────────────── 
                # ─────────────────────────────────────────────────────────────────────────────────────────────────           


                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                def wrapped_activation(
                    x,
                    act_fn=original_activation_fn,
                    rtm_module=rtm
                ):

                    # ==========================================================
                    # 1️⃣.2️⃣.1️⃣ Standard FFN activation
                    # ==========================================================
                    x = act_fn(x)

                    # ==========================================================
                    # 1️⃣.2️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 1️⃣.2️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm_module.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0 + exp_args.rtm_transition_amplification * transition_scale
                    # )

                    # x = x * (
                    #     1.0 + rtm_module.amplification * transition_scale
                    # )

                    global RTM_SCALE

                    x_base = x

                    x_mod = x * (
                        1.0 + rtm_module.amplification * transition_scale
                    )

                    x_mod = rtm_module(x_mod)
                    # ==========================================================
                    # 1️⃣.2️⃣.6️⃣ RTM refinement
                    # ==========================================================
                    # x = rtm_module(x)
                    x = x_base + RTM_SCALE * (x_mod - x_base)

                    return x
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               

                # =============================================================================
                # 📌📌 Replace decoder activation function
                # =============================================================================                 
                layer.activation_fn = wrapped_activation
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  


                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 FFN Insertion | RTM inserted into decoder layer {i}")                
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────




    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣ 🧩🧠 === Cross-Attention Insertion ♻️♻️ ================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣ 🧩🧠 === Cross-Attention Insertion ♻️♻️ ================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────

    elif exp_args.rtm_insertion_type == "cross_attn":
    
        # ================================================================================================
        # 2️⃣.0️⃣ 🧠 Build RTM_Former 🅰️🔼 
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:cross_attn🧬🧬")
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:cross_attn🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder Cross-Attention 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i >= len(model.decoder.layers) - exp_args.rtm_num_layers:

                # =============================================================================
                # 2️⃣.2️⃣.1️⃣📌📌 RTM module
                # =============================================================================
                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,
                    rtm_rotation=exp_args.rtm_rotation                    
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:

                    # 🔒 Configure RTM coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Attention Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────


                # =============================================================================
                # 2️⃣.2️⃣.2️⃣📌📌 Store original encoder-attention forward
                # =============================================================================
                original_encoder_attn_forward = layer.encoder_attn.forward

                # =============================================================================
                # 2️⃣.2️⃣.3️⃣📌📌 Wrapped Cross-Attention forward
                # =============================================================================
                def wrapped_encoder_attn_forward(
                    query,
                    key,
                    value,
                    **kwargs
                ):

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.1️⃣ Original Cross-Attention
                    # ==========================================================
                    x, attn = original_encoder_attn_forward(
                        query=query,
                        key=key,
                        value=value,
                        **kwargs
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0
                    #     +
                    #     exp_args.rtm_transition_amplification
                    #     *
                    #     transition_scale
                    # )

                    x = x * (
                        1.0
                        +
                        rtm.amplification
                        *
                        transition_scale
                    )

                    return x, attn
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────


                # =============================================================================
                # 📌📌 Replace encoder-attention forward
                # =============================================================================
                layer.encoder_attn.forward = wrapped_encoder_attn_forward
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────

                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 Cross-Attention Insertion | RTM inserted into decoder layer {i}") 

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────



    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 3️⃣ 🧩🧠 === FFN Depth-Weighted Insertion === ♻️♻️ =========================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 3️⃣ 🧩🧠 === FFN Depth-Weighted Insertion === ♻️♻️ =========================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    elif exp_args.rtm_insertion_type == "ffn_depth_weighted":

        # ================================================================================================
        # 1️⃣.0️⃣ 🧠 Build RTM_Former  
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: Depth-Weighted RTM_Former for insertion_typ:ffn_depth_weighted🧬🧬")
        print(f"♻️♻️ Loaded Model: Depth-Weighted RTM_Former for insertion_typ:ffn_depth_weighted🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # 🔒 IMPORTANT: reset RNG after importing RTM_Former, before the real training model is built (For Repeatable Initialization)
        set_seed_torch(seed1)
        set_seed_main(seed2)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print("\nMODEL INIT CHECK")
        print(model.decoder.layers[0].fc1.weight[0, :10])
        print()
        # ==========================================================
        # 🚦🔍 Select layers
        # ========================================================== 
        selected_layers = list(
            range(
                len(model.decoder.layers) - exp_args.rtm_num_layers,
                len(model.decoder.layers)
            )
        )
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────────────────────────────────       
        # =============================================================================
        # 1️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder FFNs 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i in selected_layers:

                original_activation_fn = layer.activation_fn

                layer_rank = selected_layers.index(i)


                # =====================================================
                # ⏭️⏭️ Forward direction Layer scale ⏭️⏭️
                # =====================================================               
                if exp_args.rtm_layer_scale_direction == "forward":
                    layer_scale = exp_args.rtm_layer_scale_min + (
                        1.0 - exp_args.rtm_layer_scale_min
                    ) * (
                        layer_rank / max(1, exp_args.rtm_num_layers - 1)
                    )
                # =====================================================
                # ⏪⏪ Reverse direction Layer scale ⏪⏪
                # =====================================================  
                elif exp_args.rtm_layer_scale_direction == "reverse":
                    layer_scale = 1.0 - (
                        1.0 - exp_args.rtm_layer_scale_min
                    ) * (
                        layer_rank / max(1, exp_args.rtm_num_layers - 1)
                    )
                # =====================================================
                # ⏪🔒 Reverse direction + Fixed Final Layer Scale ⏪🔒
                # =====================================================
                elif exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                    if layer_rank == exp_args.rtm_num_layers - 1:
                        layer_scale = exp_args.rtm_final_layer_scale
                    else:
                        layer_scale = 1.0 - (
                            1.0 - exp_args.rtm_layer_scale_min
                        ) * (
                            layer_rank / max(1, exp_args.rtm_num_layers - 2)
                        ) 
                # ────────────────────────────────────────────────────────────────────────────────────────
                else:
                    raise ValueError(f"Unknown RTM layer scale direction: {exp_args.rtm_layer_scale_direction}")
                # ────────────────────────────────────────────────────────────────────────────────────────

                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,  
                    gate_type=exp_args.rtm_gate_type,
                    rtm_rotation=exp_args.rtm_rotation                                     
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility + Parameter Tracking)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:
                    
                    # 🔒 Configure RTM learnable coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Activation Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ─────────────────────────────────────────────────────────────────────────────────────────────────                
                # ─────────────────────────────────────────────────────────────────────────────────────────────────

                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                def wrapped_activation(
                    x,
                    act_fn=original_activation_fn,
                    rtm_module=rtm,
                    layer_scale=layer_scale
                ):

                    # ==========================================================
                    # 1️⃣.2️⃣.1️⃣ Standard FFN activation
                    # ==========================================================
                    x = act_fn(x)

                    # ==========================================================
                    # 1️⃣.2️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 1️⃣.2️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm_module.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0 + exp_args.rtm_transition_amplification * transition_scale
                    # )

                    # x = x * (
                    #     1.0 + rtm_module.amplification * transition_scale
                    # )

                    global RTM_SCALE

                    x_base = x

                    x_mod = x * (
                        1.0 + rtm_module.amplification * transition_scale
                    )

                    x_mod = rtm_module(x_mod)
                    # ==========================================================
                    # 1️⃣.2️⃣.6️⃣ RTM refinement
                    # ==========================================================

                    # ----------------------------------------------------------
                    # 🏔️ Depth-Aware RTM Scaling (Layer-Adaptive Modulation)
                    # ---------------------------------------------------------- 
                    if exp_args.rtm_depth_weight:

                        x = x_base + RTM_SCALE * layer_scale * (x_mod - x_base)

                    # ----------------------------------------------------------
                    # ⚖️ Uniform RTM Scaling (Same Modulation Across Layers)
                    # ----------------------------------------------------------
                    else:

                        x = x_base + RTM_SCALE * (x_mod - x_base)
                    # ─────────────────────────────────────────────────────────────────────────────────────────────────

                    return x
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               

                # =============================================================================
                # 📌📌 Replace decoder activation function
                # =============================================================================                 
                layer.activation_fn = wrapped_activation
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  


                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 FFN Insertion | RTM inserted into decoder layer {i} | layer_scale={layer_scale:.3f}")              
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: Depth-Weighted RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    else:
        raise ValueError(f"Unknown RTM insertion type: {exp_args.rtm_insertion_type}")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print(f"📣📣 RTM_Former for insertion_typ={exp_args.rtm_insertion_type} | Loaded Model={exp_args.net }♻️♻️")
    print(f"📣📣 RTM_Former for insertion_typ={exp_args.rtm_insertion_type} | Loaded Model={exp_args.net }♻️♻️")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────



    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  ======== 🔖  5.1.1 Model Configuration Hyperparameter Summary ==================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 
    if exp_args.net == "RTM_Former":

        rtm_log = (
            f"📣 RTM Configuration             : "
            f"alpha={exp_args.rtm_alpha} | "
            f"amp={exp_args.rtm_transition_amplification} | "
            f"layers={exp_args.rtm_num_layers} | "
            f"insertion={exp_args.rtm_insertion_type} | "
            f"scale_init={exp_args.rtm_scale} | "
            f"start_epoch={exp_args.rtm_start_epoch} | "
            f"full_epoch={exp_args.rtm_full_epoch}"
        )

        # Depth-weighted RTM only
        if exp_args.rtm_insertion_type == "ffn_depth_weighted":

            rtm_log += (
                f" | layer_scale_min={exp_args.rtm_layer_scale_min}"
                f" | layer_direction={exp_args.rtm_layer_scale_direction}"
                f" | gate={exp_args.rtm_gate_type}"
            )

            # Only exists for reverse_final_fixed
            if exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                rtm_log += (
                    f" | final_layer_scale={exp_args.rtm_final_layer_scale}"
                )

        print(rtm_log)


    else:
        print(
            f"📣 Baseline Configuration        : "
            f"model={exp_args.net}"
        )
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ======== 🔖🔖  5.2  Insert model to device 🔖🔖================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

model = model.to(DEVICE)
 # ─────────────────────────────────────────────────────────────────────────────────────────────────
print(f"🚀🚀 Model moved to device: {DEVICE} ♻️♻️")
print(f"🚀🚀 Model moved to device: {DEVICE} ♻️♻️")
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ======== 🔖🔖 5.3  Model Validation & Sanity Checks 🔖🔖=======================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 🔍 Print custom activations to confirm they are used
for name, module in model.named_modules():
    if isinstance(module, nn.Module) and module.__class__.__name__ in ["TanhExp", "FFTGate"]:
        print(f"🔎 Found custom activation in {name}: {module}")


# ✅ Sanity check Device
if DEVICE.type == "cuda":
    print(f"🚀 Model moved to CUDA: {torch.cuda.get_device_name(DEVICE)}")
else:
    print("⚠️ Using CPU (CUDA not available)")
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 6. PATH DEFINATION | XXX --------------------------------------------------------#####################
########################################################################################################################


# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 1️⃣ 📌📌 ======== ENSURE DIRECTORY EXIST ========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# # ─────────────────────────────────────────────────────────────────────────────────────────────────
# # 🟡 === Checkpoint directories ===
# if not os.path.exists('checkpoint'):
#     os.makedirs('checkpoint')
# # ─────────────────────────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Results directories ===
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}')

if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === LR Log directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Training Log directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Model Summary directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')
# ────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣ 📌📌 ======== DEFINE DIRECTORY ==============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.1️⃣🔖 ==== Checkpoint Directory Selection | Normal Run vs Crash Run | save every epoch ===== 
# ────────────────────────────────────────────────────────────────────────────────────────────────

# 🅰️💾 Normal run: save checkpoints in standard folder
if not exp_args.crash_run:
    checkpoint_dir = f'./checkpoints/{exp_args.dataset_name}/{exp_args.net}/'

    # 📁 Create checkpoint folder if it does not exist
    os.makedirs(checkpoint_dir, exist_ok=True)

    print(f"🅰️💾 Normal Run Enabled | Saving checkpoints to: {checkpoint_dir}")
#---------------------------------------------------------------------------------------------------
# 🅱️♻️ Crash run: save checkpoints in recovery folder
else:
    checkpoint_dir = f'./checkpoints_crash_run/{exp_args.dataset_name}/{exp_args.net}/'

    # 📁 Create checkpoint folder if it does not exist    
    os.makedirs(checkpoint_dir, exist_ok=True)

    print(f"🅱️♻️ Crash Run Enabled | Saving recovery checkpoints to: {checkpoint_dir}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.2️⃣🔖 === Main Test & Train Results === 
train_results_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/Train_{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
test_results_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/Test_{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.3️⃣🔖  === LR, Training & Summary logs  === 
LR_save_paths = {
    "LR_history": f"./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_LR_log.txt"
}
save_paths = {
    "log_history": f"./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_training_logs.txt"
}
# ────────────────────────────────────────────────────────────────────────────────────────────────

# 2️⃣.4️⃣🔖 ===  Model summary path === 
model_summary_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_model_summary.txt'
model_summary_full_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_model_summary_full.txt'
# ────────────────────────────────────────────────────────────────────────────────────────────────

# # ✅🔖 === Path template for saving checkpoints at every epoch | Part2 (if scratch occured) ===
# checkpoint_dir = f'./checkpoints/{exp_args.dataset_name}/tanhexp_AfterCrash_41_49epochs/'
# os.makedirs(checkpoint_dir, exist_ok=True)
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 7. TRAINING INITIALIZATION & OPTIMIZATION SETUP | XXX ---------------------------#####################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ========  7.1 AMP GradScaler ===============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Initialize AMP GradScaler once globall
scaler = torch.cuda.amp.GradScaler()
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  Training Criterion ===============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

criterion = LabelSmoothingCrossEntropy(smoothing=exp_args.smoothing)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ========   Optimizer Configuration =========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ♻️ Adam optimizer (β1=0.9, β2=0.98, eps=1e-9) ♻️
# ─────────────────────────────────────────────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=exp_args.lr,
    betas=(0.9, 0.98),
    eps=exp_args.eps,
    weight_decay=exp_args.weight_decay,
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 4️⃣ ========  Learning Rate Scheduler ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ✅  Transformer LR schedule (warmup + inverse sqrt decay)
# --------------------------

warmup_updates = exp_args.warmup_updates   # configurable warmup steps
peak_lr = exp_args.lr

# ────────────────────────────────────────────────────────────────────────────────────────────────
def lr_lambda(step):
    step = max(1, step)
    if step == 1:
        print(f"🚀 Warmup started at step {step} (target peak LR = {peak_lr}).")
    if step == warmup_updates:
        actual_lr = optimizer.param_groups[0]["lr"]
        print(f"✅ Warmup finished at step {step}. Peak LR reached: {actual_lr:.8f}")
    if step == warmup_updates + 1:
        print(f"📉 Inverse square root decay started at step {step}.")
    
    if step <= warmup_updates:
        # Linear warmup
        scale = step / warmup_updates
    else:
        # Inverse square root decay
        scale = (warmup_updates ** 0.5) / (step ** 0.5)
    return scale

# ────────────────────────────────────────────────────────────────────────────────────────────────

lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 5️⃣ ========  Global Tracking Variables ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ global initialization
best_bleu = 0.0
total_test_duration = 0.0
RTM_SCALE_LOG = ""
# ─────────────────────────────────────────────────────────────────────────────────────────────────

Python: 3.10.16 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:19:12) [MSC v.1929 64 bit (AMD64)]
Torch: 2.2.0+cu121 | CUDA available: True
Fairseq: 0.12.2
OmegaConf: 2.0.6
Hydra-Core: 1.0.7
✅ Current working directory: C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH
✅ sys.path updated:
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\python310.zip
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\DLLs
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env
   📂 
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\win32
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\win32\lib
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\Pythonwin
   📂 C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH
   📂 C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH\models
   📂 C:\

c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\modules\module.py:1150: UserWarning: expandable_segments not supported on this platform (Triggered internally at ..\c10/cuda/CUDAAllocatorConfig.h:30.)
  return t.to(device, dtype if t.is_floating_point() or t.is_complex() else None, non_blocking)


In [ ]:
########################################################################################################################
####-------| NOTE 11. TEST ALL CHECKPOINTS (BLEU per epoch)  | XXX -----------------------------########################
########################################################################################################################
########################################################################################################################
####-------| NOTE 11. TEST ALL CHECKPOINTS (BLEU per epoch)  | XXX -----------------------------########################
########################################################################################################################


# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ======== Paths for checkpoint BLEU sweeps ===================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Paths for checkpoint BLEU sweeps
test_checkpoint_results_path = (
    f'./Results/{exp_args.dataset_name}/{exp_args.net}/'
    f'TestCheckpoints_{exp_args.net}_'
    f'{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
)
# ────────────────────────────────────────────────────────────────────────────────────────────────


# # ✅ Paths for checkpoint BLEU sweeps | Part2 (if Cratch occured)
# test_checkpoint_results_path = (
#     f'./Results/{exp_args.dataset_name}/{exp_args.act_name}/'
#     f'TestCheckpoints_{exp_args.act_name}_{exp_args.net}_'
#     f'{exp_args.dataset_name}_{exp_args.optimizer1}_tanhexp_AfterCrash_41_49epochs.txt'
# )
# ────────────────────────────────────────────────────────────────────────────────────────────────


# # ─────────────────────────────────────────────────────────────────────────────────────────────────
# #  2️⃣ ========  BLEU Detokenization (🔖 EN/FR/DE) ================================================
# # ───────────────────────────────────────────────────────────────────────────────────────────────── 
# # ────────────────────────────────────────────────────────────────────────────────────────────────
# def detokenize_for_bleu(s: str) -> str:
#     s = s.replace("@@ ", "").replace("@@", "")  # BPE
#     s = s.replace("▁", " ")                     # SentencePiece
#     s = " ".join(s.split())                     # normalize spaces
#     s = re.sub(r"\s+([?.!,])", r"\1", s)        # punctuation
#     return s.strip()
# # ────────────────────────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  BLEU Detokenization (🔖 Chinese) =================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
def detokenize_for_bleu(s: str) -> str:
    # For Chinese, don't remove SentencePiece boundaries or spaces | Only remove BPE continuation markers
    # sacreBLEU's "zh" tokenizer expects raw characters.
    return s.replace("@@ ", "").replace("@@", "").strip()
# ────────────────────────────────────────────────────────────────────────────────────────────────




# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ======== Test_Checkpoint Function ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ test_checkpoint function
def test_checkpoint_epochs(task, model, criterion, checkpoint_dir, test_checkpoint_results_path):
    """
    Evaluate BLEU (beam=5) for all checkpoints in checkpoint_dir.
    Save results to test_checkpoint_results_path in the same format as normal test logs.
    """
# ────────────────────────────────────────────────────────────────────────────────────────────────
    global RTM_SCALE, RTM_SCALE_LOG
    # ────────────────────────────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Collect checkpoints sorted by epoch number in filename
    def extract_epoch_num(filename):
        match = re.search(r"epoch_(\d+)", filename)
        return int(match.group(1)) if match else float("inf")  # inf if no epoch number

    ckpts = sorted(
        glob.glob(os.path.join(checkpoint_dir, "*.pt")),
        key=lambda x: extract_epoch_num(os.path.basename(x))
    )

    if not ckpts:
        print(f"⚠️ No checkpoints found in {checkpoint_dir}")
        return

    # # ────────────────────────────────────────────────────────────────────────────────────────────────
    # # ✅ Reset output file
    # os.makedirs(os.path.dirname(test_checkpoint_results_path), exist_ok=True)
    # with open(test_checkpoint_results_path, "w", encoding="utf-8") as f:
    #     f.write("")
    # # ────────────────────────────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 📦 Reset output file + write model status
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    #1️⃣ Create output folder if it does not exist
    os.makedirs(os.path.dirname(test_checkpoint_results_path), exist_ok=True)

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Normal run: reset file
    # ✅ Crash run : append to existing file
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣♻️ Normal run: reset file
    if not exp_args.crash_run:

        with open(test_checkpoint_results_path, "w", encoding="utf-8") as f:
            f.write("")

    # 3️⃣⏭️ Crash run : append to existing file
    else:

        with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
            f.write("\n♻️ Resuming checkpoint testing after crash\n\n")
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 4️⃣✍️  Write test result header (model information)
    with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
        f.write("")

        # ---------------------------------------------------------------------------
        # 4️⃣.1️⃣🏷️ Testing Model Status
        # ---------------------------------------------------------------------------
        if exp_args.net == "RTM_Former":

            f.write(
                f"Model: RTM_Former\n\n"
            )

        else:

            f.write(
                f"Model: Baseline Fairseq Transformer\n\n"
            )        
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ────────────────────────────────────────────────────────────────────────────────────────────────



    # ────────────────────────────────────────────────────────────────────────────────────────────────
    best_bleu = 0.0
    total_test_duration = 0.0
    # ────────────────────────────────────────────────────────────────────────────────────────────────


    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    for ckpt in ckpts:

        # # ────────────────────────────────────────────────────────────────────────────────
        # # ✅ 📌📌 Load checkpoint (OLD) 🅰️🔼 
        # # ────────────────────────────────────────────────────────────────────────────────        
        # print(f"🔄 Loading checkpoint: {ckpt}")
        # state = torch.load(ckpt, map_location=DEVICE)
        # model.load_state_dict(state["model_state"], strict=False)
        # model.to(DEVICE)
        # model.eval()
        # --------------------------------------------------------------------------------------------------
        # ────────────────────────────────────────────────────────────────────────────────
        # ✅📌📌 Load checkpoint (NEW) 🅱️🔼 Restore RTM schedule for checkpoint testing using same strategy as main loop
        # ────────────────────────────────────────────────────────────────────────────────
        if exp_args.net == "Transformer":
            print(f"🔄 Loading checkpoint Baseline Transformer : {ckpt}")
            state = torch.load(ckpt, map_location=DEVICE)
            model.load_state_dict(state["model_state"], strict=False)
            model.to(DEVICE)
            model.eval()
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        elif exp_args.net == "RTM_Former":
            print(f"🔄 Loading checkpoint RTM_Former : {ckpt}")
            state = torch.load(ckpt, map_location=DEVICE)
            model.load_state_dict(state["model_state"], strict=False)            
            

            RTM_START_EPOCH = exp_args.rtm_start_epoch
            RTM_FULL_EPOCH = exp_args.rtm_full_epoch
            ckpt_epoch = state["epoch"]

            if ckpt_epoch < RTM_START_EPOCH:
                RTM_SCALE = exp_args.rtm_scale
            else:
                progress = min(
                    1.0,
                    (ckpt_epoch - RTM_START_EPOCH) / (RTM_FULL_EPOCH - RTM_START_EPOCH)
                )

                RTM_SCALE = 0.5 * (1.0 - math.cos(math.pi * progress))

            print(f"📐 Checkpoint RTM_SCALE for epoch {ckpt_epoch}: {RTM_SCALE:.3f}")

            if exp_args.rtm_insertion_type == "ffn_depth_weighted":

                if exp_args.rtm_layer_scale_direction == "forward":
                    depth_scales = [
                        exp_args.rtm_layer_scale_min
                        + (1.0 - exp_args.rtm_layer_scale_min)
                        * (j / max(1, exp_args.rtm_num_layers - 1))
                        for j in range(exp_args.rtm_num_layers)
                    ]

                elif exp_args.rtm_layer_scale_direction == "reverse":
                    depth_scales = [
                        1.0
                        - (1.0 - exp_args.rtm_layer_scale_min)
                        * (j / max(1, exp_args.rtm_num_layers - 1))
                        for j in range(exp_args.rtm_num_layers)
                    ]

                elif exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                    depth_scales = []

                    for j in range(exp_args.rtm_num_layers):
                        if j == exp_args.rtm_num_layers - 1:
                            scale_j = exp_args.rtm_final_layer_scale
                        else:
                            scale_j = (
                                1.0
                                - (1.0 - exp_args.rtm_layer_scale_min)
                                * (j / max(1, exp_args.rtm_num_layers - 2))
                            )

                        depth_scales.append(scale_j)

                else:
                    raise ValueError(
                        f"Unknown RTM layer scale direction: {exp_args.rtm_layer_scale_direction}"
                    )

                final_scales = [RTM_SCALE * s for s in depth_scales]

                RTM_SCALE_LOG = f" | RTM_SCALE={RTM_SCALE:.3f}"
                RTM_SCALE_LOG += (
                    f" | RTM_layer_scales({exp_args.rtm_layer_scale_direction})="
                    f"{[round(s, 3) for s in final_scales]}"
                )

                print(
                    f"📐 Checkpoint final RTM layer scales "
                    f"({exp_args.rtm_layer_scale_direction}): "
                    f"{[round(s, 3) for s in final_scales]}"
                )
            model.to(DEVICE)
            model.eval()               
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        else:
            raise ValueError(f"Unknown model type: {exp_args.net}")
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ────────────────────────────────────────────────────────────────────────────────────────────────


        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Build test iterator
        test_iter = task.get_batch_iterator(
            dataset=task.dataset("test"),
            max_tokens=fairseq_args.max_tokens,   # cut in half option: fairseq_args.max_tokens // 2
            max_positions=(exp_args.max_source_positions, exp_args.max_target_positions),
            seed=seed1,
            num_shards=exp_args.num_shards,
            shard_id=exp_args.shard_id,
            num_workers=exp_args.num_workers,
            ignore_invalid_inputs=exp_args.ignore_invalid_inputs,
        ).next_epoch_itr(shuffle=False)
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ References
        true_references = []
        for sample in test_iter:
            targets = sample["target"]
            for t in targets:
                t_no_pad = t[t != tgt_dict.pad()]
                ref_str = tgt_dict.string(t_no_pad, bpe_symbol="@@")
                true_references.append(detokenize_for_bleu(ref_str))
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Rebuild iterator for decoding
        test_iter = task.get_batch_iterator(
            dataset=task.dataset("test"),
            max_tokens=fairseq_args.max_tokens,    # cut in half option: fairseq_args.max_tokens // 2
            max_positions=(exp_args.max_source_positions, exp_args.max_target_positions),
            seed=seed1,
            num_shards=exp_args.num_shards,
            shard_id=exp_args.shard_id,
            num_workers=exp_args.num_workers,
            ignore_invalid_inputs=exp_args.ignore_invalid_inputs,
        ).next_epoch_itr(shuffle=False)
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Build generator with beam=5
        gen_cfg = deepcopy(cfg.generation)
        gen_cfg.beam = exp_args.beam         # 🎀🔖 5
        gen_cfg.lenpen = exp_args.lenpen     # 🎀🔖 1.0 
        generator = task.build_generator([model], gen_cfg)
        # ────────────────────────────────────────────────────────────────────────────────────────────────


 

        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Run evaluation
        sys_outputs, test_loss = [], 0.0
        start_time = time.time()

        with torch.no_grad():
            with tqdm(enumerate(test_iter), total=len(test_iter), desc=f"Testing {os.path.basename(ckpt)}") as progress:
                for i, sample in progress:   # 👈 keep i!
                    sample = utils.move_to_cuda(sample) if torch.cuda.is_available() else sample
                    src_tokens = sample["net_input"]["src_tokens"]
                    prev_output_tokens = sample["net_input"]["prev_output_tokens"]
                    target = sample["target"]
                    src_lengths = (src_tokens != src_dict.pad()).sum(dim=1)

                    with torch.cuda.amp.autocast():
                        logits, _ = model(src_tokens, src_lengths, prev_output_tokens)
                        pad = tgt_dict.pad()
                        logits2d = logits.view(-1, logits.size(-1))
                        target1d = target.view(-1)
                        mask = target1d != pad
                        loss = criterion(logits2d[mask], target1d[mask]) if mask.any() else logits2d.sum() * 0.0
                    test_loss += loss.item()

                    hypos = task.inference_step(generator, [model], sample)
                    for j, hypos_j in enumerate(hypos):
                        hypo_tokens = [tok for tok in hypos_j[0]["tokens"].tolist() if tok != tgt_dict.pad()]
                        sys_str = tgt_dict.string(hypo_tokens, bpe_symbol="@@")
                        sys_outputs.append(detokenize_for_bleu(sys_str))

                    # ✅ Update progress bar postfix with running average
                    progress.set_postfix(loss=round(test_loss / (i + 1), 3))
       # ────────────────────────────────────────────────────────────────────────────────────────────────




       # ────────────────────────────────────────────────────────────────────────────────────────────────
        avg_loss = test_loss / max(1, len(test_iter))
        # bleu = sacrebleu.corpus_bleu(sys_outputs, [true_references])                 #🔖 EN/FR/DE
        bleu = sacrebleu.corpus_bleu(sys_outputs, [true_references], tokenize="zh")    #🔖 Chinese

        duration = time.time() - start_time
        total_test_duration += duration
       # ────────────────────────────────────────────────────────────────────────────────────────────────


       # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Write results
        with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
            f.write(f"Checkpoint {os.path.basename(ckpt)} | Test Loss: {avg_loss:.3f} | sacreBLEU: {bleu.score:.2f} | Beam=5\n")

        print(f"✅ {os.path.basename(ckpt)} | Loss {avg_loss:.3f} | sacreBLEU {bleu.score:.2f}")


        # 🧹 add visual spacing between runs
        tqdm.write("")  
       # ────────────────────────────────────────────────────────────────────────────────────────────────

       # ────────────────────────────────────────────────────────────────────────────────────────────────
        if bleu.score > best_bleu:
            best_bleu = bleu.score

    with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
        f.write(f"\n🏆 Best BLEU Score: {best_bleu:.2f}\n")
        total_mins, total_secs = divmod(total_test_duration, 60)
        f.write(f"\n🕒 Total Test Time over all checkpoints: {int(total_mins)} min {total_secs:.2f} sec\n")

    print(f"🏆 Best BLEU over checkpoints: {best_bleu:.2f}")
    # ────────────────────────────────────────────────────────────────────────────────────────────────



# ────────────────────────────────────────────────────────────────────────────────────────────────
# # ✅ Free unused GPU memory BEFORE training starts
# torch.cuda.empty_cache()
# torch.cuda.reset_peak_memory_stats()

# 4️⃣ Call the function
test_checkpoint_epochs(task, model, criterion, checkpoint_dir, test_checkpoint_results_path)
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────────────────────────

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_25_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 25: 1.000


Testing epoch_25_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt:   0%|          | 0/82 [00:00<?, ?it/s]c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)
c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\functional.py:5109: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
Testing epoch_25_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:50<00:00,  1.63it/s, loss=4.44]


✅ epoch_25_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.439 | sacreBLEU 18.89

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_26_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 26: 1.000


Testing epoch_26_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:38<00:00,  2.14it/s, loss=4.39]


✅ epoch_26_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.386 | sacreBLEU 19.23

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_27_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 27: 1.000


...

Testing epoch_35_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:45<00:00,  1.81it/s, loss=4.3] 


✅ epoch_35_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.300 | sacreBLEU 19.84

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_36_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 36: 1.000


Testing epoch_36_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:39<00:00,  2.08it/s, loss=4.3] 


✅ epoch_36_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.300 | sacreBLEU 19.40

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_37_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 37: 1.000


: 

Restarted pytorch_env (Python 3.10.16)

In [ ]:
#####-------------------------------- NOTE MAIN IWSLT17 EN-ZH NOTE --------------------------------------------------#####
##########################################################################################################################
######################|--------------------------------------------------------------|####################################
###################################🔗 MAIN | TRAIN | TEST LOOP 🔗########################################################
######################|--------------------------------------------------------------|####################################
##########################################################################################################################
#####-------------------------------- NOTE MAIN IWSLT17 EN-ZH NOTE --------------------------------------------------#####



# 📄  main_IWSLT17En_Zh.py
########################################################################################################################
####-------| NOTE 1.1 IMPORTS LIBRARIES  | XXX -------------------------------------------------------##################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 📜 === Enable flexible CUDA memory allocation to reduce fragmentation ===
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ======================================================================================================
# 📜 === Core Libraries ===
# ======================================================================================================
import sys
import numpy as np
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import time
import random
import math
import re
import glob
from copy import deepcopy
import sacrebleu  # ✅ for BLEU evaluation
from timm.loss import LabelSmoothingCrossEntropy
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ======================================================================================================
# 📜 === Fairseq & Supporting Imports ===
# ======================================================================================================
import importlib.metadata as importlib_metadata
import fairseq  # type: ignore
from fairseq.data import Dictionary, data_utils, iterators          # type: ignore
from fairseq.tasks.translation import TranslationTask               # type: ignore
from fairseq import options                                         # type: ignore
from fairseq.dataclass.utils import convert_namespace_to_omegaconf  # type: ignore                                             
from fairseq import utils                                           # type: ignore 
import hydra                                                        # type: ignore  
# ────────────────────────────────────────────────────────────────────────────────────────────────
from ptflops import get_model_complexity_info
from calflops import calculate_flops
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ======================================================================================================
# ♻️ === Print environment summary for sanity check ===
# ======================================================================================================
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Fairseq:", fairseq.__version__)
print("OmegaConf:", importlib_metadata.version("omegaconf"))
print("Hydra-Core:", importlib_metadata.version("hydra-core"))
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 1.2. DEFINE PROJECT PATH | XXX ---------------------------------------------------####################
########################################################################################################################

# ✅ Define working directory
Project_PATH = r"C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH"
if os.getcwd() != Project_PATH:
    os.chdir(Project_PATH)
print(f"✅ Current working directory: {os.getcwd()}")


# ✅ Define absolute paths
PROJECT_PATH = Project_PATH
MODELS_PATH = os.path.join(Project_PATH, "models")
ACTIVATION_PATH = os.path.join(Project_PATH, "activation")


# ✅ Ensure necessary paths are in sys.path
for path in [PROJECT_PATH, MODELS_PATH, ACTIVATION_PATH]:
    if path not in sys.path:
        sys.path.append(path)

# ✅ Print updated sys.path for debugging
print("✅ sys.path updated:")
for path in sys.path:
    print("   📂", path)
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 1.3. OTHER IMPORTS | XXX ---------------------------------------------------------####################
########################################################################################################################

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 📜 ============  Imput parser safe for afno because of --fno-bias, etc  =======================
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 1️⃣✅ Import parser from parser_IWSLT17En_Zh.py
from parser_IWSLT17En_Zh import get_parser

# ✅ Create parser and parse arguments
parser = get_parser()
# args, unknown = parser.parse_known_args()

# ✅ IMPORTANT: Do NOT read Jupyter / VSCode kernel arguments
# This prevents the "--f" ambiguity issue
exp_args = parser.parse_args(args=[])

num_aug_splits = exp_args.aug_splits

print(f"✅ Parser imported successfully | num_aug_splits = {num_aug_splits}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣✅ Import Fairseq parser/config
from fairseq_config_IWSLT17En_Zh import get_fairseq_parser
# ✅ Build Fairseq configuration
fairseq_args, cfg = get_fairseq_parser(exp_args)


num_aug_splits = exp_args.aug_splits

print(f"✅ Parser imported successfully | num_aug_splits = {num_aug_splits}")

print(f"✅ Encoder Embed Dim: {fairseq_args.encoder_embed_dim}")
print(f"✅ Decoder Layers: {fairseq_args.decoder_layers}")
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 3️⃣✅ Import model summary utility
from models.model_summary.model_summary import save_model_summary
from models.model_summary.model_summary_full import save_model_summary_full

print("✅ model_summary.py loaded successfully")
print("✅ model_summary_full.py loaded successfully")
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 2. SEEDING FOR REPRODUCIBILITY | XXX ---------------------------------------------####################
########################################################################################################################

# ────────────────────────────────────────────────────────────────────────────────────────────────
def set_seed_torch(seed):
    torch.manual_seed(seed)
# ────────────────────────────────────────────────────────────────────────────────────────────────
def set_seed_main(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True      # ✅ Default: "False" for Faster, non-deterministic kernels | "True" to Ensure deterministic behavior for CuDNN (Slower)
    torch.backends.cudnn.benchmark = False         # ✅ Default: "True" for Autotune kernels for performance   | "False" Disable CuDNN's autotuning for reproducibility (Slower)

    torch.backends.cuda.matmul.allow_tf32 = False  # ✅ Disable TF32 (strict reproducibility)
    torch.backends.cudnn.allow_tf32 = False        # ✅ Disable TF32 (strict reproducibility)

    # torch.use_deterministic_algorithms(True, warn_only=True)

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ ============= Define Seed =============
seed1, seed2 = exp_args.seed1, exp_args.seed2
set_seed_torch(seed1)  
set_seed_main(seed2)  
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 3.1. PROCESSING DATASET 1 | XXX ---- TOKENIZATION & DATASET PREPARATION |---------####################
########################################################################################################################
########################################################################################################################
####-------| NOTE 3.1. PROCESSING DATASET 1 | XXX ------------| ENGLISH→CHINESE |-------------------####################
########################################################################################################################

"""
Stage 1:
Train SentencePiece tokenizer, encode the raw English→Chinese corpus,
and create aligned train/test text splits for neural machine translation.
"""

# ─────────────────────────────────────────────────────────────────────────────────────────────────
print(
    f"🔎🔎 run_dataset_tokenization: {exp_args.run_dataset_tokenization} "
    f"{'(RUNNING)✔️✔️' if exp_args.run_dataset_tokenization else '(SKIPPED)❌❌'}"
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if exp_args.run_dataset_tokenization:
    from datasets import load_dataset    
    import sentencepiece as spm

    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Path for EN→ZH dataset
    DATA_PATH = r"C:\Users\emeka\Research\ModelCUDA\Big_Data_Journal\Comparison\Code\Paper\github3\IWSLT\Transformer_Baselines_EN_ZH\Dataset"
    os.makedirs(DATA_PATH, exist_ok=True)
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  1️⃣ ======== Download and export official IWSLT17 dataset =======================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Download to your folder (as the Hugging Face cache)
    en_zh = load_dataset(
        "IWSLT/iwslt2017", "iwslt2017-en-zh",
        cache_dir=DATA_PATH,
        trust_remote_code=True
    )
    zh_en = load_dataset(
        "IWSLT/iwslt2017", "iwslt2017-zh-en",
        cache_dir=DATA_PATH,
        trust_remote_code=True
    )  # optional
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  2️⃣ ======== Dataset verification ===============================================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ sanity check (should show ~231k train)
    print(f"EN→ZH dataset: {en_zh}")
    print(f"ZH→EN dataset: {zh_en}")

    print("EN→ZH splits:", list(en_zh.keys()), {k: len(en_zh[k]) for k in en_zh.keys()})
    print("ZH→EN splits:", list(zh_en.keys()), {k: len(zh_en[k]) for k in zh_en.keys()})

    # optional assertions
    assert set(en_zh.keys()) == {"train", "validation", "test"}
    assert set(zh_en.keys()) == {"train", "validation", "test"}
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  3️⃣ ======== Export official train/validation/test splits ======================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Export official IWSLT17 splits to plain text (EN/ZH) under raw_zh_en_dataset_path
    def dump_split(ds_dict, split, src, tgt, out_dir):
        src_fp = os.path.join(out_dir, f"{split}.{src}")
        tgt_fp = os.path.join(out_dir, f"{split}.{tgt}")
        with open(src_fp, "w", encoding="utf-8") as fs, open(tgt_fp, "w", encoding="utf-8") as ft:
            for row in ds_dict[split]:
                tr = row["translation"]
                fs.write((tr[src] or "").strip() + "\n")
                ft.write((tr[tgt] or "").strip() + "\n")
        print(f"Wrote {split}: {src_fp} | {tgt_fp}")

    for split in ["train", "validation", "test"]:
        dump_split(en_zh, split, "en", "zh", DATA_PATH)
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  4️⃣ ========  Train SentencePiece model =========================================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────     

    print("🧠 Training SentencePiece model for EN→ZH...")

    # Use TRAIN split for SPM training
    SRC_EN = os.path.join(DATA_PATH, "train.en")
    TGT_ZH = os.path.join(DATA_PATH, "train.zh")

    spm_model_prefix = os.path.join(DATA_PATH, "spm_enzh")

    spm.SentencePieceTrainer.Train(
        input=f"{SRC_EN},{TGT_ZH}",
        model_prefix=spm_model_prefix,
        vocab_size=10000,
        character_coverage=0.9995,   # ✅ --character_coverage for rich charcater langauage: Japanese or Chinese = 0.9995 | other languages with small character set: 1.0
        model_type="bpe"
    )
    print(f"✅ SentencePiece model saved to: {spm_model_prefix}.model / .vocab")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  5️⃣ ========  Encode files ======================================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    print("⚙️ Encoding dataset...")

    # ✅ Load the trained SentencePiece model
    sp = spm.SentencePieceProcessor(model_file=f"{spm_model_prefix}.model")

    def encode_file(in_file, out_file):
        with open(in_file, encoding="utf-8") as fin, open(out_file, "w", encoding="utf-8") as fout:
            for line in fin:
                fout.write(" ".join(sp.encode(line.strip(), out_type=str)) + "\n")

    # ✅ Encode all official splits (no custom re-split)
    for split in ["train", "validation", "test"]:
        encode_file(os.path.join(DATA_PATH, f"{split}.en"), os.path.join(DATA_PATH, f"{split}.spm.en"))
        encode_file(os.path.join(DATA_PATH, f"{split}.zh"), os.path.join(DATA_PATH, f"{split}.spm.zh"))
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  6️⃣ ======== Use official train/validation/test splits =========================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print(f"✅ Using official IWSLT17 train/validation/test splits.")

    print(f"✅ Encoded EN→ZH data saved under: {DATA_PATH}")
    print(f"✅ Next: run fairseq-preprocess to binarize (on *.spm.en/*.spm.zh).")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 3.2. PROCESSING DATASET 2 | XXX ----- FAIRSEQ PREPROCESSING & BINARIZATION |------####################
########################################################################################################################
########################################################################################################################
####-------| NOTE 3.2. PROCESSING DATASET 2 | XXX ------------| ENGLISH→CHINESE |--------------------####################
########################################################################################################################

"""
Stage 2:
Preprocess the tokenized English→Chinese dataset using fairseq-preprocess
to build vocabularies and generate Fairseq binary training files.
"""

# ─────────────────────────────────────────────────────────────────────────────────────────────────
print(
    f"🔎🔎 run_fairseq_preprocessing: {exp_args.run_fairseq_preprocessing} "
    f"{'(RUNNING)✔️✔️' if exp_args.run_fairseq_preprocessing else '(SKIPPED)❌❌'}"
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if exp_args.run_fairseq_preprocessing:
        
    import subprocess


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  1️⃣ ======== Check dataset files before preprocessing ==========================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    # ✅ Same folder you used above
    DATA_DIR = r"C:\Users\emeka\Research\ModelCUDA\Big_Data_Journal\Comparison\Code\Paper\github3\IWSLT\Transformer_Baselines_EN_ZH\Dataset"
    DESTDIR  = r"data-bin\iwslt17.en-zh"  # name it how you like
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    required_files = [
        os.path.join(DATA_DIR, "train.spm.en"),
        os.path.join(DATA_DIR, "train.spm.zh"),
        os.path.join(DATA_DIR, "validation.spm.en"),
        os.path.join(DATA_DIR, "validation.spm.zh"),
        os.path.join(DATA_DIR, "test.spm.en"),
        os.path.join(DATA_DIR, "test.spm.zh"),
    ]

    for f in required_files:
        assert os.path.exists(f), f"❌ Missing file: {f}"

    print("✅ All SentencePiece files found.")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    
    
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  2️⃣ ======== Configure preprocessing ============================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print("🚀 Running fairseq-preprocess for EN→ZH...")

    cmd = [
        "fairseq-preprocess",
        "--source-lang", "en",             # ✅Source = English
        "--target-lang", "zh",             # ✅Target = Chinese
        # use the SPM-encoded prefixes you just created
        "--trainpref", os.path.join(DATA_DIR, "train.spm"),
        "--validpref", os.path.join(DATA_DIR, "validation.spm"),
        "--testpref",  os.path.join(DATA_DIR, "test.spm"),
        "--destdir", DESTDIR,
        "--workers", "8"
    ]
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  3️⃣ ======== Run preprocessing ==================================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    subprocess.run(cmd, check=True)
    print(f"✅ Preprocessing complete! Binarized EN→ZH data saved in {DESTDIR}")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 4. FAIRSEQ SETUP, ACTIVATION PATCHING | XXX --------------------------------------####################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ========  4.1 Device Configuration =========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  4.2 Custom Activation Registration & Patching ====================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# Ensure activation folder is in sys.path
if ACTIVATION_PATH not in sys.path:
    sys.path.append(ACTIVATION_PATH)

# Backup original get_activation_fn
orig_get_activation_fn = utils.get_activation_fn
# ────────────────────────────────────────────────────────────────────────────────────────────────
def custom_get_activation_fn(activation: str):
    activation = activation.lower()
    if activation == "tanhexp":
        from activation.TanhExp import TanhExp
        print("✅ Registered custom activation: TanhExp")
        return TanhExp()
    
    elif activation == "fftgate":
        from activation.FFTGate import FFTGate
        print("✅ Registered custom activation: FFTGate")
        return FFTGate()
    
    elif activation == "geglu":        
        from activation.GEGLU import GEGLU
        print("✅ Registered custom activation: GEGLU")
        return GEGLU()

    else:
        # ✅ fallback to original fairseq activations
        return orig_get_activation_fn(activation)
# ────────────────────────────────────────────────────────────────────────────────────────────────
# Patch fairseq utils
utils.get_activation_fn = custom_get_activation_fn

# ────────────────────────────────────────────────────────────────────────────────────────────────
print(f"⚡ Using activation function: {exp_args.act_name.lower()}")

# Test-call the patched activation to confirm it works
try:
    act_fn = utils.get_activation_fn(exp_args.act_name.lower())
    print(f"🔍 Activation function resolved: {act_fn}")
except Exception as e:
    print(f"❌ Failed to resolve activation '{exp_args.act_name.lower()}': {e}")
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Manually inject custom activation into config
cfg.model.activation_fn = exp_args.act_name.lower()
print(f"⚡ Overwrote cfg.model.activation_fn = {cfg.model.activation_fn}")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ========  4.3 Fairseq Task & Dataset Initialization ========================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Load dictionaries
src_dict = Dictionary.load(os.path.join(fairseq_args.data, f"dict.{fairseq_args.source_lang}.txt"))
tgt_dict = Dictionary.load(os.path.join(fairseq_args.data, f"dict.{fairseq_args.target_lang}.txt"))

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Setup task
task = TranslationTask.setup_task(cfg.task)
task.load_dataset("train")
task.load_dataset("test")
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ✅✅ Print translation direction (EN→ZH confirmation)
print(f"🌍 Translation direction: {fairseq_args.source_lang} → {fairseq_args.target_lang}")
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ✅✅ Inspect dataset lengths after loading
train_dataset = task.dataset("train")
test_dataset  = task.dataset("test")
# ────────────────────────────────────────────────────────────────────────────────────────────────

print(f"📏 Train set - Max source length: {train_dataset.src_sizes.max()}, Max target length: {train_dataset.tgt_sizes.max()}")
print(f"📏 Test set  - Max source length: {test_dataset.src_sizes.max()}, Max target length: {test_dataset.tgt_sizes.max()}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Inspect data you’d lose if you lower max_positions
def check_percentage(dataset, N):
    src_long = (dataset.src_sizes > N).sum()
    tgt_long = (dataset.tgt_sizes > N).sum()
    total = len(dataset)

    print(f"Dataset size: {total}")
    print(f"  > {N} tokens (source): {src_long} ({100*src_long/total:.2f}%)")
    print(f"  > {N} tokens (target): {tgt_long} ({100*tgt_long/total:.2f}%)")
    print()

# ✅✅ Check on train + test
check_percentage(train_dataset, exp_args.max_source_positions)
check_percentage(test_dataset, exp_args.max_source_positions)
# ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 5. MODEL INITIALIZATION | XXX ----------------------------------------------------####################
########################################################################################################################

# ================================================================================================
# 🏷️========================= MODEL INITIALIZATION===============================================
# ================================================================================================
# ================================================================================================
####------------------ 0️⃣ 1️⃣ 2️⃣ 3️⃣ 4️⃣ 5️⃣ 6️⃣ 7️⃣ 8️⃣ 9️⃣ ------------------------------------####



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  ======== 🔖🔖  5.1 Model Construction 🔖🔖====================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 


# ================================================================================================
# 1️⃣ 🧠 Build Fairseq Baseline Transformer
# ================================================================================================
# ================================================================================================
# 1️⃣ 🧠 Build Fairseq Baseline Transformer
# ================================================================================================

if exp_args.net == "Transformer":

    model = task.build_model(cfg.model)

    print(f"✅ Loaded Model: {model.__class__.__name__}")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ================================================================================================
# 2️⃣ 🧠 Build RTM_Former 📐🔑🏷️
# ================================================================================================
# ================================================================================================
# 2️⃣ 🧠 Build RTM_Former 📐🔑🏷️
# ================================================================================================
elif exp_args.net == "RTM_Former":


    rtm_layers = []

    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 1️⃣ 🧩🧠 === FFN Insertion === ♻️♻️ ========================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 1️⃣ 🧩🧠 === FFN Insertion === ♻️♻️ ========================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    if exp_args.rtm_insertion_type == "ffn":

        # ================================================================================================
        # 1️⃣.0️⃣ 🧠 Build RTM_Former  
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:ffn🧬🧬")
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:ffn🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print("\nMODEL INIT CHECK")
        print(model.decoder.layers[0].fc1.weight[0, :10])
        print()
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────────────────────────────────       
        # =============================================================================
        # 1️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder FFNs 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i >= len(model.decoder.layers) - exp_args.rtm_num_layers:

                original_activation_fn = layer.activation_fn

                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,
                    rtm_rotation=exp_args.rtm_rotation                 
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:

                    # 🔒 Configure RTM coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Activation Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ───────────────────────────────────────────────────────────────────────────────────────────────── 
                # ─────────────────────────────────────────────────────────────────────────────────────────────────           


                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                def wrapped_activation(
                    x,
                    act_fn=original_activation_fn,
                    rtm_module=rtm
                ):

                    # ==========================================================
                    # 1️⃣.2️⃣.1️⃣ Standard FFN activation
                    # ==========================================================
                    x = act_fn(x)

                    # ==========================================================
                    # 1️⃣.2️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 1️⃣.2️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm_module.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0 + exp_args.rtm_transition_amplification * transition_scale
                    # )

                    # x = x * (
                    #     1.0 + rtm_module.amplification * transition_scale
                    # )

                    global RTM_SCALE

                    x_base = x

                    x_mod = x * (
                        1.0 + rtm_module.amplification * transition_scale
                    )

                    x_mod = rtm_module(x_mod)
                    # ==========================================================
                    # 1️⃣.2️⃣.6️⃣ RTM refinement
                    # ==========================================================
                    # x = rtm_module(x)
                    x = x_base + RTM_SCALE * (x_mod - x_base)

                    return x
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               

                # =============================================================================
                # 📌📌 Replace decoder activation function
                # =============================================================================                 
                layer.activation_fn = wrapped_activation
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  


                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 FFN Insertion | RTM inserted into decoder layer {i}")                
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────




    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣ 🧩🧠 === Cross-Attention Insertion ♻️♻️ ================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣ 🧩🧠 === Cross-Attention Insertion ♻️♻️ ================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────

    elif exp_args.rtm_insertion_type == "cross_attn":
    
        # ================================================================================================
        # 2️⃣.0️⃣ 🧠 Build RTM_Former 🅰️🔼 
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:cross_attn🧬🧬")
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:cross_attn🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder Cross-Attention 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i >= len(model.decoder.layers) - exp_args.rtm_num_layers:

                # =============================================================================
                # 2️⃣.2️⃣.1️⃣📌📌 RTM module
                # =============================================================================
                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,
                    rtm_rotation=exp_args.rtm_rotation                    
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:

                    # 🔒 Configure RTM coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Attention Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────


                # =============================================================================
                # 2️⃣.2️⃣.2️⃣📌📌 Store original encoder-attention forward
                # =============================================================================
                original_encoder_attn_forward = layer.encoder_attn.forward

                # =============================================================================
                # 2️⃣.2️⃣.3️⃣📌📌 Wrapped Cross-Attention forward
                # =============================================================================
                def wrapped_encoder_attn_forward(
                    query,
                    key,
                    value,
                    **kwargs
                ):

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.1️⃣ Original Cross-Attention
                    # ==========================================================
                    x, attn = original_encoder_attn_forward(
                        query=query,
                        key=key,
                        value=value,
                        **kwargs
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0
                    #     +
                    #     exp_args.rtm_transition_amplification
                    #     *
                    #     transition_scale
                    # )

                    x = x * (
                        1.0
                        +
                        rtm.amplification
                        *
                        transition_scale
                    )

                    return x, attn
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────


                # =============================================================================
                # 📌📌 Replace encoder-attention forward
                # =============================================================================
                layer.encoder_attn.forward = wrapped_encoder_attn_forward
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────

                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 Cross-Attention Insertion | RTM inserted into decoder layer {i}") 

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────



    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 3️⃣ 🧩🧠 === FFN Depth-Weighted Insertion === ♻️♻️ =========================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 3️⃣ 🧩🧠 === FFN Depth-Weighted Insertion === ♻️♻️ =========================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    elif exp_args.rtm_insertion_type == "ffn_depth_weighted":

        # ================================================================================================
        # 1️⃣.0️⃣ 🧠 Build RTM_Former  
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: Depth-Weighted RTM_Former for insertion_typ:ffn_depth_weighted🧬🧬")
        print(f"♻️♻️ Loaded Model: Depth-Weighted RTM_Former for insertion_typ:ffn_depth_weighted🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # 🔒 IMPORTANT: reset RNG after importing RTM_Former, before the real training model is built (For Repeatable Initialization)
        set_seed_torch(seed1)
        set_seed_main(seed2)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print("\nMODEL INIT CHECK")
        print(model.decoder.layers[0].fc1.weight[0, :10])
        print()
        # ==========================================================
        # 🚦🔍 Select layers
        # ========================================================== 
        selected_layers = list(
            range(
                len(model.decoder.layers) - exp_args.rtm_num_layers,
                len(model.decoder.layers)
            )
        )
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────────────────────────────────       
        # =============================================================================
        # 1️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder FFNs 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i in selected_layers:

                original_activation_fn = layer.activation_fn

                layer_rank = selected_layers.index(i)


                # =====================================================
                # ⏭️⏭️ Forward direction Layer scale ⏭️⏭️
                # =====================================================               
                if exp_args.rtm_layer_scale_direction == "forward":
                    layer_scale = exp_args.rtm_layer_scale_min + (
                        1.0 - exp_args.rtm_layer_scale_min
                    ) * (
                        layer_rank / max(1, exp_args.rtm_num_layers - 1)
                    )
                # =====================================================
                # ⏪⏪ Reverse direction Layer scale ⏪⏪
                # =====================================================  
                elif exp_args.rtm_layer_scale_direction == "reverse":
                    layer_scale = 1.0 - (
                        1.0 - exp_args.rtm_layer_scale_min
                    ) * (
                        layer_rank / max(1, exp_args.rtm_num_layers - 1)
                    )
                # =====================================================
                # ⏪🔒 Reverse direction + Fixed Final Layer Scale ⏪🔒
                # =====================================================
                elif exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                    if layer_rank == exp_args.rtm_num_layers - 1:
                        layer_scale = exp_args.rtm_final_layer_scale
                    else:
                        layer_scale = 1.0 - (
                            1.0 - exp_args.rtm_layer_scale_min
                        ) * (
                            layer_rank / max(1, exp_args.rtm_num_layers - 2)
                        ) 
                # ────────────────────────────────────────────────────────────────────────────────────────
                else:
                    raise ValueError(f"Unknown RTM layer scale direction: {exp_args.rtm_layer_scale_direction}")
                # ────────────────────────────────────────────────────────────────────────────────────────

                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,  
                    gate_type=exp_args.rtm_gate_type,
                    rtm_rotation=exp_args.rtm_rotation                                     
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility + Parameter Tracking)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:
                    
                    # 🔒 Configure RTM learnable coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Activation Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ─────────────────────────────────────────────────────────────────────────────────────────────────                
                # ─────────────────────────────────────────────────────────────────────────────────────────────────

                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                def wrapped_activation(
                    x,
                    act_fn=original_activation_fn,
                    rtm_module=rtm,
                    layer_scale=layer_scale
                ):

                    # ==========================================================
                    # 1️⃣.2️⃣.1️⃣ Standard FFN activation
                    # ==========================================================
                    x = act_fn(x)

                    # ==========================================================
                    # 1️⃣.2️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 1️⃣.2️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm_module.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0 + exp_args.rtm_transition_amplification * transition_scale
                    # )

                    # x = x * (
                    #     1.0 + rtm_module.amplification * transition_scale
                    # )

                    global RTM_SCALE

                    x_base = x

                    x_mod = x * (
                        1.0 + rtm_module.amplification * transition_scale
                    )

                    x_mod = rtm_module(x_mod)
                    # ==========================================================
                    # 1️⃣.2️⃣.6️⃣ RTM refinement
                    # ==========================================================

                    # ----------------------------------------------------------
                    # 🏔️ Depth-Aware RTM Scaling (Layer-Adaptive Modulation)
                    # ---------------------------------------------------------- 
                    if exp_args.rtm_depth_weight:

                        x = x_base + RTM_SCALE * layer_scale * (x_mod - x_base)

                    # ----------------------------------------------------------
                    # ⚖️ Uniform RTM Scaling (Same Modulation Across Layers)
                    # ----------------------------------------------------------
                    else:

                        x = x_base + RTM_SCALE * (x_mod - x_base)
                    # ─────────────────────────────────────────────────────────────────────────────────────────────────

                    return x
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               

                # =============================================================================
                # 📌📌 Replace decoder activation function
                # =============================================================================                 
                layer.activation_fn = wrapped_activation
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  


                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 FFN Insertion | RTM inserted into decoder layer {i} | layer_scale={layer_scale:.3f}")              
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: Depth-Weighted RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    else:
        raise ValueError(f"Unknown RTM insertion type: {exp_args.rtm_insertion_type}")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print(f"📣📣 RTM_Former for insertion_typ={exp_args.rtm_insertion_type} | Loaded Model={exp_args.net }♻️♻️")
    print(f"📣📣 RTM_Former for insertion_typ={exp_args.rtm_insertion_type} | Loaded Model={exp_args.net }♻️♻️")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────



    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  ======== 🔖  5.1.1 Model Configuration Hyperparameter Summary ==================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 
    if exp_args.net == "RTM_Former":

        rtm_log = (
            f"📣 RTM Configuration             : "
            f"alpha={exp_args.rtm_alpha} | "
            f"amp={exp_args.rtm_transition_amplification} | "
            f"layers={exp_args.rtm_num_layers} | "
            f"insertion={exp_args.rtm_insertion_type} | "
            f"scale_init={exp_args.rtm_scale} | "
            f"start_epoch={exp_args.rtm_start_epoch} | "
            f"full_epoch={exp_args.rtm_full_epoch}"
        )

        # Depth-weighted RTM only
        if exp_args.rtm_insertion_type == "ffn_depth_weighted":

            rtm_log += (
                f" | layer_scale_min={exp_args.rtm_layer_scale_min}"
                f" | layer_direction={exp_args.rtm_layer_scale_direction}"
                f" | gate={exp_args.rtm_gate_type}"
            )

            # Only exists for reverse_final_fixed
            if exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                rtm_log += (
                    f" | final_layer_scale={exp_args.rtm_final_layer_scale}"
                )

        print(rtm_log)


    else:
        print(
            f"📣 Baseline Configuration        : "
            f"model={exp_args.net}"
        )
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ======== 🔖🔖  5.2  Insert model to device 🔖🔖================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

model = model.to(DEVICE)
 # ─────────────────────────────────────────────────────────────────────────────────────────────────
print(f"🚀🚀 Model moved to device: {DEVICE} ♻️♻️")
print(f"🚀🚀 Model moved to device: {DEVICE} ♻️♻️")
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ======== 🔖🔖 5.3  Model Validation & Sanity Checks 🔖🔖=======================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 🔍 Print custom activations to confirm they are used
for name, module in model.named_modules():
    if isinstance(module, nn.Module) and module.__class__.__name__ in ["TanhExp", "FFTGate"]:
        print(f"🔎 Found custom activation in {name}: {module}")


# ✅ Sanity check Device
if DEVICE.type == "cuda":
    print(f"🚀 Model moved to CUDA: {torch.cuda.get_device_name(DEVICE)}")
else:
    print("⚠️ Using CPU (CUDA not available)")
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 6. PATH DEFINATION | XXX --------------------------------------------------------#####################
########################################################################################################################


# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 1️⃣ 📌📌 ======== ENSURE DIRECTORY EXIST ========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# # ─────────────────────────────────────────────────────────────────────────────────────────────────
# # 🟡 === Checkpoint directories ===
# if not os.path.exists('checkpoint'):
#     os.makedirs('checkpoint')
# # ─────────────────────────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Results directories ===
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}')

if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === LR Log directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Training Log directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Model Summary directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')
# ────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣ 📌📌 ======== DEFINE DIRECTORY ==============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.1️⃣🔖 ==== Checkpoint Directory Selection | Normal Run vs Crash Run | save every epoch ===== 
# ────────────────────────────────────────────────────────────────────────────────────────────────

# 🅰️💾 Normal run: save checkpoints in standard folder
if not exp_args.crash_run:
    checkpoint_dir = f'./checkpoints/{exp_args.dataset_name}/{exp_args.net}/'

    # 📁 Create checkpoint folder if it does not exist
    os.makedirs(checkpoint_dir, exist_ok=True)

    print(f"🅰️💾 Normal Run Enabled | Saving checkpoints to: {checkpoint_dir}")
#---------------------------------------------------------------------------------------------------
# 🅱️♻️ Crash run: save checkpoints in recovery folder
else:
    checkpoint_dir = f'./checkpoints_crash_run/{exp_args.dataset_name}/{exp_args.net}/'

    # 📁 Create checkpoint folder if it does not exist    
    os.makedirs(checkpoint_dir, exist_ok=True)

    print(f"🅱️♻️ Crash Run Enabled | Saving recovery checkpoints to: {checkpoint_dir}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.2️⃣🔖 === Main Test & Train Results === 
train_results_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/Train_{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
test_results_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/Test_{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.3️⃣🔖  === LR, Training & Summary logs  === 
LR_save_paths = {
    "LR_history": f"./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_LR_log.txt"
}
save_paths = {
    "log_history": f"./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_training_logs.txt"
}
# ────────────────────────────────────────────────────────────────────────────────────────────────

# 2️⃣.4️⃣🔖 ===  Model summary path === 
model_summary_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_model_summary.txt'
model_summary_full_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_model_summary_full.txt'
# ────────────────────────────────────────────────────────────────────────────────────────────────

# # ✅🔖 === Path template for saving checkpoints at every epoch | Part2 (if scratch occured) ===
# checkpoint_dir = f'./checkpoints/{exp_args.dataset_name}/tanhexp_AfterCrash_41_49epochs/'
# os.makedirs(checkpoint_dir, exist_ok=True)
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 7. TRAINING INITIALIZATION & OPTIMIZATION SETUP | XXX ---------------------------#####################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ========  7.1 AMP GradScaler ===============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Initialize AMP GradScaler once globall
scaler = torch.cuda.amp.GradScaler()
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  Training Criterion ===============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

criterion = LabelSmoothingCrossEntropy(smoothing=exp_args.smoothing)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ========   Optimizer Configuration =========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ♻️ Adam optimizer (β1=0.9, β2=0.98, eps=1e-9) ♻️
# ─────────────────────────────────────────────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=exp_args.lr,
    betas=(0.9, 0.98),
    eps=exp_args.eps,
    weight_decay=exp_args.weight_decay,
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 4️⃣ ========  Learning Rate Scheduler ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ✅  Transformer LR schedule (warmup + inverse sqrt decay)
# --------------------------

warmup_updates = exp_args.warmup_updates   # configurable warmup steps
peak_lr = exp_args.lr

# ────────────────────────────────────────────────────────────────────────────────────────────────
def lr_lambda(step):
    step = max(1, step)
    if step == 1:
        print(f"🚀 Warmup started at step {step} (target peak LR = {peak_lr}).")
    if step == warmup_updates:
        actual_lr = optimizer.param_groups[0]["lr"]
        print(f"✅ Warmup finished at step {step}. Peak LR reached: {actual_lr:.8f}")
    if step == warmup_updates + 1:
        print(f"📉 Inverse square root decay started at step {step}.")
    
    if step <= warmup_updates:
        # Linear warmup
        scale = step / warmup_updates
    else:
        # Inverse square root decay
        scale = (warmup_updates ** 0.5) / (step ** 0.5)
    return scale

# ────────────────────────────────────────────────────────────────────────────────────────────────

lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 5️⃣ ========  Global Tracking Variables ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ global initialization
best_bleu = 0.0
total_test_duration = 0.0
RTM_SCALE_LOG = ""
# ─────────────────────────────────────────────────────────────────────────────────────────────────

Python: 3.10.16 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:19:12) [MSC v.1929 64 bit (AMD64)]
Torch: 2.2.0+cu121 | CUDA available: True
Fairseq: 0.12.2
OmegaConf: 2.0.6
Hydra-Core: 1.0.7
✅ Current working directory: C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH
✅ sys.path updated:
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\python310.zip
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\DLLs
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env
   📂 
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\win32
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\win32\lib
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\Pythonwin
   📂 C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH
   📂 C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH\models
   📂 C:\

c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\modules\module.py:1150: UserWarning: expandable_segments not supported on this platform (Triggered internally at ..\c10/cuda/CUDAAllocatorConfig.h:30.)
  return t.to(device, dtype if t.is_floating_point() or t.is_complex() else None, non_blocking)


In [ ]:
########################################################################################################################
####-------| NOTE 11. TEST ALL CHECKPOINTS (BLEU per epoch)  | XXX -----------------------------########################
########################################################################################################################
########################################################################################################################
####-------| NOTE 11. TEST ALL CHECKPOINTS (BLEU per epoch)  | XXX -----------------------------########################
########################################################################################################################


# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ======== Paths for checkpoint BLEU sweeps ===================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Paths for checkpoint BLEU sweeps
test_checkpoint_results_path = (
    f'./Results/{exp_args.dataset_name}/{exp_args.net}/'
    f'TestCheckpoints_{exp_args.net}_'
    f'{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
)
# ────────────────────────────────────────────────────────────────────────────────────────────────


# # ✅ Paths for checkpoint BLEU sweeps | Part2 (if Cratch occured)
# test_checkpoint_results_path = (
#     f'./Results/{exp_args.dataset_name}/{exp_args.act_name}/'
#     f'TestCheckpoints_{exp_args.act_name}_{exp_args.net}_'
#     f'{exp_args.dataset_name}_{exp_args.optimizer1}_tanhexp_AfterCrash_41_49epochs.txt'
# )
# ────────────────────────────────────────────────────────────────────────────────────────────────


# # ─────────────────────────────────────────────────────────────────────────────────────────────────
# #  2️⃣ ========  BLEU Detokenization (🔖 EN/FR/DE) ================================================
# # ───────────────────────────────────────────────────────────────────────────────────────────────── 
# # ────────────────────────────────────────────────────────────────────────────────────────────────
# def detokenize_for_bleu(s: str) -> str:
#     s = s.replace("@@ ", "").replace("@@", "")  # BPE
#     s = s.replace("▁", " ")                     # SentencePiece
#     s = " ".join(s.split())                     # normalize spaces
#     s = re.sub(r"\s+([?.!,])", r"\1", s)        # punctuation
#     return s.strip()
# # ────────────────────────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  BLEU Detokenization (🔖 Chinese) =================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
def detokenize_for_bleu(s: str) -> str:
    # For Chinese, don't remove SentencePiece boundaries or spaces | Only remove BPE continuation markers
    # sacreBLEU's "zh" tokenizer expects raw characters.
    return s.replace("@@ ", "").replace("@@", "").strip()
# ────────────────────────────────────────────────────────────────────────────────────────────────




# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ======== Test_Checkpoint Function ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ test_checkpoint function
def test_checkpoint_epochs(task, model, criterion, checkpoint_dir, test_checkpoint_results_path):
    """
    Evaluate BLEU (beam=5) for all checkpoints in checkpoint_dir.
    Save results to test_checkpoint_results_path in the same format as normal test logs.
    """
# ────────────────────────────────────────────────────────────────────────────────────────────────
    global RTM_SCALE, RTM_SCALE_LOG
    # ────────────────────────────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Collect checkpoints sorted by epoch number in filename
    def extract_epoch_num(filename):
        match = re.search(r"epoch_(\d+)", filename)
        return int(match.group(1)) if match else float("inf")  # inf if no epoch number

    ckpts = sorted(
        glob.glob(os.path.join(checkpoint_dir, "*.pt")),
        key=lambda x: extract_epoch_num(os.path.basename(x))
    )

    if not ckpts:
        print(f"⚠️ No checkpoints found in {checkpoint_dir}")
        return

    # # ────────────────────────────────────────────────────────────────────────────────────────────────
    # # ✅ Reset output file
    # os.makedirs(os.path.dirname(test_checkpoint_results_path), exist_ok=True)
    # with open(test_checkpoint_results_path, "w", encoding="utf-8") as f:
    #     f.write("")
    # # ────────────────────────────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 📦 Reset output file + write model status
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    #1️⃣ Create output folder if it does not exist
    os.makedirs(os.path.dirname(test_checkpoint_results_path), exist_ok=True)

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Normal run: reset file
    # ✅ Crash run : append to existing file
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣♻️ Normal run: reset file
    if not exp_args.crash_run:

        with open(test_checkpoint_results_path, "w", encoding="utf-8") as f:
            f.write("")

    # 3️⃣⏭️ Crash run : append to existing file
    else:

        with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
            f.write("\n♻️ Resuming checkpoint testing after crash\n\n")
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 4️⃣✍️  Write test result header (model information)
    with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
        f.write("")

        # ---------------------------------------------------------------------------
        # 4️⃣.1️⃣🏷️ Testing Model Status
        # ---------------------------------------------------------------------------
        if exp_args.net == "RTM_Former":

            f.write(
                f"Model: RTM_Former\n\n"
            )

        else:

            f.write(
                f"Model: Baseline Fairseq Transformer\n\n"
            )        
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ────────────────────────────────────────────────────────────────────────────────────────────────



    # ────────────────────────────────────────────────────────────────────────────────────────────────
    best_bleu = 0.0
    total_test_duration = 0.0
    # ────────────────────────────────────────────────────────────────────────────────────────────────


    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    for ckpt in ckpts:

        # # ────────────────────────────────────────────────────────────────────────────────
        # # ✅ 📌📌 Load checkpoint (OLD) 🅰️🔼 
        # # ────────────────────────────────────────────────────────────────────────────────        
        # print(f"🔄 Loading checkpoint: {ckpt}")
        # state = torch.load(ckpt, map_location=DEVICE)
        # model.load_state_dict(state["model_state"], strict=False)
        # model.to(DEVICE)
        # model.eval()
        # --------------------------------------------------------------------------------------------------
        # ────────────────────────────────────────────────────────────────────────────────
        # ✅📌📌 Load checkpoint (NEW) 🅱️🔼 Restore RTM schedule for checkpoint testing using same strategy as main loop
        # ────────────────────────────────────────────────────────────────────────────────
        if exp_args.net == "Transformer":
            print(f"🔄 Loading checkpoint Baseline Transformer : {ckpt}")
            state = torch.load(ckpt, map_location=DEVICE)
            model.load_state_dict(state["model_state"], strict=False)
            model.to(DEVICE)
            model.eval()
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        elif exp_args.net == "RTM_Former":
            print(f"🔄 Loading checkpoint RTM_Former : {ckpt}")
            state = torch.load(ckpt, map_location=DEVICE)
            model.load_state_dict(state["model_state"], strict=False)            
            

            RTM_START_EPOCH = exp_args.rtm_start_epoch
            RTM_FULL_EPOCH = exp_args.rtm_full_epoch
            ckpt_epoch = state["epoch"]

            if ckpt_epoch < RTM_START_EPOCH:
                RTM_SCALE = exp_args.rtm_scale
            else:
                progress = min(
                    1.0,
                    (ckpt_epoch - RTM_START_EPOCH) / (RTM_FULL_EPOCH - RTM_START_EPOCH)
                )

                RTM_SCALE = 0.5 * (1.0 - math.cos(math.pi * progress))

            print(f"📐 Checkpoint RTM_SCALE for epoch {ckpt_epoch}: {RTM_SCALE:.3f}")

            if exp_args.rtm_insertion_type == "ffn_depth_weighted":

                if exp_args.rtm_layer_scale_direction == "forward":
                    depth_scales = [
                        exp_args.rtm_layer_scale_min
                        + (1.0 - exp_args.rtm_layer_scale_min)
                        * (j / max(1, exp_args.rtm_num_layers - 1))
                        for j in range(exp_args.rtm_num_layers)
                    ]

                elif exp_args.rtm_layer_scale_direction == "reverse":
                    depth_scales = [
                        1.0
                        - (1.0 - exp_args.rtm_layer_scale_min)
                        * (j / max(1, exp_args.rtm_num_layers - 1))
                        for j in range(exp_args.rtm_num_layers)
                    ]

                elif exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                    depth_scales = []

                    for j in range(exp_args.rtm_num_layers):
                        if j == exp_args.rtm_num_layers - 1:
                            scale_j = exp_args.rtm_final_layer_scale
                        else:
                            scale_j = (
                                1.0
                                - (1.0 - exp_args.rtm_layer_scale_min)
                                * (j / max(1, exp_args.rtm_num_layers - 2))
                            )

                        depth_scales.append(scale_j)

                else:
                    raise ValueError(
                        f"Unknown RTM layer scale direction: {exp_args.rtm_layer_scale_direction}"
                    )

                final_scales = [RTM_SCALE * s for s in depth_scales]

                RTM_SCALE_LOG = f" | RTM_SCALE={RTM_SCALE:.3f}"
                RTM_SCALE_LOG += (
                    f" | RTM_layer_scales({exp_args.rtm_layer_scale_direction})="
                    f"{[round(s, 3) for s in final_scales]}"
                )

                print(
                    f"📐 Checkpoint final RTM layer scales "
                    f"({exp_args.rtm_layer_scale_direction}): "
                    f"{[round(s, 3) for s in final_scales]}"
                )
            model.to(DEVICE)
            model.eval()               
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        else:
            raise ValueError(f"Unknown model type: {exp_args.net}")
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ────────────────────────────────────────────────────────────────────────────────────────────────


        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Build test iterator
        test_iter = task.get_batch_iterator(
            dataset=task.dataset("test"),
            max_tokens=fairseq_args.max_tokens,   # cut in half option: fairseq_args.max_tokens // 2
            max_positions=(exp_args.max_source_positions, exp_args.max_target_positions),
            seed=seed1,
            num_shards=exp_args.num_shards,
            shard_id=exp_args.shard_id,
            num_workers=exp_args.num_workers,
            ignore_invalid_inputs=exp_args.ignore_invalid_inputs,
        ).next_epoch_itr(shuffle=False)
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ References
        true_references = []
        for sample in test_iter:
            targets = sample["target"]
            for t in targets:
                t_no_pad = t[t != tgt_dict.pad()]
                ref_str = tgt_dict.string(t_no_pad, bpe_symbol="@@")
                true_references.append(detokenize_for_bleu(ref_str))
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Rebuild iterator for decoding
        test_iter = task.get_batch_iterator(
            dataset=task.dataset("test"),
            max_tokens=fairseq_args.max_tokens,    # cut in half option: fairseq_args.max_tokens // 2
            max_positions=(exp_args.max_source_positions, exp_args.max_target_positions),
            seed=seed1,
            num_shards=exp_args.num_shards,
            shard_id=exp_args.shard_id,
            num_workers=exp_args.num_workers,
            ignore_invalid_inputs=exp_args.ignore_invalid_inputs,
        ).next_epoch_itr(shuffle=False)
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Build generator with beam=5
        gen_cfg = deepcopy(cfg.generation)
        gen_cfg.beam = exp_args.beam         # 🎀🔖 5
        gen_cfg.lenpen = exp_args.lenpen     # 🎀🔖 1.0 
        generator = task.build_generator([model], gen_cfg)
        # ────────────────────────────────────────────────────────────────────────────────────────────────


 

        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Run evaluation
        sys_outputs, test_loss = [], 0.0
        start_time = time.time()

        with torch.no_grad():
            with tqdm(enumerate(test_iter), total=len(test_iter), desc=f"Testing {os.path.basename(ckpt)}") as progress:
                for i, sample in progress:   # 👈 keep i!
                    sample = utils.move_to_cuda(sample) if torch.cuda.is_available() else sample
                    src_tokens = sample["net_input"]["src_tokens"]
                    prev_output_tokens = sample["net_input"]["prev_output_tokens"]
                    target = sample["target"]
                    src_lengths = (src_tokens != src_dict.pad()).sum(dim=1)

                    with torch.cuda.amp.autocast():
                        logits, _ = model(src_tokens, src_lengths, prev_output_tokens)
                        pad = tgt_dict.pad()
                        logits2d = logits.view(-1, logits.size(-1))
                        target1d = target.view(-1)
                        mask = target1d != pad
                        loss = criterion(logits2d[mask], target1d[mask]) if mask.any() else logits2d.sum() * 0.0
                    test_loss += loss.item()

                    hypos = task.inference_step(generator, [model], sample)
                    for j, hypos_j in enumerate(hypos):
                        hypo_tokens = [tok for tok in hypos_j[0]["tokens"].tolist() if tok != tgt_dict.pad()]
                        sys_str = tgt_dict.string(hypo_tokens, bpe_symbol="@@")
                        sys_outputs.append(detokenize_for_bleu(sys_str))

                    # ✅ Update progress bar postfix with running average
                    progress.set_postfix(loss=round(test_loss / (i + 1), 3))
       # ────────────────────────────────────────────────────────────────────────────────────────────────




       # ────────────────────────────────────────────────────────────────────────────────────────────────
        avg_loss = test_loss / max(1, len(test_iter))
        # bleu = sacrebleu.corpus_bleu(sys_outputs, [true_references])                 #🔖 EN/FR/DE
        bleu = sacrebleu.corpus_bleu(sys_outputs, [true_references], tokenize="zh")    #🔖 Chinese

        duration = time.time() - start_time
        total_test_duration += duration
       # ────────────────────────────────────────────────────────────────────────────────────────────────


       # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Write results
        with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
            f.write(f"Checkpoint {os.path.basename(ckpt)} | Test Loss: {avg_loss:.3f} | sacreBLEU: {bleu.score:.2f} | Beam=5\n")

        print(f"✅ {os.path.basename(ckpt)} | Loss {avg_loss:.3f} | sacreBLEU {bleu.score:.2f}")


        # 🧹 add visual spacing between runs
        tqdm.write("")  
       # ────────────────────────────────────────────────────────────────────────────────────────────────

       # ────────────────────────────────────────────────────────────────────────────────────────────────
        if bleu.score > best_bleu:
            best_bleu = bleu.score

    with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
        f.write(f"\n🏆 Best BLEU Score: {best_bleu:.2f}\n")
        total_mins, total_secs = divmod(total_test_duration, 60)
        f.write(f"\n🕒 Total Test Time over all checkpoints: {int(total_mins)} min {total_secs:.2f} sec\n")

    print(f"🏆 Best BLEU over checkpoints: {best_bleu:.2f}")
    # ────────────────────────────────────────────────────────────────────────────────────────────────



# ────────────────────────────────────────────────────────────────────────────────────────────────
# # ✅ Free unused GPU memory BEFORE training starts
# torch.cuda.empty_cache()
# torch.cuda.reset_peak_memory_stats()

# 4️⃣ Call the function
test_checkpoint_epochs(task, model, criterion, checkpoint_dir, test_checkpoint_results_path)
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────────────────────────

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_37_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 37: 1.000


Testing epoch_37_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt:   0%|          | 0/82 [00:00<?, ?it/s]c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)
c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\functional.py:5109: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
Testing epoch_37_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:42<00:00,  1.94it/s, loss=4.28]


✅ epoch_37_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.278 | sacreBLEU 19.95

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_38_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 38: 1.000


Testing epoch_38_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:44<00:00,  1.83it/s, loss=4.29]


✅ epoch_38_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.286 | sacreBLEU 19.67

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_39_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 39: 1.000


...

Testing epoch_47_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:39<00:00,  2.08it/s, loss=4.23]


✅ epoch_47_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.231 | sacreBLEU 20.23

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_48_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 48: 1.000


Testing epoch_48_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:36<00:00,  2.24it/s, loss=4.24]


✅ epoch_48_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.239 | sacreBLEU 20.21

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_49_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 49: 1.000


: 

Restarted pytorch_env (Python 3.10.16)

In [ ]:
#####-------------------------------- NOTE MAIN IWSLT17 EN-ZH NOTE --------------------------------------------------#####
##########################################################################################################################
######################|--------------------------------------------------------------|####################################
###################################🔗 MAIN | TRAIN | TEST LOOP 🔗########################################################
######################|--------------------------------------------------------------|####################################
##########################################################################################################################
#####-------------------------------- NOTE MAIN IWSLT17 EN-ZH NOTE --------------------------------------------------#####



# 📄  main_IWSLT17En_Zh.py
########################################################################################################################
####-------| NOTE 1.1 IMPORTS LIBRARIES  | XXX -------------------------------------------------------##################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 📜 === Enable flexible CUDA memory allocation to reduce fragmentation ===
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ======================================================================================================
# 📜 === Core Libraries ===
# ======================================================================================================
import sys
import numpy as np
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import time
import random
import math
import re
import glob
from copy import deepcopy
import sacrebleu  # ✅ for BLEU evaluation
from timm.loss import LabelSmoothingCrossEntropy
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ======================================================================================================
# 📜 === Fairseq & Supporting Imports ===
# ======================================================================================================
import importlib.metadata as importlib_metadata
import fairseq  # type: ignore
from fairseq.data import Dictionary, data_utils, iterators          # type: ignore
from fairseq.tasks.translation import TranslationTask               # type: ignore
from fairseq import options                                         # type: ignore
from fairseq.dataclass.utils import convert_namespace_to_omegaconf  # type: ignore                                             
from fairseq import utils                                           # type: ignore 
import hydra                                                        # type: ignore  
# ────────────────────────────────────────────────────────────────────────────────────────────────
from ptflops import get_model_complexity_info
from calflops import calculate_flops
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ======================================================================================================
# ♻️ === Print environment summary for sanity check ===
# ======================================================================================================
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Fairseq:", fairseq.__version__)
print("OmegaConf:", importlib_metadata.version("omegaconf"))
print("Hydra-Core:", importlib_metadata.version("hydra-core"))
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 1.2. DEFINE PROJECT PATH | XXX ---------------------------------------------------####################
########################################################################################################################

# ✅ Define working directory
Project_PATH = r"C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH"
if os.getcwd() != Project_PATH:
    os.chdir(Project_PATH)
print(f"✅ Current working directory: {os.getcwd()}")


# ✅ Define absolute paths
PROJECT_PATH = Project_PATH
MODELS_PATH = os.path.join(Project_PATH, "models")
ACTIVATION_PATH = os.path.join(Project_PATH, "activation")


# ✅ Ensure necessary paths are in sys.path
for path in [PROJECT_PATH, MODELS_PATH, ACTIVATION_PATH]:
    if path not in sys.path:
        sys.path.append(path)

# ✅ Print updated sys.path for debugging
print("✅ sys.path updated:")
for path in sys.path:
    print("   📂", path)
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 1.3. OTHER IMPORTS | XXX ---------------------------------------------------------####################
########################################################################################################################

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 📜 ============  Imput parser safe for afno because of --fno-bias, etc  =======================
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 1️⃣✅ Import parser from parser_IWSLT17En_Zh.py
from parser_IWSLT17En_Zh import get_parser

# ✅ Create parser and parse arguments
parser = get_parser()
# args, unknown = parser.parse_known_args()

# ✅ IMPORTANT: Do NOT read Jupyter / VSCode kernel arguments
# This prevents the "--f" ambiguity issue
exp_args = parser.parse_args(args=[])

num_aug_splits = exp_args.aug_splits

print(f"✅ Parser imported successfully | num_aug_splits = {num_aug_splits}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣✅ Import Fairseq parser/config
from fairseq_config_IWSLT17En_Zh import get_fairseq_parser
# ✅ Build Fairseq configuration
fairseq_args, cfg = get_fairseq_parser(exp_args)


num_aug_splits = exp_args.aug_splits

print(f"✅ Parser imported successfully | num_aug_splits = {num_aug_splits}")

print(f"✅ Encoder Embed Dim: {fairseq_args.encoder_embed_dim}")
print(f"✅ Decoder Layers: {fairseq_args.decoder_layers}")
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 3️⃣✅ Import model summary utility
from models.model_summary.model_summary import save_model_summary
from models.model_summary.model_summary_full import save_model_summary_full

print("✅ model_summary.py loaded successfully")
print("✅ model_summary_full.py loaded successfully")
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 2. SEEDING FOR REPRODUCIBILITY | XXX ---------------------------------------------####################
########################################################################################################################

# ────────────────────────────────────────────────────────────────────────────────────────────────
def set_seed_torch(seed):
    torch.manual_seed(seed)
# ────────────────────────────────────────────────────────────────────────────────────────────────
def set_seed_main(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True      # ✅ Default: "False" for Faster, non-deterministic kernels | "True" to Ensure deterministic behavior for CuDNN (Slower)
    torch.backends.cudnn.benchmark = False         # ✅ Default: "True" for Autotune kernels for performance   | "False" Disable CuDNN's autotuning for reproducibility (Slower)

    torch.backends.cuda.matmul.allow_tf32 = False  # ✅ Disable TF32 (strict reproducibility)
    torch.backends.cudnn.allow_tf32 = False        # ✅ Disable TF32 (strict reproducibility)

    # torch.use_deterministic_algorithms(True, warn_only=True)

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ ============= Define Seed =============
seed1, seed2 = exp_args.seed1, exp_args.seed2
set_seed_torch(seed1)  
set_seed_main(seed2)  
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 3.1. PROCESSING DATASET 1 | XXX ---- TOKENIZATION & DATASET PREPARATION |---------####################
########################################################################################################################
########################################################################################################################
####-------| NOTE 3.1. PROCESSING DATASET 1 | XXX ------------| ENGLISH→CHINESE |-------------------####################
########################################################################################################################

"""
Stage 1:
Train SentencePiece tokenizer, encode the raw English→Chinese corpus,
and create aligned train/test text splits for neural machine translation.
"""

# ─────────────────────────────────────────────────────────────────────────────────────────────────
print(
    f"🔎🔎 run_dataset_tokenization: {exp_args.run_dataset_tokenization} "
    f"{'(RUNNING)✔️✔️' if exp_args.run_dataset_tokenization else '(SKIPPED)❌❌'}"
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if exp_args.run_dataset_tokenization:
    from datasets import load_dataset    
    import sentencepiece as spm

    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Path for EN→ZH dataset
    DATA_PATH = r"C:\Users\emeka\Research\ModelCUDA\Big_Data_Journal\Comparison\Code\Paper\github3\IWSLT\Transformer_Baselines_EN_ZH\Dataset"
    os.makedirs(DATA_PATH, exist_ok=True)
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  1️⃣ ======== Download and export official IWSLT17 dataset =======================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Download to your folder (as the Hugging Face cache)
    en_zh = load_dataset(
        "IWSLT/iwslt2017", "iwslt2017-en-zh",
        cache_dir=DATA_PATH,
        trust_remote_code=True
    )
    zh_en = load_dataset(
        "IWSLT/iwslt2017", "iwslt2017-zh-en",
        cache_dir=DATA_PATH,
        trust_remote_code=True
    )  # optional
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  2️⃣ ======== Dataset verification ===============================================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ sanity check (should show ~231k train)
    print(f"EN→ZH dataset: {en_zh}")
    print(f"ZH→EN dataset: {zh_en}")

    print("EN→ZH splits:", list(en_zh.keys()), {k: len(en_zh[k]) for k in en_zh.keys()})
    print("ZH→EN splits:", list(zh_en.keys()), {k: len(zh_en[k]) for k in zh_en.keys()})

    # optional assertions
    assert set(en_zh.keys()) == {"train", "validation", "test"}
    assert set(zh_en.keys()) == {"train", "validation", "test"}
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  3️⃣ ======== Export official train/validation/test splits ======================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Export official IWSLT17 splits to plain text (EN/ZH) under raw_zh_en_dataset_path
    def dump_split(ds_dict, split, src, tgt, out_dir):
        src_fp = os.path.join(out_dir, f"{split}.{src}")
        tgt_fp = os.path.join(out_dir, f"{split}.{tgt}")
        with open(src_fp, "w", encoding="utf-8") as fs, open(tgt_fp, "w", encoding="utf-8") as ft:
            for row in ds_dict[split]:
                tr = row["translation"]
                fs.write((tr[src] or "").strip() + "\n")
                ft.write((tr[tgt] or "").strip() + "\n")
        print(f"Wrote {split}: {src_fp} | {tgt_fp}")

    for split in ["train", "validation", "test"]:
        dump_split(en_zh, split, "en", "zh", DATA_PATH)
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  4️⃣ ========  Train SentencePiece model =========================================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────     

    print("🧠 Training SentencePiece model for EN→ZH...")

    # Use TRAIN split for SPM training
    SRC_EN = os.path.join(DATA_PATH, "train.en")
    TGT_ZH = os.path.join(DATA_PATH, "train.zh")

    spm_model_prefix = os.path.join(DATA_PATH, "spm_enzh")

    spm.SentencePieceTrainer.Train(
        input=f"{SRC_EN},{TGT_ZH}",
        model_prefix=spm_model_prefix,
        vocab_size=10000,
        character_coverage=0.9995,   # ✅ --character_coverage for rich charcater langauage: Japanese or Chinese = 0.9995 | other languages with small character set: 1.0
        model_type="bpe"
    )
    print(f"✅ SentencePiece model saved to: {spm_model_prefix}.model / .vocab")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  5️⃣ ========  Encode files ======================================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    print("⚙️ Encoding dataset...")

    # ✅ Load the trained SentencePiece model
    sp = spm.SentencePieceProcessor(model_file=f"{spm_model_prefix}.model")

    def encode_file(in_file, out_file):
        with open(in_file, encoding="utf-8") as fin, open(out_file, "w", encoding="utf-8") as fout:
            for line in fin:
                fout.write(" ".join(sp.encode(line.strip(), out_type=str)) + "\n")

    # ✅ Encode all official splits (no custom re-split)
    for split in ["train", "validation", "test"]:
        encode_file(os.path.join(DATA_PATH, f"{split}.en"), os.path.join(DATA_PATH, f"{split}.spm.en"))
        encode_file(os.path.join(DATA_PATH, f"{split}.zh"), os.path.join(DATA_PATH, f"{split}.spm.zh"))
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  6️⃣ ======== Use official train/validation/test splits =========================================
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print(f"✅ Using official IWSLT17 train/validation/test splits.")

    print(f"✅ Encoded EN→ZH data saved under: {DATA_PATH}")
    print(f"✅ Next: run fairseq-preprocess to binarize (on *.spm.en/*.spm.zh).")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 3.2. PROCESSING DATASET 2 | XXX ----- FAIRSEQ PREPROCESSING & BINARIZATION |------####################
########################################################################################################################
########################################################################################################################
####-------| NOTE 3.2. PROCESSING DATASET 2 | XXX ------------| ENGLISH→CHINESE |--------------------####################
########################################################################################################################

"""
Stage 2:
Preprocess the tokenized English→Chinese dataset using fairseq-preprocess
to build vocabularies and generate Fairseq binary training files.
"""

# ─────────────────────────────────────────────────────────────────────────────────────────────────
print(
    f"🔎🔎 run_fairseq_preprocessing: {exp_args.run_fairseq_preprocessing} "
    f"{'(RUNNING)✔️✔️' if exp_args.run_fairseq_preprocessing else '(SKIPPED)❌❌'}"
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if exp_args.run_fairseq_preprocessing:
        
    import subprocess


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  1️⃣ ======== Check dataset files before preprocessing ==========================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    # ✅ Same folder you used above
    DATA_DIR = r"C:\Users\emeka\Research\ModelCUDA\Big_Data_Journal\Comparison\Code\Paper\github3\IWSLT\Transformer_Baselines_EN_ZH\Dataset"
    DESTDIR  = r"data-bin\iwslt17.en-zh"  # name it how you like
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    required_files = [
        os.path.join(DATA_DIR, "train.spm.en"),
        os.path.join(DATA_DIR, "train.spm.zh"),
        os.path.join(DATA_DIR, "validation.spm.en"),
        os.path.join(DATA_DIR, "validation.spm.zh"),
        os.path.join(DATA_DIR, "test.spm.en"),
        os.path.join(DATA_DIR, "test.spm.zh"),
    ]

    for f in required_files:
        assert os.path.exists(f), f"❌ Missing file: {f}"

    print("✅ All SentencePiece files found.")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    
    
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  2️⃣ ======== Configure preprocessing ============================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print("🚀 Running fairseq-preprocess for EN→ZH...")

    cmd = [
        "fairseq-preprocess",
        "--source-lang", "en",             # ✅Source = English
        "--target-lang", "zh",             # ✅Target = Chinese
        # use the SPM-encoded prefixes you just created
        "--trainpref", os.path.join(DATA_DIR, "train.spm"),
        "--validpref", os.path.join(DATA_DIR, "validation.spm"),
        "--testpref",  os.path.join(DATA_DIR, "test.spm"),
        "--destdir", DESTDIR,
        "--workers", "8"
    ]
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  3️⃣ ======== Run preprocessing ==================================================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 

    subprocess.run(cmd, check=True)
    print(f"✅ Preprocessing complete! Binarized EN→ZH data saved in {DESTDIR}")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 4. FAIRSEQ SETUP, ACTIVATION PATCHING | XXX --------------------------------------####################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ========  4.1 Device Configuration =========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  4.2 Custom Activation Registration & Patching ====================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# Ensure activation folder is in sys.path
if ACTIVATION_PATH not in sys.path:
    sys.path.append(ACTIVATION_PATH)

# Backup original get_activation_fn
orig_get_activation_fn = utils.get_activation_fn
# ────────────────────────────────────────────────────────────────────────────────────────────────
def custom_get_activation_fn(activation: str):
    activation = activation.lower()
    if activation == "tanhexp":
        from activation.TanhExp import TanhExp
        print("✅ Registered custom activation: TanhExp")
        return TanhExp()
    
    elif activation == "fftgate":
        from activation.FFTGate import FFTGate
        print("✅ Registered custom activation: FFTGate")
        return FFTGate()
    
    elif activation == "geglu":        
        from activation.GEGLU import GEGLU
        print("✅ Registered custom activation: GEGLU")
        return GEGLU()

    else:
        # ✅ fallback to original fairseq activations
        return orig_get_activation_fn(activation)
# ────────────────────────────────────────────────────────────────────────────────────────────────
# Patch fairseq utils
utils.get_activation_fn = custom_get_activation_fn

# ────────────────────────────────────────────────────────────────────────────────────────────────
print(f"⚡ Using activation function: {exp_args.act_name.lower()}")

# Test-call the patched activation to confirm it works
try:
    act_fn = utils.get_activation_fn(exp_args.act_name.lower())
    print(f"🔍 Activation function resolved: {act_fn}")
except Exception as e:
    print(f"❌ Failed to resolve activation '{exp_args.act_name.lower()}': {e}")
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Manually inject custom activation into config
cfg.model.activation_fn = exp_args.act_name.lower()
print(f"⚡ Overwrote cfg.model.activation_fn = {cfg.model.activation_fn}")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ========  4.3 Fairseq Task & Dataset Initialization ========================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Load dictionaries
src_dict = Dictionary.load(os.path.join(fairseq_args.data, f"dict.{fairseq_args.source_lang}.txt"))
tgt_dict = Dictionary.load(os.path.join(fairseq_args.data, f"dict.{fairseq_args.target_lang}.txt"))

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Setup task
task = TranslationTask.setup_task(cfg.task)
task.load_dataset("train")
task.load_dataset("test")
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ✅✅ Print translation direction (EN→ZH confirmation)
print(f"🌍 Translation direction: {fairseq_args.source_lang} → {fairseq_args.target_lang}")
# ────────────────────────────────────────────────────────────────────────────────────────────────

# ✅✅ Inspect dataset lengths after loading
train_dataset = task.dataset("train")
test_dataset  = task.dataset("test")
# ────────────────────────────────────────────────────────────────────────────────────────────────

print(f"📏 Train set - Max source length: {train_dataset.src_sizes.max()}, Max target length: {train_dataset.tgt_sizes.max()}")
print(f"📏 Test set  - Max source length: {test_dataset.src_sizes.max()}, Max target length: {test_dataset.tgt_sizes.max()}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅✅ Inspect data you’d lose if you lower max_positions
def check_percentage(dataset, N):
    src_long = (dataset.src_sizes > N).sum()
    tgt_long = (dataset.tgt_sizes > N).sum()
    total = len(dataset)

    print(f"Dataset size: {total}")
    print(f"  > {N} tokens (source): {src_long} ({100*src_long/total:.2f}%)")
    print(f"  > {N} tokens (target): {tgt_long} ({100*tgt_long/total:.2f}%)")
    print()

# ✅✅ Check on train + test
check_percentage(train_dataset, exp_args.max_source_positions)
check_percentage(test_dataset, exp_args.max_source_positions)
# ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 5. MODEL INITIALIZATION | XXX ----------------------------------------------------####################
########################################################################################################################

# ================================================================================================
# 🏷️========================= MODEL INITIALIZATION===============================================
# ================================================================================================
# ================================================================================================
####------------------ 0️⃣ 1️⃣ 2️⃣ 3️⃣ 4️⃣ 5️⃣ 6️⃣ 7️⃣ 8️⃣ 9️⃣ ------------------------------------####



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  ======== 🔖🔖  5.1 Model Construction 🔖🔖====================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 


# ================================================================================================
# 1️⃣ 🧠 Build Fairseq Baseline Transformer
# ================================================================================================
# ================================================================================================
# 1️⃣ 🧠 Build Fairseq Baseline Transformer
# ================================================================================================

if exp_args.net == "Transformer":

    model = task.build_model(cfg.model)

    print(f"✅ Loaded Model: {model.__class__.__name__}")
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ================================================================================================
# 2️⃣ 🧠 Build RTM_Former 📐🔑🏷️
# ================================================================================================
# ================================================================================================
# 2️⃣ 🧠 Build RTM_Former 📐🔑🏷️
# ================================================================================================
elif exp_args.net == "RTM_Former":


    rtm_layers = []

    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 1️⃣ 🧩🧠 === FFN Insertion === ♻️♻️ ========================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 1️⃣ 🧩🧠 === FFN Insertion === ♻️♻️ ========================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    if exp_args.rtm_insertion_type == "ffn":

        # ================================================================================================
        # 1️⃣.0️⃣ 🧠 Build RTM_Former  
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:ffn🧬🧬")
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:ffn🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print("\nMODEL INIT CHECK")
        print(model.decoder.layers[0].fc1.weight[0, :10])
        print()
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────────────────────────────────       
        # =============================================================================
        # 1️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder FFNs 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i >= len(model.decoder.layers) - exp_args.rtm_num_layers:

                original_activation_fn = layer.activation_fn

                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,
                    rtm_rotation=exp_args.rtm_rotation                 
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:

                    # 🔒 Configure RTM coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Activation Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ───────────────────────────────────────────────────────────────────────────────────────────────── 
                # ─────────────────────────────────────────────────────────────────────────────────────────────────           


                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                def wrapped_activation(
                    x,
                    act_fn=original_activation_fn,
                    rtm_module=rtm
                ):

                    # ==========================================================
                    # 1️⃣.2️⃣.1️⃣ Standard FFN activation
                    # ==========================================================
                    x = act_fn(x)

                    # ==========================================================
                    # 1️⃣.2️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 1️⃣.2️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm_module.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0 + exp_args.rtm_transition_amplification * transition_scale
                    # )

                    # x = x * (
                    #     1.0 + rtm_module.amplification * transition_scale
                    # )

                    global RTM_SCALE

                    x_base = x

                    x_mod = x * (
                        1.0 + rtm_module.amplification * transition_scale
                    )

                    x_mod = rtm_module(x_mod)
                    # ==========================================================
                    # 1️⃣.2️⃣.6️⃣ RTM refinement
                    # ==========================================================
                    # x = rtm_module(x)
                    x = x_base + RTM_SCALE * (x_mod - x_base)

                    return x
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               

                # =============================================================================
                # 📌📌 Replace decoder activation function
                # =============================================================================                 
                layer.activation_fn = wrapped_activation
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  


                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 FFN Insertion | RTM inserted into decoder layer {i}")                
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────




    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣ 🧩🧠 === Cross-Attention Insertion ♻️♻️ ================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣ 🧩🧠 === Cross-Attention Insertion ♻️♻️ ================================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────

    elif exp_args.rtm_insertion_type == "cross_attn":
    
        # ================================================================================================
        # 2️⃣.0️⃣ 🧠 Build RTM_Former 🅰️🔼 
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:cross_attn🧬🧬")
        print(f"♻️♻️ Loaded Model: RTM_Former for insertion_typ:cross_attn🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder Cross-Attention 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i >= len(model.decoder.layers) - exp_args.rtm_num_layers:

                # =============================================================================
                # 2️⃣.2️⃣.1️⃣📌📌 RTM module
                # =============================================================================
                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,
                    rtm_rotation=exp_args.rtm_rotation                    
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:

                    # 🔒 Configure RTM coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Attention Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────


                # =============================================================================
                # 2️⃣.2️⃣.2️⃣📌📌 Store original encoder-attention forward
                # =============================================================================
                original_encoder_attn_forward = layer.encoder_attn.forward

                # =============================================================================
                # 2️⃣.2️⃣.3️⃣📌📌 Wrapped Cross-Attention forward
                # =============================================================================
                def wrapped_encoder_attn_forward(
                    query,
                    key,
                    value,
                    **kwargs
                ):

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.1️⃣ Original Cross-Attention
                    # ==========================================================
                    x, attn = original_encoder_attn_forward(
                        query=query,
                        key=key,
                        value=value,
                        **kwargs
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 2️⃣.2️⃣.3️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0
                    #     +
                    #     exp_args.rtm_transition_amplification
                    #     *
                    #     transition_scale
                    # )

                    x = x * (
                        1.0
                        +
                        rtm.amplification
                        *
                        transition_scale
                    )

                    return x, attn
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────


                # =============================================================================
                # 📌📌 Replace encoder-attention forward
                # =============================================================================
                layer.encoder_attn.forward = wrapped_encoder_attn_forward
                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                # ─────────────────────────────────────────────────────────────────────────────────────────────────

                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 Cross-Attention Insertion | RTM inserted into decoder layer {i}") 

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────



    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 3️⃣ 🧩🧠 === FFN Depth-Weighted Insertion === ♻️♻️ =========================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    # 3️⃣ 🧩🧠 === FFN Depth-Weighted Insertion === ♻️♻️ =========================================== 
    # ───────────────────────────────────────────────────────────────────────────────────────────────
    elif exp_args.rtm_insertion_type == "ffn_depth_weighted":

        # ================================================================================================
        # 1️⃣.0️⃣ 🧠 Build RTM_Former  
        # ================================================================================================
        from models.RTM_Former import RTM_Former
        print(f"♻️♻️ Loaded Model: Depth-Weighted RTM_Former for insertion_typ:ffn_depth_weighted🧬🧬")
        print(f"♻️♻️ Loaded Model: Depth-Weighted RTM_Former for insertion_typ:ffn_depth_weighted🧬🧬")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────

        # 🔒 IMPORTANT: reset RNG after importing RTM_Former, before the real training model is built (For Repeatable Initialization)
        set_seed_torch(seed1)
        set_seed_main(seed2)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # =============================================================================
        # 2️⃣.1️⃣📌📌 Load baseline Fairseq Transformer 🅰️🔼 
        # =============================================================================
        model = task.build_model(cfg.model)

        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print("\nMODEL INIT CHECK")
        print(model.decoder.layers[0].fc1.weight[0, :10])
        print()
        # ==========================================================
        # 🚦🔍 Select layers
        # ========================================================== 
        selected_layers = list(
            range(
                len(model.decoder.layers) - exp_args.rtm_num_layers,
                len(model.decoder.layers)
            )
        )
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        # ─────────────────────────────────────────────────────────────────────────────────────────────────       
        # =============================================================================
        # 1️⃣.2️⃣📌📌 Inject/Apply RTM enhancement to decoder FFNs 🅱️🔼
        # =============================================================================
        for i, layer in enumerate(model.decoder.layers):

            if i in selected_layers:

                original_activation_fn = layer.activation_fn

                layer_rank = selected_layers.index(i)


                # =====================================================
                # ⏭️⏭️ Forward direction Layer scale ⏭️⏭️
                # =====================================================               
                if exp_args.rtm_layer_scale_direction == "forward":
                    layer_scale = exp_args.rtm_layer_scale_min + (
                        1.0 - exp_args.rtm_layer_scale_min
                    ) * (
                        layer_rank / max(1, exp_args.rtm_num_layers - 1)
                    )
                # =====================================================
                # ⏪⏪ Reverse direction Layer scale ⏪⏪
                # =====================================================  
                elif exp_args.rtm_layer_scale_direction == "reverse":
                    layer_scale = 1.0 - (
                        1.0 - exp_args.rtm_layer_scale_min
                    ) * (
                        layer_rank / max(1, exp_args.rtm_num_layers - 1)
                    )
                # =====================================================
                # ⏪🔒 Reverse direction + Fixed Final Layer Scale ⏪🔒
                # =====================================================
                elif exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                    if layer_rank == exp_args.rtm_num_layers - 1:
                        layer_scale = exp_args.rtm_final_layer_scale
                    else:
                        layer_scale = 1.0 - (
                            1.0 - exp_args.rtm_layer_scale_min
                        ) * (
                            layer_rank / max(1, exp_args.rtm_num_layers - 2)
                        ) 
                # ────────────────────────────────────────────────────────────────────────────────────────
                else:
                    raise ValueError(f"Unknown RTM layer scale direction: {exp_args.rtm_layer_scale_direction}")
                # ────────────────────────────────────────────────────────────────────────────────────────

                rtm = RTM_Former(
                    dim=layer.fc1.out_features,
                    alpha=exp_args.rtm_alpha,
                    amplification=exp_args.rtm_transition_amplification,  
                    gate_type=exp_args.rtm_gate_type,
                    rtm_rotation=exp_args.rtm_rotation                                     
                )
                #--------------------------------------------------------------------------------------------------
                #--------------------------------------------------------------------------------------------------
                # 🧩🔒 Register RTM Module (Architecture Visibility + Parameter Tracking)
                #--------------------------------------------------------------------------------------------------
                if exp_args.rtm_register_module:
                    
                    # 🔒 Configure RTM learnable coefficient settings
                    rtm.alpha.requires_grad = False
                    rtm.amplification.requires_grad = False

                    layer.rtm_module = rtm

                #--------------------------------------------------------------------------------------------------
                # 🔗🔒 Closure-Based RTM Injection (Activation Wrapper Only)
                #--------------------------------------------------------------------------------------------------
                else:

                    pass
                # ─────────────────────────────────────────────────────────────────────────────────────────────────                
                # ─────────────────────────────────────────────────────────────────────────────────────────────────

                # ─────────────────────────────────────────────────────────────────────────────────────────────────
                def wrapped_activation(
                    x,
                    act_fn=original_activation_fn,
                    rtm_module=rtm,
                    layer_scale=layer_scale
                ):

                    # ==========================================================
                    # 1️⃣.2️⃣.1️⃣ Standard FFN activation
                    # ==========================================================
                    x = act_fn(x)

                    # ==========================================================
                    # 1️⃣.2️⃣.2️⃣ Previous hidden state
                    # ==========================================================
                    prev_x = torch.roll(
                        x,
                        shifts=1,
                        dims=0
                    )

                    prev_x[0] = x[0]

                    # ==========================================================
                    # 1️⃣.2️⃣.3️⃣ Transition dynamics ⏭️🔖
                    # ==========================================================
                    transition = x - prev_x

                    transition = transition / (
                        1e-6 +
                        transition.abs().mean(
                            dim=-1,
                            keepdim=True
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.4️⃣ Transition-conditioned scaling
                    # ==========================================================
                    transition_scale = torch.sigmoid(
                        rtm_module.alpha * torch.tanh(
                            transition
                        )
                    )

                    # ==========================================================
                    # 1️⃣.2️⃣.5️⃣ Transition-conditioned amplification
                    # ==========================================================
                    # x = x * (
                    #     1.0 + exp_args.rtm_transition_amplification * transition_scale
                    # )

                    # x = x * (
                    #     1.0 + rtm_module.amplification * transition_scale
                    # )

                    global RTM_SCALE

                    x_base = x

                    x_mod = x * (
                        1.0 + rtm_module.amplification * transition_scale
                    )

                    x_mod = rtm_module(x_mod)
                    # ==========================================================
                    # 1️⃣.2️⃣.6️⃣ RTM refinement
                    # ==========================================================

                    # ----------------------------------------------------------
                    # 🏔️ Depth-Aware RTM Scaling (Layer-Adaptive Modulation)
                    # ---------------------------------------------------------- 
                    if exp_args.rtm_depth_weight:

                        x = x_base + RTM_SCALE * layer_scale * (x_mod - x_base)

                    # ----------------------------------------------------------
                    # ⚖️ Uniform RTM Scaling (Same Modulation Across Layers)
                    # ----------------------------------------------------------
                    else:

                        x = x_base + RTM_SCALE * (x_mod - x_base)
                    # ─────────────────────────────────────────────────────────────────────────────────────────────────

                    return x
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               # ─────────────────────────────────────────────────────────────────────────────────────────────────
               

                # =============================================================================
                # 📌📌 Replace decoder activation function
                # =============================================================================                 
                layer.activation_fn = wrapped_activation
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  
                # ─────────────────────────────────────────────────────────────────────────────────────────────────  


                # ==========================================================
                # 🚦🔍 RTM inserted into decoder layer 
                # ==========================================================                  
                rtm_layers.append(i)
                print(f"📣 FFN Insertion | RTM inserted into decoder layer {i} | layer_scale={layer_scale:.3f}")              
        # ─────────────────────────────────────────────────────────────────────────────────────────────────
        print(f"✅ Loaded Model: Depth-Weighted RTM_Former")
        print(f"✅ Loaded Model: {model.__class__.__name__}")
        # ─────────────────────────────────────────────────────────────────────────────────────────────────


    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    else:
        raise ValueError(f"Unknown RTM insertion type: {exp_args.rtm_insertion_type}")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────

    print(f"📣📣 RTM_Former for insertion_typ={exp_args.rtm_insertion_type} | Loaded Model={exp_args.net }♻️♻️")
    print(f"📣📣 RTM_Former for insertion_typ={exp_args.rtm_insertion_type} | Loaded Model={exp_args.net }♻️♻️")
    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────────────────────────────────────────────



    # ─────────────────────────────────────────────────────────────────────────────────────────────────
    #  ======== 🔖  5.1.1 Model Configuration Hyperparameter Summary ==================================
    # ───────────────────────────────────────────────────────────────────────────────────────────────── 
    if exp_args.net == "RTM_Former":

        rtm_log = (
            f"📣 RTM Configuration             : "
            f"alpha={exp_args.rtm_alpha} | "
            f"amp={exp_args.rtm_transition_amplification} | "
            f"layers={exp_args.rtm_num_layers} | "
            f"insertion={exp_args.rtm_insertion_type} | "
            f"scale_init={exp_args.rtm_scale} | "
            f"start_epoch={exp_args.rtm_start_epoch} | "
            f"full_epoch={exp_args.rtm_full_epoch}"
        )

        # Depth-weighted RTM only
        if exp_args.rtm_insertion_type == "ffn_depth_weighted":

            rtm_log += (
                f" | layer_scale_min={exp_args.rtm_layer_scale_min}"
                f" | layer_direction={exp_args.rtm_layer_scale_direction}"
                f" | gate={exp_args.rtm_gate_type}"
            )

            # Only exists for reverse_final_fixed
            if exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                rtm_log += (
                    f" | final_layer_scale={exp_args.rtm_final_layer_scale}"
                )

        print(rtm_log)


    else:
        print(
            f"📣 Baseline Configuration        : "
            f"model={exp_args.net}"
        )
    # ─────────────────────────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ======== 🔖🔖  5.2  Insert model to device 🔖🔖================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

model = model.to(DEVICE)
 # ─────────────────────────────────────────────────────────────────────────────────────────────────
print(f"🚀🚀 Model moved to device: {DEVICE} ♻️♻️")
print(f"🚀🚀 Model moved to device: {DEVICE} ♻️♻️")
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ======== 🔖🔖 5.3  Model Validation & Sanity Checks 🔖🔖=======================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 🔍 Print custom activations to confirm they are used
for name, module in model.named_modules():
    if isinstance(module, nn.Module) and module.__class__.__name__ in ["TanhExp", "FFTGate"]:
        print(f"🔎 Found custom activation in {name}: {module}")


# ✅ Sanity check Device
if DEVICE.type == "cuda":
    print(f"🚀 Model moved to CUDA: {torch.cuda.get_device_name(DEVICE)}")
else:
    print("⚠️ Using CPU (CUDA not available)")
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 6. PATH DEFINATION | XXX --------------------------------------------------------#####################
########################################################################################################################


# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 1️⃣ 📌📌 ======== ENSURE DIRECTORY EXIST ========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# # ─────────────────────────────────────────────────────────────────────────────────────────────────
# # 🟡 === Checkpoint directories ===
# if not os.path.exists('checkpoint'):
#     os.makedirs('checkpoint')
# # ─────────────────────────────────────────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Results directories ===
# ─────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}')

if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === LR Log directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Training Log directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 🟡 === Model Summary directory ===
# ────────────────────────────────────────────────────────────────────────────────────────────────
if not os.path.exists(f'./Results/{exp_args.dataset_name}/{exp_args.net}'):
    os.makedirs(f'./Results/{exp_args.dataset_name}/{exp_args.net}')
# ────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣ 📌📌 ======== DEFINE DIRECTORY ==============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.1️⃣🔖 ==== Checkpoint Directory Selection | Normal Run vs Crash Run | save every epoch ===== 
# ────────────────────────────────────────────────────────────────────────────────────────────────

# 🅰️💾 Normal run: save checkpoints in standard folder
if not exp_args.crash_run:
    checkpoint_dir = f'./checkpoints/{exp_args.dataset_name}/{exp_args.net}/'

    # 📁 Create checkpoint folder if it does not exist
    os.makedirs(checkpoint_dir, exist_ok=True)

    print(f"🅰️💾 Normal Run Enabled | Saving checkpoints to: {checkpoint_dir}")
#---------------------------------------------------------------------------------------------------
# 🅱️♻️ Crash run: save checkpoints in recovery folder
else:
    checkpoint_dir = f'./checkpoints_crash_run/{exp_args.dataset_name}/{exp_args.net}/'

    # 📁 Create checkpoint folder if it does not exist    
    os.makedirs(checkpoint_dir, exist_ok=True)

    print(f"🅱️♻️ Crash Run Enabled | Saving recovery checkpoints to: {checkpoint_dir}")

# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.2️⃣🔖 === Main Test & Train Results === 
train_results_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/Train_{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
test_results_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/Test_{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
# ────────────────────────────────────────────────────────────────────────────────────────────────
# 2️⃣.3️⃣🔖  === LR, Training & Summary logs  === 
LR_save_paths = {
    "LR_history": f"./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_LR_log.txt"
}
save_paths = {
    "log_history": f"./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_training_logs.txt"
}
# ────────────────────────────────────────────────────────────────────────────────────────────────

# 2️⃣.4️⃣🔖 ===  Model summary path === 
model_summary_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_model_summary.txt'
model_summary_full_path = f'./Results/{exp_args.dataset_name}/{exp_args.net}/{exp_args.net}_{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}_model_summary_full.txt'
# ────────────────────────────────────────────────────────────────────────────────────────────────

# # ✅🔖 === Path template for saving checkpoints at every epoch | Part2 (if scratch occured) ===
# checkpoint_dir = f'./checkpoints/{exp_args.dataset_name}/tanhexp_AfterCrash_41_49epochs/'
# os.makedirs(checkpoint_dir, exist_ok=True)
# ────────────────────────────────────────────────────────────────────────────────────────────────






########################################################################################################################
####-------| NOTE 7. TRAINING INITIALIZATION & OPTIMIZATION SETUP | XXX ---------------------------#####################
########################################################################################################################

# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ========  7.1 AMP GradScaler ===============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Initialize AMP GradScaler once globall
scaler = torch.cuda.amp.GradScaler()
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  Training Criterion ===============================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 

criterion = LabelSmoothingCrossEntropy(smoothing=exp_args.smoothing)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ========   Optimizer Configuration =========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ♻️ Adam optimizer (β1=0.9, β2=0.98, eps=1e-9) ♻️
# ─────────────────────────────────────────────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=exp_args.lr,
    betas=(0.9, 0.98),
    eps=exp_args.eps,
    weight_decay=exp_args.weight_decay,
)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 4️⃣ ========  Learning Rate Scheduler ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ✅  Transformer LR schedule (warmup + inverse sqrt decay)
# --------------------------

warmup_updates = exp_args.warmup_updates   # configurable warmup steps
peak_lr = exp_args.lr

# ────────────────────────────────────────────────────────────────────────────────────────────────
def lr_lambda(step):
    step = max(1, step)
    if step == 1:
        print(f"🚀 Warmup started at step {step} (target peak LR = {peak_lr}).")
    if step == warmup_updates:
        actual_lr = optimizer.param_groups[0]["lr"]
        print(f"✅ Warmup finished at step {step}. Peak LR reached: {actual_lr:.8f}")
    if step == warmup_updates + 1:
        print(f"📉 Inverse square root decay started at step {step}.")
    
    if step <= warmup_updates:
        # Linear warmup
        scale = step / warmup_updates
    else:
        # Inverse square root decay
        scale = (warmup_updates ** 0.5) / (step ** 0.5)
    return scale

# ────────────────────────────────────────────────────────────────────────────────────────────────

lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
# ─────────────────────────────────────────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────────────────────────────────────────
# 5️⃣ ========  Global Tracking Variables ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ global initialization
best_bleu = 0.0
total_test_duration = 0.0
RTM_SCALE_LOG = ""
# ─────────────────────────────────────────────────────────────────────────────────────────────────

Python: 3.10.16 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:19:12) [MSC v.1929 64 bit (AMD64)]
Torch: 2.2.0+cu121 | CUDA available: True
Fairseq: 0.12.2
OmegaConf: 2.0.6
Hydra-Core: 1.0.7
✅ Current working directory: C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH
✅ sys.path updated:
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\python310.zip
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\DLLs
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env
   📂 
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\win32
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\win32\lib
   📂 c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\Pythonwin
   📂 C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH
   📂 C:\Users\emeka\Research\ModelCUDA\Transformers\Transformer_Baselines_EN_ZH\models
   📂 C:\

c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\modules\module.py:1150: UserWarning: expandable_segments not supported on this platform (Triggered internally at ..\c10/cuda/CUDAAllocatorConfig.h:30.)
  return t.to(device, dtype if t.is_floating_point() or t.is_complex() else None, non_blocking)


In [ ]:
########################################################################################################################
####-------| NOTE 11. TEST ALL CHECKPOINTS (BLEU per epoch)  | XXX -----------------------------########################
########################################################################################################################
########################################################################################################################
####-------| NOTE 11. TEST ALL CHECKPOINTS (BLEU per epoch)  | XXX -----------------------------########################
########################################################################################################################


# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  1️⃣ ======== Paths for checkpoint BLEU sweeps ===================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ Paths for checkpoint BLEU sweeps
test_checkpoint_results_path = (
    f'./Results/{exp_args.dataset_name}/{exp_args.net}/'
    f'TestCheckpoints_{exp_args.net}_'
    f'{exp_args.dataset_name}_{exp_args.optimizer1}_{exp_args.mode_name}.txt'
)
# ────────────────────────────────────────────────────────────────────────────────────────────────


# # ✅ Paths for checkpoint BLEU sweeps | Part2 (if Cratch occured)
# test_checkpoint_results_path = (
#     f'./Results/{exp_args.dataset_name}/{exp_args.act_name}/'
#     f'TestCheckpoints_{exp_args.act_name}_{exp_args.net}_'
#     f'{exp_args.dataset_name}_{exp_args.optimizer1}_tanhexp_AfterCrash_41_49epochs.txt'
# )
# ────────────────────────────────────────────────────────────────────────────────────────────────


# # ─────────────────────────────────────────────────────────────────────────────────────────────────
# #  2️⃣ ========  BLEU Detokenization (🔖 EN/FR/DE) ================================================
# # ───────────────────────────────────────────────────────────────────────────────────────────────── 
# # ────────────────────────────────────────────────────────────────────────────────────────────────
# def detokenize_for_bleu(s: str) -> str:
#     s = s.replace("@@ ", "").replace("@@", "")  # BPE
#     s = s.replace("▁", " ")                     # SentencePiece
#     s = " ".join(s.split())                     # normalize spaces
#     s = re.sub(r"\s+([?.!,])", r"\1", s)        # punctuation
#     return s.strip()
# # ────────────────────────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  2️⃣ ========  BLEU Detokenization (🔖 Chinese) =================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
def detokenize_for_bleu(s: str) -> str:
    # For Chinese, don't remove SentencePiece boundaries or spaces | Only remove BPE continuation markers
    # sacreBLEU's "zh" tokenizer expects raw characters.
    return s.replace("@@ ", "").replace("@@", "").strip()
# ────────────────────────────────────────────────────────────────────────────────────────────────




# ─────────────────────────────────────────────────────────────────────────────────────────────────
#  3️⃣ ======== Test_Checkpoint Function ===========================================================
# ───────────────────────────────────────────────────────────────────────────────────────────────── 
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ✅ test_checkpoint function
def test_checkpoint_epochs(task, model, criterion, checkpoint_dir, test_checkpoint_results_path):
    """
    Evaluate BLEU (beam=5) for all checkpoints in checkpoint_dir.
    Save results to test_checkpoint_results_path in the same format as normal test logs.
    """
# ────────────────────────────────────────────────────────────────────────────────────────────────
    global RTM_SCALE, RTM_SCALE_LOG
    # ────────────────────────────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Collect checkpoints sorted by epoch number in filename
    def extract_epoch_num(filename):
        match = re.search(r"epoch_(\d+)", filename)
        return int(match.group(1)) if match else float("inf")  # inf if no epoch number

    ckpts = sorted(
        glob.glob(os.path.join(checkpoint_dir, "*.pt")),
        key=lambda x: extract_epoch_num(os.path.basename(x))
    )

    if not ckpts:
        print(f"⚠️ No checkpoints found in {checkpoint_dir}")
        return

    # # ────────────────────────────────────────────────────────────────────────────────────────────────
    # # ✅ Reset output file
    # os.makedirs(os.path.dirname(test_checkpoint_results_path), exist_ok=True)
    # with open(test_checkpoint_results_path, "w", encoding="utf-8") as f:
    #     f.write("")
    # # ────────────────────────────────────────────────────────────────────────────────────────────────

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 📦 Reset output file + write model status
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    #1️⃣ Create output folder if it does not exist
    os.makedirs(os.path.dirname(test_checkpoint_results_path), exist_ok=True)

    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ✅ Normal run: reset file
    # ✅ Crash run : append to existing file
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 2️⃣♻️ Normal run: reset file
    if not exp_args.crash_run:

        with open(test_checkpoint_results_path, "w", encoding="utf-8") as f:
            f.write("")

    # 3️⃣⏭️ Crash run : append to existing file
    else:

        with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
            f.write("\n♻️ Resuming checkpoint testing after crash\n\n")
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # 4️⃣✍️  Write test result header (model information)
    with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
        f.write("")

        # ---------------------------------------------------------------------------
        # 4️⃣.1️⃣🏷️ Testing Model Status
        # ---------------------------------------------------------------------------
        if exp_args.net == "RTM_Former":

            f.write(
                f"Model: RTM_Former\n\n"
            )

        else:

            f.write(
                f"Model: Baseline Fairseq Transformer\n\n"
            )        
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ────────────────────────────────────────────────────────────────────────────────────────────────



    # ────────────────────────────────────────────────────────────────────────────────────────────────
    best_bleu = 0.0
    total_test_duration = 0.0
    # ────────────────────────────────────────────────────────────────────────────────────────────────


    # ────────────────────────────────────────────────────────────────────────────────────────────────
    # ────────────────────────────────────────────────────────────────────────────────────────────────
    for ckpt in ckpts:

        # # ────────────────────────────────────────────────────────────────────────────────
        # # ✅ 📌📌 Load checkpoint (OLD) 🅰️🔼 
        # # ────────────────────────────────────────────────────────────────────────────────        
        # print(f"🔄 Loading checkpoint: {ckpt}")
        # state = torch.load(ckpt, map_location=DEVICE)
        # model.load_state_dict(state["model_state"], strict=False)
        # model.to(DEVICE)
        # model.eval()
        # --------------------------------------------------------------------------------------------------
        # ────────────────────────────────────────────────────────────────────────────────
        # ✅📌📌 Load checkpoint (NEW) 🅱️🔼 Restore RTM schedule for checkpoint testing using same strategy as main loop
        # ────────────────────────────────────────────────────────────────────────────────
        if exp_args.net == "Transformer":
            print(f"🔄 Loading checkpoint Baseline Transformer : {ckpt}")
            state = torch.load(ckpt, map_location=DEVICE)
            model.load_state_dict(state["model_state"], strict=False)
            model.to(DEVICE)
            model.eval()
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        elif exp_args.net == "RTM_Former":
            print(f"🔄 Loading checkpoint RTM_Former : {ckpt}")
            state = torch.load(ckpt, map_location=DEVICE)
            model.load_state_dict(state["model_state"], strict=False)            
            

            RTM_START_EPOCH = exp_args.rtm_start_epoch
            RTM_FULL_EPOCH = exp_args.rtm_full_epoch
            ckpt_epoch = state["epoch"]

            if ckpt_epoch < RTM_START_EPOCH:
                RTM_SCALE = exp_args.rtm_scale
            else:
                progress = min(
                    1.0,
                    (ckpt_epoch - RTM_START_EPOCH) / (RTM_FULL_EPOCH - RTM_START_EPOCH)
                )

                RTM_SCALE = 0.5 * (1.0 - math.cos(math.pi * progress))

            print(f"📐 Checkpoint RTM_SCALE for epoch {ckpt_epoch}: {RTM_SCALE:.3f}")

            if exp_args.rtm_insertion_type == "ffn_depth_weighted":

                if exp_args.rtm_layer_scale_direction == "forward":
                    depth_scales = [
                        exp_args.rtm_layer_scale_min
                        + (1.0 - exp_args.rtm_layer_scale_min)
                        * (j / max(1, exp_args.rtm_num_layers - 1))
                        for j in range(exp_args.rtm_num_layers)
                    ]

                elif exp_args.rtm_layer_scale_direction == "reverse":
                    depth_scales = [
                        1.0
                        - (1.0 - exp_args.rtm_layer_scale_min)
                        * (j / max(1, exp_args.rtm_num_layers - 1))
                        for j in range(exp_args.rtm_num_layers)
                    ]

                elif exp_args.rtm_layer_scale_direction == "reverse_final_fixed":
                    depth_scales = []

                    for j in range(exp_args.rtm_num_layers):
                        if j == exp_args.rtm_num_layers - 1:
                            scale_j = exp_args.rtm_final_layer_scale
                        else:
                            scale_j = (
                                1.0
                                - (1.0 - exp_args.rtm_layer_scale_min)
                                * (j / max(1, exp_args.rtm_num_layers - 2))
                            )

                        depth_scales.append(scale_j)

                else:
                    raise ValueError(
                        f"Unknown RTM layer scale direction: {exp_args.rtm_layer_scale_direction}"
                    )

                final_scales = [RTM_SCALE * s for s in depth_scales]

                RTM_SCALE_LOG = f" | RTM_SCALE={RTM_SCALE:.3f}"
                RTM_SCALE_LOG += (
                    f" | RTM_layer_scales({exp_args.rtm_layer_scale_direction})="
                    f"{[round(s, 3) for s in final_scales]}"
                )

                print(
                    f"📐 Checkpoint final RTM layer scales "
                    f"({exp_args.rtm_layer_scale_direction}): "
                    f"{[round(s, 3) for s in final_scales]}"
                )
            model.to(DEVICE)
            model.eval()               
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        else:
            raise ValueError(f"Unknown model type: {exp_args.net}")
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ────────────────────────────────────────────────────────────────────────────────────────────────


        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Build test iterator
        test_iter = task.get_batch_iterator(
            dataset=task.dataset("test"),
            max_tokens=fairseq_args.max_tokens,   # cut in half option: fairseq_args.max_tokens // 2
            max_positions=(exp_args.max_source_positions, exp_args.max_target_positions),
            seed=seed1,
            num_shards=exp_args.num_shards,
            shard_id=exp_args.shard_id,
            num_workers=exp_args.num_workers,
            ignore_invalid_inputs=exp_args.ignore_invalid_inputs,
        ).next_epoch_itr(shuffle=False)
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ References
        true_references = []
        for sample in test_iter:
            targets = sample["target"]
            for t in targets:
                t_no_pad = t[t != tgt_dict.pad()]
                ref_str = tgt_dict.string(t_no_pad, bpe_symbol="@@")
                true_references.append(detokenize_for_bleu(ref_str))
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Rebuild iterator for decoding
        test_iter = task.get_batch_iterator(
            dataset=task.dataset("test"),
            max_tokens=fairseq_args.max_tokens,    # cut in half option: fairseq_args.max_tokens // 2
            max_positions=(exp_args.max_source_positions, exp_args.max_target_positions),
            seed=seed1,
            num_shards=exp_args.num_shards,
            shard_id=exp_args.shard_id,
            num_workers=exp_args.num_workers,
            ignore_invalid_inputs=exp_args.ignore_invalid_inputs,
        ).next_epoch_itr(shuffle=False)
        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Build generator with beam=5
        gen_cfg = deepcopy(cfg.generation)
        gen_cfg.beam = exp_args.beam         # 🎀🔖 5
        gen_cfg.lenpen = exp_args.lenpen     # 🎀🔖 1.0 
        generator = task.build_generator([model], gen_cfg)
        # ────────────────────────────────────────────────────────────────────────────────────────────────


 

        # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Run evaluation
        sys_outputs, test_loss = [], 0.0
        start_time = time.time()

        with torch.no_grad():
            with tqdm(enumerate(test_iter), total=len(test_iter), desc=f"Testing {os.path.basename(ckpt)}") as progress:
                for i, sample in progress:   # 👈 keep i!
                    sample = utils.move_to_cuda(sample) if torch.cuda.is_available() else sample
                    src_tokens = sample["net_input"]["src_tokens"]
                    prev_output_tokens = sample["net_input"]["prev_output_tokens"]
                    target = sample["target"]
                    src_lengths = (src_tokens != src_dict.pad()).sum(dim=1)

                    with torch.cuda.amp.autocast():
                        logits, _ = model(src_tokens, src_lengths, prev_output_tokens)
                        pad = tgt_dict.pad()
                        logits2d = logits.view(-1, logits.size(-1))
                        target1d = target.view(-1)
                        mask = target1d != pad
                        loss = criterion(logits2d[mask], target1d[mask]) if mask.any() else logits2d.sum() * 0.0
                    test_loss += loss.item()

                    hypos = task.inference_step(generator, [model], sample)
                    for j, hypos_j in enumerate(hypos):
                        hypo_tokens = [tok for tok in hypos_j[0]["tokens"].tolist() if tok != tgt_dict.pad()]
                        sys_str = tgt_dict.string(hypo_tokens, bpe_symbol="@@")
                        sys_outputs.append(detokenize_for_bleu(sys_str))

                    # ✅ Update progress bar postfix with running average
                    progress.set_postfix(loss=round(test_loss / (i + 1), 3))
       # ────────────────────────────────────────────────────────────────────────────────────────────────




       # ────────────────────────────────────────────────────────────────────────────────────────────────
        avg_loss = test_loss / max(1, len(test_iter))
        # bleu = sacrebleu.corpus_bleu(sys_outputs, [true_references])                 #🔖 EN/FR/DE
        bleu = sacrebleu.corpus_bleu(sys_outputs, [true_references], tokenize="zh")    #🔖 Chinese

        duration = time.time() - start_time
        total_test_duration += duration
       # ────────────────────────────────────────────────────────────────────────────────────────────────


       # ────────────────────────────────────────────────────────────────────────────────────────────────
        # ✅ Write results
        with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
            f.write(f"Checkpoint {os.path.basename(ckpt)} | Test Loss: {avg_loss:.3f} | sacreBLEU: {bleu.score:.2f} | Beam=5\n")

        print(f"✅ {os.path.basename(ckpt)} | Loss {avg_loss:.3f} | sacreBLEU {bleu.score:.2f}")


        # 🧹 add visual spacing between runs
        tqdm.write("")  
       # ────────────────────────────────────────────────────────────────────────────────────────────────

       # ────────────────────────────────────────────────────────────────────────────────────────────────
        if bleu.score > best_bleu:
            best_bleu = bleu.score

    with open(test_checkpoint_results_path, "a", encoding="utf-8") as f:
        f.write(f"\n🏆 Best BLEU Score: {best_bleu:.2f}\n")
        total_mins, total_secs = divmod(total_test_duration, 60)
        f.write(f"\n🕒 Total Test Time over all checkpoints: {int(total_mins)} min {total_secs:.2f} sec\n")

    print(f"🏆 Best BLEU over checkpoints: {best_bleu:.2f}")
    # ────────────────────────────────────────────────────────────────────────────────────────────────



# ────────────────────────────────────────────────────────────────────────────────────────────────
# # ✅ Free unused GPU memory BEFORE training starts
# torch.cuda.empty_cache()
# torch.cuda.reset_peak_memory_stats()

# 4️⃣ Call the function
test_checkpoint_epochs(task, model, criterion, checkpoint_dir, test_checkpoint_results_path)
# ────────────────────────────────────────────────────────────────────────────────────────────────
# ────────────────────────────────────────────────────────────────────────────────────────────────

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\epoch_49_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt
📐 Checkpoint RTM_SCALE for epoch 49: 1.000


Testing epoch_49_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt:   0%|          | 0/82 [00:00<?, ?it/s]c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)
c:\Users\emeka\anaconda3\envs\pytorch_env\lib\site-packages\torch\nn\functional.py:5109: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
Testing epoch_49_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt: 100%|██████████| 82/82 [00:35<00:00,  2.31it/s, loss=4.23]


✅ epoch_49_RTM_Former_IWSLT17_En_Zh_Seed1_1_EXP3.pt | Loss 4.226 | sacreBLEU 20.60

🔄 Loading checkpoint RTM_Former : ./checkpoints_crash_run/IWSLT17_En_Zh/RTM_Former\RTM_Former_IWSLT17_En_Zh_LR0_0001_Adam_Seed1_1_EXP3_LastEpoch49.pt
📐 Checkpoint RTM_SCALE for epoch 49: 1.000


Testing RTM_Former_IWSLT17_En_Zh_LR0_0001_Adam_Seed1_1_EXP3_LastEpoch49.pt: 100%|██████████| 82/82 [00:32<00:00,  2.49it/s, loss=4.23]


✅ RTM_Former_IWSLT17_En_Zh_LR0_0001_Adam_Seed1_1_EXP3_LastEpoch49.pt | Loss 4.226 | sacreBLEU 20.60

🏆 Best BLEU over checkpoints: 20.60
